# Parcel Update
* Amy Fish, afish@trpa.gov
* Andy McClary, amcclary@trpa.gov
* Mason Bindl, mbindl@trpa.gov

## Setup

### Imports, Functions, and Global Variables

In [16]:
# import packages
import urllib
import json
import requests
import os
import shutil
import sys
import re
import logging

from datetime import datetime 
import time
from zipfile import ZipFile
from io import BytesIO

import pandas as pd
import pyodbc

import arcpy
from arcgis.features import GeoAccessor, GeoSeriesAccessor
from arcgis.gis import GIS

from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

# import traceback
# from pytz import timezone
# import pytz
import pathlib
# from IPython.display import display
# import getpass
from time import strftime
# import linecache
# import ssl

# environment settings
arcpy.env.workspace = "//Trpa-fs01/GIS/PARCELUPDATE/Workspace/ParcelStaging.gdb"
arcpy.env.overwriteOutput = True
arcpy.env.outputCoordinateSystem = arcpy.SpatialReference(26910)

# set workspace and sde connections 
workspace = "//Trpa-fs01/GIS/PARCELUPDATE/Workspace/Staging"

# network path to connection files
filePath = "//Trpa-fs01/GIS/PARCELUPDATE/Workspace/"
# database file path 
sdeBase    = os.path.join(filePath, "Vector.sde/")
sdeCollect = os.path.join(filePath, "Collection.sde")
sdeTabular = os.path.join(filePath, "Tabular.sde")

# portal signin
## TRPA_ADMIN credentials 
portal_user = "TRPA_PORTAL_ADMIN"
portal_pwd = "@dmin6224"
portal_url = "https://maps.trpa.org/portal/"
# sign in
arcpy.SignInToPortal(portal_url, portal_user, portal_pwd)

### Functions ###
# time a function function
## use as decorator @timer
def timer(func):
    def wrapper(*args, **kwargs):
        start_time = time.time()
        result = func(*args, **kwargs)
        end_time = time.time()
        print(f"Function {func.__name__} took {end_time - start_time} seconds to execute.")
        return result
    return wrapper

# set none to '' for all cells
@timer
def replace_null_values_with_blank(fc):
    field_list = get_text_fields(fc)
    with arcpy.da.UpdateCursor(fc, field_list) as cursor: 
        for row in cursor: 
            for i in range(len(row)): 
                if row[i] is None: 
                    row[i] = "" 
            cursor.updateRow(row)
            
@timer           
def UpdateFieldFromDictionary(featureclass, field, update_dictionary):
    record_count = 0
    with arcpy.da.UpdateCursor(featureclass, field) as cursor:
        for row in cursor:
            key_field_value = row[0]
            if key_field_value in update_dictionary:
                row[0] = update_dictionary[key_field_value]
                cursor.updateRow(row)
                record_count+=record_count
    print(f"{record_count} rows were updated")
                    
# combine duplicate records, creating multipart and dissolved polygons 
@timer
def CombineAPNs(fc, fld_dissolve):    
    from time import strftime  
    print ("Started combining APNs: " + strftime("%Y-%m-%d %H:%M:%S"))

    # get unique values from field
    value_list = [r[0] for r in arcpy.da.SearchCursor(fc, (fld_dissolve))]
    unique_vals = list(set(value_list))
    
    if len(value_list) !=len(unique_vals):
        seen = set()
        dup_vals = set()
        for x in value_list:
            if x in seen:
                dup_vals.add(x)
            else:
                seen.add(x)
        print(dup_vals)
        dup_vals.remove('')
        for unique_val in dup_vals:
            geoms = [r[0] for r in arcpy.da.SearchCursor(fc, ('SHAPE@', fld_dissolve)) if r[1] == unique_val]
            #Probably don't need this as there will always be more than one geometry
            if len(geoms) > 1:
                print(unique_val)    
                diss_geom = DissolveGeoms(geoms)

                # update the first feature with new geometry and delete the others
                where = "{} = '{}'".format(fld_dissolve, unique_val)
                cnt = 0
                with arcpy.da.UpdateCursor(fc, ('SHAPE@'), where) as curs:
                    for row in curs:
                        cnt += 1
                        if cnt == 1:
                            row[0] = diss_geom
                            curs.updateRow(row)
                        else:
                            curs.deleteRow()
    else:
        print("No duplicates!")
    print ("Finished combining APNs: " + strftime("%Y-%m-%d %H:%M:%S"))
    
# union all geometry inputs into one dissolved geometry
@timer
def DissolveGeoms(geoms):
    cnt = 0
    for geom in geoms:
        cnt += 1
        if cnt == 1:
            diss_geom = geom
        else:
            diss_geom = diss_geom.union(geom)
    return diss_geom

# moves attribute values from one feature class to the other using an aspatial join
@timer
def fieldJoinCalc(updateFC, updateFieldsList, sourceFC, sourceFieldsList):
    from time import strftime  
    print ("Started data transfer: " + strftime("%Y-%m-%d %H:%M:%S"))
#     log.info("Started data transfer: " + strftime("%Y-%m-%d %H:%M:%S"))
    # Use list comprehension to build a dictionary from arcpy SearchCursor  
    valueDict = {r[0]:(r[1:]) for r in arcpy.da.SearchCursor(sourceFC, sourceFieldsList)}  
   
    with arcpy.da.UpdateCursor(updateFC, updateFieldsList) as updateRows:  
        for updateRow in updateRows:  
            # store the Join value of the row being updated in a keyValue variable  
            keyValue = updateRow[0]  
            # verify that the keyValue is in the Dictionary  
            if keyValue in valueDict:  
                # transfer the value stored under the keyValue from the dictionary to the updated field.  
                updateRow[1] = valueDict[keyValue][0]  
                updateRows.updateRow(updateRow)    
    del valueDict  
    print ("Finished data transfer: " + strftime("%Y-%m-%d %H:%M:%S"))
#     log.info("Finished data transfer: " + strftime("%Y-%m-%d %H:%M:%S"))

# transfer attributes frome one feature class field to another while using multiple fields to create the keys
@timer
def fieldJoinCalc_multikey(updateFC, updateFieldsList_key, updateFieldsList_value, sourceFC, sourceFieldsList_key, sourceFieldsList_value):
    from time import strftime  
    print ("Started data transfer: " + strftime("%Y-%m-%d %H:%M:%S"))
#     log.info("Started data transfer: " + strftime("%Y-%m-%d %H:%M:%S"))
    # Use list comprehension to build a dictionary from arcpy SearchCursor  
    total_count=0
    valueDict = {(r[0]+r[1]):(r[2]) for r in arcpy.da.SearchCursor(sourceFC, (sourceFieldsList_key + sourceFieldsList_value))}  
    with arcpy.da.UpdateCursor(updateFC, (updateFieldsList_key+ updateFieldsList_value)) as updateRows:  
        for updateRow in updateRows:  
            # store the Join value of the row being updated in a keyValue variable  
            keyValue = updateRow[0]+updateRow[1]
            # verify that the keyValue is in the Dictionary  
            if keyValue in valueDict:
                total_count +=1
                if (total_count%1000)==0:
                    print (f"Updating row {total_count}")
                # transfer the value stored under the keyValue from the dictionary to the updated field.  
                updateRow[2] = valueDict[keyValue]  
                updateRows.updateRow(updateRow)    
    del valueDict  
    print ("Finished data transfer: " + strftime("%Y-%m-%d %H:%M:%S"))
#     log.info("Finished data transfer: " + strftime("%Y-%m-%d %H:%M:%S"))

# find attribute level differences in two identical data frames
#Gonna have to make this a compound key as well - need to handle duplicate APNs?
@timer
def differenceDictionary(df1, df2, key_field, fields_to_ignore):
    #Generate a list of columns in common
    common_columns = list(set(df1.columns) & set(df2.columns))
    # keep only the common columns in both dataframes
    df1 = df1[common_columns]
    df2 = df2[common_columns]
    #Trim spaces
    df1 = df1.applymap(lambda x: x.strip() if isinstance(x, str) else x)
    df2 = df2.applymap(lambda x: x.strip() if isinstance(x, str) else x)
    
    #Force the column types to match
    for field in fields_to_ignore:
        df1 = df1.drop(field, axis=1)
        df2 = df2.drop(field, axis=1)
    for column in df2.columns:
        if df1[column].dtype != df2[column].dtype:
            print(column)
            print (df2[column].dtype)
            #This handles nulls
            if df1[column].dtype=='int64':
                df1[column]=df1[column].astype('Int64')
            df2.loc[:, column] = df2[column].astype(df1[column].dtype)
    #    
    df1 = df1.set_index(key_field)
    df2 = df2.set_index(key_field)
    df1.sort_index(inplace=True)
    df2.sort_index(inplace=True)
    common_columns = list(set(df1.columns) & set(df2.columns))
    df1 = df1[common_columns]
    df2 = df2[common_columns]
    diff_df = df1.compare(df2)
    #
    new_values =diff_df.loc[:,pd.IndexSlice[:,'other']].droplevel(1,axis=1)
    #
    dict_update = new_values.to_dict('index')
    #
    new_dict = {k: {a: b for a, b in v.items() if not pd.isnull(b)} 
                for k, v in dict_update.items()}
    keys_to_remove = []
    for outer_key, inner_dict in new_dict.items():
        inner_keys_to_remove = []
        for inner_key, value in inner_dict.items():
            if not value:
                inner_keys_to_remove.append(inner_key)
        for inner_key in inner_keys_to_remove:
            del inner_dict[inner_key]
        if not inner_dict:
            keys_to_remove.append(outer_key)

    for outer_key in keys_to_remove:
        del new_dict[outer_key]
    return new_dict

# use the differences dictionary to update attributes in feature service or feature class
@timer
def update_fc_from_dict(update_dict,key_field, fc):
    #This gets our update cursor down to fields that need to be updated
    update_fields = set(field for values in update_dict.values() for field in values.keys())
    # create a SQL query to filter the feature class based on the key field values
    key_field_values = tuple(update_dict.keys())
    print("Updating Attributes started: " + strftime("%Y-%m-%d %H:%M:%S"))
    # update the attributes using the nested dictionary
    apn_issues =list()
    with arcpy.da.UpdateCursor(fc, [key_field] + list(update_fields)) as cursor:
        total_count=0
        for row in cursor:
            key_field_value = row[0]
            if key_field_value in update_dict:
                try:
                    update_values = update_dict[key_field_value]
                    total_count +=1
                    if (total_count%1000)==0:
                        print (f"Updating row {total_count} at "+ strftime("%Y-%m-%d %H:%M:%S"))
                    for field, value in update_values.items():
                        index = cursor.fields.index(field)
                        row[index] = value
                    cursor.updateRow(row)
                        #print("Updated APN/Field: "+str(row[0])+" / "+str(field))
                except Exception as e:
                    apn_issues.append(key_field_value)
                    # Print the error message
                    print(f"Error updating {key_field_value}: {e}")
                    continue
    print("Updating Attributes Finished: " + strftime("%Y-%m-%d %H:%M:%S"))
    print(f"Total updated{total_count}")
    return apn_issues

#Seperated out into two functions so we can use this function to make old new table in SQL
@timer
def make_old_new_dataframe(old_feature_class, new_feature_class, TRPA_boundary, prefix_remove):
    df_old = pd.DataFrame.spatial.from_featureclass(old_feature_class)
    df_new = pd.DataFrame.spatial.from_featureclass(new_feature_class)
    df_merge = pd.merge(df_old, df_new,  how='outer', on=['APN'], indicator=True)
    df_merge.query('_merge!="both"', inplace=True)
    df_merge.loc[df_merge['_merge']=='right_only', 'Status']='New APN'
    # define Left Only as Old APNs
    df_merge.loc[df_merge['_merge']=='left_only', 'Status']='Old APN'
    df_merge.dropna(subset=['APN'], inplace=True) 
    df_merge = df_merge.loc[~df_merge['APN'].str.startswith(prefix_remove)]
    #
    date = time.strftime("%m%d%Y")
    df_merge['DiscoveryDate'] = date
    df_merge['DiscoveryDate'] = pd.to_datetime(df_merge['DiscoveryDate'], format='%m%d%Y')
    TRPA_BNDY_Fields = [col for col in df_merge.columns if 'WITHIN_TRPA_BNDY' in col]
    df_merge['TRPA_Boundary'] = df_merge[TRPA_BNDY_Fields].sum(axis=1)
    # final list of fields
    df_merge = df_merge[['APN','Status','DiscoveryDate','TRPA_Boundary']]
    if TRPA_boundary == 'Yes':
        df_merge = df_merge.loc[df_merge['TRPA_Boundary']>=1]
    return df_merge

# get the list of old and new parcels
#Think this through to handle duplicate APNs
@timer
def old_new_parcels_list(old_feature_class, 
                         new_feature_class, 
                         TRPA_boundary, 
                         prefix_remove, 
                         old_new):
    df_merge = make_old_new_dataframe(old_feature_class, new_feature_class, TRPA_boundary, prefix_remove)
    parcel_list = df_merge.loc[df_merge['Status']==old_new,'APN'].tolist()
    return parcel_list

#Identify differences between APNs that haven't changed
@timer
def return_matching_apns(feature_class_old, 
                         feature_class_new, 
                         parcels_ignore):
    dfOld = pd.DataFrame.spatial.from_featureclass(feature_class_old)
    dfOld = dfOld[~dfOld['APN'].isin(parcels_ignore['APN'])]
    dfNew = pd.DataFrame.spatial.from_featureclass(feature_class_new)
    dfNew = dfNew.loc[dfNew['WITHIN_TRPA_BNDY']==1]
    matching_apns  = pd.merge(dfOld, dfNew,  how='inner', on=['APN'])
    matching_apns =pd.unique(matching_apns['APN'])
    return matching_apns

# deletes parcels
@timer
def delete_old_parcels(featureLayer, oldAPNs):
    delete_count = 0
    with arcpy.da.UpdateCursor(featureLayer, ["APN"]) as cursor:
        for row in cursor:
            apn = row[0]
            if apn in oldAPNs:
                cursor.deleteRow()
                delete_count +=1
    print(f"{delete_count} rows deleted from {featureLayer}.")

# inserts new parcels
@timer
def insert_new_parcels(featureLayer, new_APNs, new_parcels, fields):
    new_count = 0
    where_clause = f"{arcpy.AddFieldDelimiters(featureLayer, 'APN')} IN "+str(tuple(new_APNs))
    print(where_clause)
    with arcpy.da.SearchCursor(new_parcels, fields, where_clause) as search_cursor:
    # Open an insert cursor to the destination feature class
        with arcpy.da.InsertCursor(featureLayer, fields) as insert_cursor:
            # insert the rows from the serach cursor
            for row in search_cursor:
                insert_cursor.insertRow(row)
                new_count +=1
                print(f"{new_count} rows inserted into {featureLayer}.")
                
# updates @SHAPE that aren't identical to existing shapes
@timer
def update_parcel_geometry(featureLayer, new_parcels):
    newShapes = arcpy.management.SelectLayerByLocation(
    in_layer=new_parcels,
    overlap_type="ARE_IDENTICAL_TO",
    select_features=featureLayer,
    search_distance=None,
    selection_type="NEW_SELECTION",
    invert_spatial_relationship="INVERT")
    # update SHAPE object with new value
    #Changed this to Jurisdiction to work with Parcel_Base
    updateFieldsList_key= ['APN', 'JURISDICTION']
    updateFieldsList_value = ['SHAPE@']
    sourceFieldsList_key = ['APN', 'JURISDICTION']
    sourceFieldsList_value = ['SHAPE@']
    fieldJoinCalc_multikey(featureLayer, updateFieldsList_key, updateFieldsList_value, newShapes, sourceFieldsList_key, sourceFieldsList_value)

    # Get the count of selected features
    result = arcpy.management.GetCount(newShapes)
    count = int(result.getOutput(0))
    # number of shapes shifted
    print(f"{count} shapes shifted.")
    
@timer
def generate_spatial_dataframe(feature_class, data_type_mapping, fields_to_exclude): 
    # Get the field names and data types
    fields = arcpy.ListFields(feature_class)
    field_names = [field.name for field in fields if field.name not in fields_to_exclude]
    field_data_types = {field.name: field.type for field in fields if field.name not in fields_to_exclude}

    # Create a dictionary to store the data
    data = {}

    # Iterate through the rows and populate the dictionary
    with arcpy.da.SearchCursor(feature_class, field_names) as cursor:
        for row in cursor:
            for i, field_name in enumerate(field_names):
                if field_name not in data:
                    data[field_name] = []
                data_type = data_type_mapping.get(field_data_types[field_name], str)
                if row[i] is not None:
                    data[field_name].append(data_type(row[i]))
                else:
                    data[field_name].append(row[i])

    # Create a pandas DataFrame from the dictionary
    df = pd.DataFrame(data)

    return df

def get_text_fields(feature_class):
    field_list = []
    fields = arcpy.ListFields(feature_class)
    for field in fields:
        if field.type == 'String':
            field_list.append(field.name)
    return field_list

# Parcel AOI to select parcels to keep (includes TRPA Boundary and Olympic Valley Watershed)
parcelAOI = "Parcel_AOI"

#sde feature classes to use in attribution stage
sde_Impervious       = sdeBase + "\\sde.SDE.Impervious\\sde.SDE.Impervious_2019"
sde_Bailey           = sdeBase + "\\sde.SDE.Soils\sde.SDE.land_capability_Bailey_Soils"
sde_RegionalLandUse  = os.path.join(sdeBase,"sde.SDE.Planning/sde.SDE.RegionalLandUse")
sde_NRCSSoils1974    = sdeBase + "\\sde.SDE.Soils\\sde.SDE.NRCS_Soils_1974"
sde_NRCSSoils2003    = sdeBase + "\\sde.SDE.Soils\\sde.SDE.NRCS_Soils_2003"
sde_Catchment        = sdeBase + "\\sde.SDE.WaterQuality\\sde.SDE.TMDL_Catchment"
sde_HydroArea        = sdeBase + "\\sde.SDE.Water\\sde.SDE.Hydro_Areas"
sde_Watershed        = sdeBase + "\\sde.SDE.Water\\sde.SDE.Watershed"
sde_FireDistrict     = sdeBase + "\\sde.SDE.Jurisdictions\\sde.SDE.FireDistricts"
sde_LocalPlan        = sdeBase + "\\sde.SDE.Planning\\sde.SDE.LocalPlan"
sde_SpecialDistrict  = sdeBase + "\\sde.SDE.Planning\\sde.SDE.SpecialPlanningDistrict"
sde_CSLT             = sdeBase + "\\sde.SDE.Jurisdictions\\sde.SDE.CSLT"
sde_CurrentParcels   = sdeBase + "\\sde.SDE.Parcels\\sde.SDE.Parcel_Master"
sde_Zoning           = sdeBase + "\\sde.SDE.Planning\\sde.SDE.District"
sde_TownCenter       = sdeBase + "\\sde.SDE.Planning\\sde.SDE.TownCenter"
sde_TownCenterBuffer = sdeBase + "\\sde.SDE.Planning\\sde.SDE.TownCenter_Buffer"
sde_Index1987        = sdeBase + "\\sde.SDE.Index\\sde.SDE.AssessorMapIndex_1987"
sde_TRPAboundary     = sdeBase + "\\sde.SDE.Jurisdictions\\sde.SDE.TRPA_bdy"
sde_BonusUnitboundary= sdeBase + "\\sde.SDE.Planning\\sde.SDE.Bonus_unit_boundary"
sde_UrbanArea        = sdeBase + "\\sde.SDE.Jurisdictions\\sde.SDE.UrbanAreas"
sde_Zip              = sdeBase + "\\sde.SDE.Jurisdictions\\sde.SDE.Postal_ZIP"
sde_TAZ              = sdeBase + "\\sde.SDE.Transportation\\sde.SDE.Transportation_Analysis_Zone"
sde_Littoral         = sdeBase + "\\sde.SDE.Shorezone\\sde.SDE.LittoralParcel"
sde_Tolerance        = sdeBase + "\\sde.SDE.Shorezone\\sde.SDE.Tolerance_District"

# in memory fcs to use in the attribution stage
memory = "memory" + "\\"
ParcelPoint_RegionalLandUse = memory + "ParcelPoint_RegionalLandUse"
ParcelPoint_Soils74         = memory + "ParcelPoint_Soils74"
ParcelPoint_Soils03         = memory + "ParcelPoint_Soils03"
ParcelPoint_Catchment       = memory + "ParcelPoint_Catchment"
ParcelPoint_HydroArea       = memory + "ParcelPoint_HydroArea"
ParcelPoint_Watershed       = memory + "ParcelPoint_Watershed"
ParcelPoint_FireDistrict    = memory + "ParcelPoint_FireDistrict"
ParcelPoint_LocalPlan       = memory + "ParcelPoint_LocalPlan"
ParcelPoint_TownCenter      = memory + "ParcelPoint_TownCenter"
ParcelPoint_TownCenterBuffer= memory + "ParcelPoint_TownCenterBuffer"
ParcelPoint_Zoning          = memory + "ParcelPoint_Zoning"
ParcelPoint_SpecialDistrict = memory + "ParcelPoint_SpecialDistrict"
ParcelPoint_Index1987       = memory + "ParcelPoint_Index1987"
ParcelPoint_PstlTown        = memory + "ParcelPoint_PstlTown"
ParcelPoint_PstlZip         = memory + "ParcelPoint_PstlZip"
ParcelPoint_CSLT            = memory + "ParcelPoint_CSLT"
ParcelPoint_TAZ             = memory + "ParcelPoint_TAZ"
ParcelPoint_Design          = memory + "ParcelPoint_Design"
ParcelPoint_Littoral        = memory + "ParcelPoint_Littoral"
ParcelPoint_Tolerance       = memory + "ParcelPoint_Tolerance"

# Set up fields to add to FGDB.
baseFields = [
# apn ppno
['APN_TRPA', 'TEXT', 'APN', 50],
['PPNO_TRPA', 'DOUBLE','PPNO'],
['JURISDICTION_TRPA', 'TEXT', 'Jurisdiction', 4],
['COUNTY_TRPA', 'TEXT', 'County', 2],
 # parcel address   
['HSE_NUMBR_TRPA', 'TEXT', 'House Number', 25],
['UNIT_NUMBR_TRPA', 'TEXT', 'Unit Number', 50],
['STR_DIR_TRPA', 'TEXT','Street Direction', 5],
['STR_NAME_TRPA', 'TEXT', 'Street Name', 100],
['STR_SUFFIX_TRPA', 'TEXT', 'Street Suffix', 6],
['APO_ADDRESS_TRPA', 'TEXT', 'Full Address', 100],
['PSTL_TOWN_TRPA', 'TEXT', 'Postal Town', 25],
['PSTL_STATE_TRPA', 'TEXT', 'Postal State', 2],
['PSTL_ZIP5_TRPA', 'TEXT', 'Postal Zip Code', 5],
# owner info
['OWN_FIRST_TRPA', 'TEXT', 'Owner First Name', 255],
['OWN_LAST_TRPA', 'TEXT', 'Owner Last Name', 255],
['OWN_FULL_TRPA', 'TEXT', 'Owner Name', 255],
    # swap this in soon
# ['OWNER_NAME_TRPA', 'TEXT', 'Owner Name', 255],
['MAIL_ADD1_TRPA', 'TEXT', 'Mailing Address', 100],
['MAIL_CITY_TRPA', 'TEXT', 'Mailing City', 50],
['MAIL_STATE_TRPA', 'TEXT', 'Mailing State', 25],
['MAIL_ZIP5_TRPA', 'TEXT', 'Mailing Zip Code', 5],
# value fields  
['AS_LANDVALUE_TRPA', 'LONG','Assessed Land Value'],
['AS_IMPROVALUE_TRPA', 'LONG','Assessed Improved Value'],
['AS_SUM_TRPA', 'LONG', 'Assessed Sum Value'],
['TAX_LANDVALUE_TRPA', 'LONG','Tax Land Value'],
['TAX_IMPROVALUE_TRPA', 'LONG','Tax Improved Value'],
['TAX_SUM_TRPA', 'LONG','Tax Sum'],
['TAX_YEAR_TRPA', 'TEXT','Tax Year', 5],
# jurisdiction land use fields
['COUNTY_LANDUSE_CODE_TRPA', 'TEXT', 'County Landuse Code', 50],
['COUNTY_LANDUSE_TRPA', 'TEXT', 'County Landuse', 250],
# Fields for building info
["YEAR_BUILT_TRPA", "SHORT", 'Year Built', 5],
['UNITS_TRPA', 'DOUBLE', 'Units', 5],
["BEDROOMS_TRPA", "DOUBLE",'Bedrooms'],
['BATHROOMS_TRPA', 'DOUBLE', 'Bathrooms'],
['BUILDING_SQFT_TRPA', 'DOUBLE', 'Building Size'],
# fields to add? 
["VHR_TRPA", "TEXT", "Vacation Home Rental", 3],
["HOA_TRPA", "TEXT", "Home Owners Association", 3]
]

trpaFields = [
# land use
['OWNERSHIP_TYPE_TRPA', 'TEXT', 'Ownership Type', 50],
['EXISTING_LANDUSE_TRPA', 'TEXT', 'Existing Landuse', 50],
['REGIONAL_LANDUSE_TRPA', 'TEXT', 'Regional Landuse', 50], 
# Fields for soil, watershed, etc...
['ESTIMATED_COVERAGE_ALLOWED_TRPA', 'DOUBLE', "Estimate of Coverage Allowed (Bailey, sq.ft.)"],
['IMPERVIOUS_SURFACE_SQFT_TRPA', 'DOUBLE', "Impervious Surface (Remote Sensing, sq.ft.)"],
['SOIL_1974_TRPA', 'TEXT','NRCS Soils 1974', 5],
["SOIL_2003_TRPA", "TEXT", "NRCS Soils 2003", 5],
["CATCHMENT_TRPA", "TEXT", "Catchment", 150],
["HRA_NAME_TRPA", "TEXT", "Hydrologic Resource Area", 30],
["WATERSHED_NUMBER_TRPA", "SHORT", "Watershed Number"],
["WATERSHED_NAME_TRPA", "TEXT", "Watershed Name", 30],
["PRIORITY_WATERSHED_TRPA", "TEXT", "Priority Watershed", 2],
["FIREPD_TRPA", "TEXT", "Fire Protection District", 25],
# Fields for Planning purposes
["PLAN_ID_TRPA", "TEXT", 'Plan ID',8],
["PLAN_NAME_TRPA", "TEXT", 'Plan Name', 40],
["PLAN_TYPE_TRPA", "TEXT", 'Plan Type', 40],
["ZONING_ID_TRPA", "TEXT", 'Zoning ID', 50],
["ZONING_DESCRIPTION_TRPA", "TEXT", 'Zoning Description',500],
["TOWN_CENTER_TRPA", "TEXT",'Town Center', 50],
["LOCATION_TO_TOWNCENTER_TRPA", "TEXT", 'Location Relative to Town Center', 50],
["TOLERANCE_ID_TRPA", "TEXT", 'Tolerance ID', 50],
["TAZ_TRPA", "DOUBLE",'Transportation Analysis Zone'],
["INDEX_1987_TRPA", "TEXT", "1987 Parcel Map Index",10],
["LITTORAL_TRPA", "SHORT", "Littoral"],
["WITHIN_TRPA_BNDY_TRPA", "SHORT","Within TRPA Boundary?"],
["WITHIN_BONUSUNIT_BNDY_TRPA", "SHORT", "Within Bonus Unit Boundary"],
["LOCAL_PLAN_HYPERLINK_TRPA", "TEXT", "Local Plan Hyperlink", 255],
["DESIGN_GUIDELINES_HYPERLINK_TRPA", "TEXT", "Design Guidelines", 255],
["LTINFO_HYPERLINK_TRPA", "TEXT", "LTinfo Parcel Details", 255],
["INDEX_1987_HYPERLINK_TRPA", "TEXT", "Index 1987 Hyperlink", 255],
# Fields for Parcel Size
["PARCEL_ACRES_TRPA", "DOUBLE", "Acres"],
["PARCEL_SQFT_TRPA", "DOUBLE", "Square Feet"] 
]

### Logging

In [40]:
print(sdeBase)
print(sde_RegionalLandUse)

//Trpa-fs01/GIS/PARCELUPDATE/Workspace/Vector.sde/
//Trpa-fs01/GIS/PARCELUPDATE/Workspace/Vector.sde/sde.SDE.Planning/sde.SDE.RegionalLandUse


In [3]:
# create logger
logger = logging.getLogger(__name__)
# set log level for all handlers to debug
logger.setLevel(logging.DEBUG)

# create console handler and set level to debug
# best for development or debugging
consoleHandler = logging.StreamHandler()
consoleHandler.setLevel(logging.DEBUG)

# create formatter
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')

# add formatter to ch
consoleHandler.setFormatter(formatter)

# add ch to logger
logger.addHandler(consoleHandler)

###################
# USE LOGGER
###################

# example usage
logger.debug('debug message')
logger.info('info message')
logger.warning('warn message')
logger.error('error message')
logger.critical('critical message')

2023-05-10 08:57:58,927 - __main__ - DEBUG - debug message
2023-05-10 08:57:58,929 - __main__ - INFO - info message
2023-05-10 08:57:58,930 - __main__ - WARNING - warn message
2023-05-10 08:57:58,931 - __main__ - ERROR - error message
2023-05-10 08:57:58,932 - __main__ - CRITICAL - critical message


In [6]:
parcelLog = 'ParcelETL_'+str(time.strftime("%m%d%Y"))+'.log'
print(parcelLog)

ParcelETL_05102023.log


In [7]:

###################
# SETUP LOGGER
###################
# create logger
logger = logging.getLogger(__name__)
# set log level for all handlers to debug
logger.setLevel(logging.DEBUG)

# create console handler and set level to debug
consoleHandler = logging.StreamHandler()
consoleHandler.setLevel(logging.DEBUG)

# create file handler and set level to debug
parcelLog = 'ParcelETL_'+str(time.strftime("%m%d%Y"))+'.log'
fileHandler = logging.FileHandler(parcelLog)
fileHandler.setLevel(logging.DEBUG)

# create formatter
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')

# add formatter to handlers
consoleHandler.setFormatter(formatter)
fileHandler.setFormatter(formatter)

# add handlers to logger
logger.addHandler(consoleHandler)
logger.addHandler(fileHandler)

# set log level for development
logger.setLevel(logging.INFO)

###################
# USE LOGGER
###################

# context/ session information
context_user = "User XYZ"
a = 50
b = 50

# calculation
logger.debug("User %s provided the numbers %s and %s for calculation", context_user, a, b)
try:
    result = a / b
    logger.info("Calculation successful for user %s with a result of: %s", context_user, result)
except ZeroDivisionError as error:
    logger.error("Calculation was unsuccessful for user %s with the inputs %s and %s", context_user, a, b)
    logger.error(error)

2023-05-10 09:00:29,265 - __main__ - INFO - Calculation successful for user User XYZ with a result of: 1.0
2023-05-10 09:00:29,265 - __main__ - INFO - Calculation successful for user User XYZ with a result of: 1.0


In [9]:
# Set logging.
def setup_logging(log_filename,logging_level):
    log_format = "%(asctime)s %(levelname)-8s %(message)s"
    log_date_format = "%a, %d %b %Y %H:%M:%S"
    logging.basicConfig(filename=log_filename,
                        filemode='a',
                        level=logging_level,
                        format=log_format,
                        datefmt=log_date_format)
    
# Set up logging.
logDate = time.strftime("%Y%m%d")
logFilename = os.path.join(workspace, parcelLog)
setup_logging(logFilename, 'INFO')
logging.info('Starting Script')

## Extract

### Carson City County

In [2]:
## Carson City County GET Data
# parameters for get data from rest service
params = {'where': '1=1', 'outFields': '*', 'f': 'pjson', 'returnGeometry': True}
r = requests.get('https://gis.carson.org/arcgis/rest/services/CarsonCity/CarsonCityNV_OpenData/FeatureServer/36/query', params)
data = r.json()

# save JSON as a Feature class
json_path = os.path.join(workspace,'CCtemp.json')

# delete existing/old json file
os.remove(json_path)

# open and write data to json file
with open(json_path, 'w') as f:
    json.dump(data, f)

# delete the existing table
arcpy.management.Delete('Parcel_CC_Features')
print("Deleted existing table")

# json object to table
arcpy.JSONToFeatures_conversion(json_path, 'Parcel_CC_Features')
print("Saved CC Staging Feature class")

# get data from rest service
params = {'where': '1=1', 'outFields': '*', 'f': 'pjson', 'returnGeometry': True}
r = requests.get('https://gis.carson.org/arcgis/rest/services/CarsonCity/CarsonCityNV_OpenData/FeatureServer/42/query', params)
data = r.json()

# save JSON as a Feature class
json_path = os.path.join(workspace,'CCtemp.json')

# delete existing/old json file
os.remove(json_path)

# open and write data to json file
with open(json_path, 'w') as f:
    json.dump(data, f)

# delete the existing table
arcpy.management.Delete('Parcel_CC_Table')
print("Deleted existing table")

# json object to table
arcpy.JSONToFeatures_conversion(json_path, 'Parcel_CC_Table')
print("Saved CC Staging Table")


# The qualifiedFieldNames environment is used by Copy Features when persisting 
# the join field names.
arcpy.env.qualifiedFieldNames = False

# Set local variables
inFeatures = "Parcel_CC_Features"
joinTable  = "Parcel_CC_Table"
joinField  = "APN"
outFeature = "Parcel_CC_Extracted"

# Join the feature layer to a table
cc_join = arcpy.management.AddJoin(inFeatures, 
                                           joinField, 
                                           joinTable, 
                                           joinField)

# Copy the joined layer to a new permanent feature class
arcpy.management.CopyFeatures(cc_join, outFeature)
print("Carson Parcels Extracted")

Deleted existing table
Saved CC Staging Feature class
Deleted existing table
Saved CC Staging Table
Carson Parcels Extracted


### Douglas County

In [3]:
baseURL = "https://gisservices.douglasnv.us/server/rest/services/TRPA_Parcels/FeatureServer/0"
fields = "*"
outfc = "Parcel_DG_Extracted"

# Get record extract limit
urlstring = baseURL + "?f=json"
j = urllib.request.urlopen(urlstring)
js = json.load(j)
maxrc = int(js["maxRecordCount"])
print("Record extract limit: %s" % maxrc)

# Get object ids of features
where = "1=1"
urlstring = baseURL + "/query?where={}&returnIdsOnly=true&f=json".format(where)
j = urllib.request.urlopen(urlstring)
js = json.load(j)
idfield = js["objectIdFieldName"]
idlist = js["objectIds"]
idlist.sort()
numrec = len(idlist)
print("Number of target records: %s" % numrec)

# Gather features
print ("Gathering records...")
fs = dict()
for i in range(0, numrec, maxrc):
    torec = i + (maxrc - 1)
    if torec > numrec:
        torec = numrec - 1
    fromid = idlist[i]
    toid = idlist[torec]
    where = "{} >= {} and {} <= {}".format(idfield, fromid, idfield, toid)
    print ("  {}".format(where))
    urlstring = baseURL + "/query?where={}&returnGeometry=true&outFields={}&f=json".format(where,fields)
    # build that feature set!
    fs[i] = arcpy.FeatureSet()
    fs[i].load(urlstring)

# Save features
print("Saving features...")
fslist = []
for key,value in fs.items():
    fslist.append(value)
arcpy.Merge_management(fslist, outfc)

print("Douglas Parcels Extracted")

Record extract limit: 10000
Number of target records: 30134
Gathering records...
  OBJECTID >= 964197 and OBJECTID <= 974196
  OBJECTID >= 974197 and OBJECTID <= 984196
  OBJECTID >= 984197 and OBJECTID <= 994196
  OBJECTID >= 994197 and OBJECTID <= 994330
Saving features...
Douglas Parcels Extracted


### El Dorado County

In [6]:
# Set up Zip path.
zipPath = workspace
# setup output feature class
outfc = "Parcel_EL_Extracted"

# Check if zip from failed attempt still exists
existingZip = pathlib.Path(zipPath + r"\zipfolder")
if existingZip.exists():
    shutil.rmtree(zipPath + r"\zipfolder")
    logging.info('Previous zip folder deleted')

# Setup the params for the extraction GP tool. The boundary is a polygon the grabs the whole county.
payload = {'f': 'json', 'env:outSR': '6418', 'Layers_to_Clip': '["Parcels"]', 'Area_of_Interest': '{"geometryType":"esriGeometryPolygon","features":[{"geometry":{"rings":[[[-13490599.294393552,4646257.881632805],[-13490599.294393552,4735689.204726496],[-13336502.24537058,4735689.204726496],[-13336502.24537058,4646257.881632805],[-13490599.294393552,4646257.881632805]]],"spatialReference":{"wkid":102100}}}],"sr":{"wkid":102100}}', 'Feature_Format': 'File Geodatabase - GDB - .gdb'}

# Make the request to the GP service.
logging.info('Requesting parcels from EDC')
job = requests.get(r"https://see-eldorado.edcgov.us/arcgis/rest/services/uGOTNETandEXTRACTS/geoservices/GPServer/Extract%20Data%20Task/submitJob",params=payload)
jobJson = job.json()

# Check to make sure the job was accepted and get the JobID.
if 'jobId' in jobJson:
    jobID = jobJson['jobId']
    jobStatus = jobJson['jobStatus']
    jobURL = r"https://see-eldorado.edcgov.us/arcgis/rest/services/uGOTNETandEXTRACTS/geoservices/GPServer/Extract%20Data%20Task/jobs"
    if jobStatus == 'esriJobSubmitted' or jobStatus == 'esriJobExecuting':
        logging.info('EDC job submitted')

    # Check the status of the job, when done grab the resulting ZIP file link.
    while jobStatus == 'esriJobSubmitted' or jobStatus == 'esriJobExecuting':
        time.sleep(5)
        jobCheck = requests.get(jobURL+"/"+jobID+"?f=json")
        jobJson = jobCheck.json()
        if 'jobStatus' in jobJson:
            jobStatus = jobJson['jobStatus']
            if jobStatus == "esriJobSucceeded":
                if 'results' in jobJson:
                    logging.info('EDC server job completed')
                    resultURL = jobJson['results']['Output_Zip_File']['paramUrl']

                    # Grab the ZIP link.
                    logging.info('Downloading ZIP from EDC')
                    jobResult = requests.get(jobURL+"/"+jobID+r"/"+resultURL+r"?f=json&returnType=data")
            if jobStatus == "esriJobFailed":
                logging.error('EDC server job failure')
                if 'messages' in jobJson:
                    logging.error(jobJson['messages'])
                raise ValueError('EDC job failed!')

# Get the ZIP file.
parcelsZip = requests.get(jobResult.json()['value']['url'])
logging.info('Downloaded ZIP from EDC')

# Save the ZIP into memory.
zipFile = ZipFile(BytesIO(parcelsZip.content))

# Unzip the ZIP to the defined path.
for each in zipFile.namelist():
    if not each.endswith('/'):
        root, name = os.path.split(each)
        directory = os.path.normpath(os.path.join(zipPath, root))
        if not os.path.isdir(directory):
            os.makedirs(directory)
        open(os.path.join(directory, name), 'wb').write(zipFile.read(each))
logging.info('Unzipped files in ' + str(zipPath))

# Setup env for parcel FGDB and set overwrite to true.
zipFolder = zipPath + r"\zipfolder"
# arcpy.env.overwriteOutput = True
in_features = os.path.join(workspace, "zipfolder\data.gdb\Parcels")

# Export to staging gdb
arcpy.management.CopyFeatures(in_features, outfc)
print("El Dorado Parcels Extracted")

El Dorado Parcels Extracted


### Placer County 

In [7]:
#Parameters
hostedFeatureService = 'true'
agsService = 'false'

# username and password to get the token via the AGOL shared group
# username = 'mbindl'
# password = getpass.getpass()
username = 'TRPA_ADMIN'
password = 'TRP@g1sT3am'

baseURL = "https://services9.arcgis.com/NENkjkswKTzMfG3A/arcgis/rest/services/Parcels_with_Mega/FeatureServer/0"
fields = "*"
outdata = 'Parcel_PL_Extracted'
token = ''

# Disable warnings
requests.packages.urllib3.disable_warnings()

#Report error function
def PrintException():
    exc_type, exc_obj, tb = sys.exc_info()
    f = tb.tb_frame
    lineno = tb.tb_lineno
    filename = f.f_code.co_filename
    linecache.checkcache(filename)
    line = linecache.getline(filename, lineno, f.f_globals)
    arcpy.AddError('Error:  Line {} -- "{}": {}'.format(lineno, line.strip(), exc_obj))
    sys.exit()

#generate token for AGOL Hosted Feature Service

if username and password:
    try:
        tokenURL = 'https://www.arcgis.com/sharing/rest/generateToken'
        params = {'f': 'pjson', 'username': username, 'password': password, 'referer': 'https://www.arcgis.com', 'expiration': str(21600)}
        response = requests.post(tokenURL, data = params, verify = False)
        token = response.json()['token']
    except:
        PrintException()
else:
    token = ''

print('Token: '+token)

# Get record extract limit 
urlstring = baseURL + "?token="+token+"&f=json" 
j = requests.get(urlstring, verify=False)
js = j.json() 
maxrc = int(js["maxRecordCount"]) 
print("Record extract limit: %s" % maxrc)

# Get object ids of features
where = "1%3D1"
urlstring = baseURL + "/query?where=1%3D1&returnIdsOnly=true&f=json&token="+token
j = requests.get(urlstring, verify=True)
js = j.json() 
idfield = js["objectIdFieldName"]
idlist = js["objectIds"]
idlist.sort()
numrec = len(idlist)
print("Number of target records: %s" % numrec)

# Gather features
print ("Gathering records...")
fs = {}
for i in range(0, numrec, maxrc):
    torec = i + (maxrc - 1)
    if torec > numrec:
        torec = numrec - 1
    fromid = idlist[i]
    toid = idlist[torec]
    where = "{} >= {} and {} <= {}".format(idfield, fromid, idfield, toid)
    print ("  {}".format(where))
    urlstring = baseURL + f'/query?where={where}&returnGeometry=true&outFields={fields}&f=json&token='+token
    fs[i] = arcpy.FeatureSet()
    fs[i].load(urlstring)

# Save features
print("Saving features...")
fslist = []
for key,value in fs.items():
    fslist.append(value)
arcpy.Merge_management(fslist, outdata)
print("Done")


Token: f-uYEbunyeKBEXT7UOJb_mRCQAugzfdih9wZLR3yW5dsGL-hl56BtZMoXNqcaEmSe7amIhk-5Curzo49pv0mRPc7yFdM2fyMCUK0XsNqP8Js3m1ppo0wPOArBUXsW_svOAN2CK2SZiQM8JPUcSQITOCobcqcx0zbSCIU7C9SJB5-NKZRMaeJlnd9MOCjP5ca
Record extract limit: 2000
Number of target records: 196297
Gathering records...
  OBJECTID >= 1 and OBJECTID <= 2000
  OBJECTID >= 2001 and OBJECTID <= 4000
  OBJECTID >= 4001 and OBJECTID <= 6000
  OBJECTID >= 6001 and OBJECTID <= 8000
  OBJECTID >= 8001 and OBJECTID <= 10000
  OBJECTID >= 10001 and OBJECTID <= 12000
  OBJECTID >= 12001 and OBJECTID <= 14000
  OBJECTID >= 14001 and OBJECTID <= 16000
  OBJECTID >= 16001 and OBJECTID <= 18000
  OBJECTID >= 18001 and OBJECTID <= 20000
  OBJECTID >= 20001 and OBJECTID <= 22000
  OBJECTID >= 22001 and OBJECTID <= 24000
  OBJECTID >= 24001 and OBJECTID <= 26000
  OBJECTID >= 26001 and OBJECTID <= 28000
  OBJECTID >= 28001 and OBJECTID <= 30000
  OBJECTID >= 30001 and OBJECTID <= 32000
  OBJECTID >= 32001 and OBJECTID <= 34000
  OBJECTID >= 340

In [19]:
#Parameters
hostedFeatureService = 'true'
agsService = 'false'

# ## Need to use TRPA Admin user for this ###
# username = 'mbindl'
# password = getpass.getpass()
username = 'TRPA_ADMIN'
password = 'TRP@g1sT3am'

baseURL = "https://services9.arcgis.com/NENkjkswKTzMfG3A/arcgis/rest/services/vwJurisdictionsGIS/FeatureServer/2"
fields = "*"
outdata = "Parcel_PL_Table"
token = ''

# Disable warnings
requests.packages.urllib3.disable_warnings()

#Report error function
def PrintException():
    exc_type, exc_obj, tb = sys.exc_info()
    f = tb.tb_frame
    lineno = tb.tb_lineno
    filename = f.f_code.co_filename
    linecache.checkcache(filename)
    line = linecache.getline(filename, lineno, f.f_globals)
    arcpy.AddError('Error:  Line {} -- "{}": {}'.format(lineno, line.strip(), exc_obj))
    sys.exit()

#generate token for AGOL Hosted Feature Service

if username and password:
    try:
        tokenURL = 'https://www.arcgis.com/sharing/rest/generateToken'
        params = {'f': 'pjson', 'username': username, 'password': password, 'referer': 'https://www.arcgis.com', 'expiration': str(21600)}
        response = requests.post(tokenURL, data = params, verify = False)
        token = response.json()['token']
    except:
        PrintException()
else:
    token = ''

print('Token: '+token)

# Get record extract limit 
urlstring = baseURL + "?token="+token+"&f=json" 
j = requests.get(urlstring, verify=False)
js = j.json() 
maxrc = int(js["maxRecordCount"]) 
print("Record extract limit: %s" % maxrc)

# Get object ids of features
where = "1%3D1"
urlstring = baseURL + "/query?where=1%3D1&returnIdsOnly=true&f=json&token="+token
j = requests.get(urlstring, verify=True)
js = j.json() 
idfield = js["objectIdFieldName"]
idlist = js["objectIds"]
idlist.sort()
numrec = len(idlist)
print("Number of target records: %s" % numrec)

# Gather features
print ("Gathering records...")
fs = {}
for i in range(0, numrec, maxrc):
    torec = i + (maxrc - 1)
    if torec > numrec:
        torec = numrec - 1
    fromid = idlist[i]
    toid = idlist[torec]
    where = "{} >= {} and {} <= {}".format(idfield, fromid, idfield, toid)
    print ("  {}".format(where))
    urlstring = baseURL + f'/query?where={where}&outFields={fields}&f=json&token='+token
    fs[i] = arcpy.RecordSet()
    fs[i].load(urlstring)

# Save features
print("Saving features...")
fslist = []
for key,value in fs.items():
    fslist.append(value)
arcpy.Merge_management(fslist, outdata)
print("Done")


Token: Y8vQwV1e8TARwfjB3lFBV7p-5MQdFIEeWpudZJ994PKCscnyasJqSGm7PlxP13pAlldJEPwcVhu1Ywxuh-sEP-4WIRgR9e3x5kctGWXHjeIcVWPE7wMlWZtJCObPLcBnCV6H2UYS1wFS_xNPWy2V7YHeGT4x4dp3vMLsbJm3VVYWfGURfCT_sMxmBLFzMry3
Record extract limit: 2000
Number of target records: 172423
Gathering records...
  ESRI_OID >= 1 and ESRI_OID <= 2000
  ESRI_OID >= 2001 and ESRI_OID <= 4000
  ESRI_OID >= 4001 and ESRI_OID <= 6000
  ESRI_OID >= 6001 and ESRI_OID <= 8000
  ESRI_OID >= 8001 and ESRI_OID <= 10000
  ESRI_OID >= 10001 and ESRI_OID <= 12000
  ESRI_OID >= 12001 and ESRI_OID <= 14000
  ESRI_OID >= 14001 and ESRI_OID <= 16000
  ESRI_OID >= 16001 and ESRI_OID <= 18000
  ESRI_OID >= 18001 and ESRI_OID <= 20000
  ESRI_OID >= 20001 and ESRI_OID <= 22000
  ESRI_OID >= 22001 and ESRI_OID <= 24000
  ESRI_OID >= 24001 and ESRI_OID <= 26000
  ESRI_OID >= 26001 and ESRI_OID <= 28000
  ESRI_OID >= 28001 and ESRI_OID <= 30000
  ESRI_OID >= 30001 and ESRI_OID <= 32000
  ESRI_OID >= 32001 and ESRI_OID <= 34000
  ESRI_OID >= 340

In [32]:
# The qualifiedFieldNames environment is used by Copy Features when persisting 
# the join field names.
arcpy.env.qualifiedFieldNames = False

# Set local variables
inFeatures = "Parcel_PL_"
joinTable  = "Parcel_PL_Table"
joinField  = ""
outFeature = "Parcel_PL_"

# arcpy.management.CalculateField(inFeatures, "APN", 
#                                 '!APN!.replace("-","")', "PYTHON3")

# Join the feature layer to a table
pl_joined_table = arcpy.management.AddJoin(inFeatures, 
                                           joinField, 
                                           joinTable, 
                                           joinField)

# Copy the layer to a new permanent feature class
arcpy.management.CopyFeatures(pl_joined_table, outFeature)

# See field names and aliases
totalRecords = arcpy.management.GetCount(outFeature)
print('{} has {} records'.format(outFeature, totalRecords[0]))
resultFields = arcpy.ListFields(result)
print([field.name for field in resultFields])
print([field.aliasName for field in resultFields])

Parcel_PL_Extracted has 197408 records
['OBJECTID', 'Shape', 'FEEPARCEL', 'APN', 'GISAPN', 'BOOK', 'BOOK_PAGE', 'ROLL_YEAR', 'JURISDICTION', 'PARCEL_LEVEL', 'TRANSACTIO', 'TRA', 'PARCELTYPE', 'TAX_CD', 'TAXABLEX', 'TAX_DESC', 'USE_CD', 'USE_CD_N', 'ACRES', 'EFFECTIVEY', 'STR_SQFT', 'OWNER1', 'OWNER2', 'ADR1', 'ADR2', 'CITY', 'STATE', 'ZIP', 'STREETNUM', 'STREETNAME', 'STREETTYPE', 'STREETDIR', 'SP_APT', 'COMMUNITY', 'ASMT_DESC', 'LANDVALUE', 'STRUCTURE', 'NEIGHBORHOODCODE', 'APPRAISERID', 'SITUSID', 'ASMT', 'GIS_ACRES', 'OBJECTID_1', 'FEEPARCEL_1', 'TRANSACTIO_1', 'TRA_1', 'PARCELTYPE_1', 'TAX_CD_1', 'TAXABLEX_1', 'TAX_DESC_1', 'USE_CD_1', 'USE_CD_N_1', 'ACRES_1', 'EFFECTIVEY_1', 'STR_SQFT_1', 'OWNER1_1', 'OWNER2_1', 'ADR1_1', 'ADR2_1', 'CITY_1', 'STATE_1', 'ZIP_1', 'STREETNUM_1', 'STREETNAME_1', 'STREETTYPE_1', 'STREETDIR_1', 'SP_APT_1', 'COMMUNITY_1', 'ASMT_DESC_1', 'LANDVALUE_1', 'STRUCTURE_1', 'NEIGHBORHOODCODE_1', 'APPRAISERID_1', 'SITUSID_1', 'ASMT_1', 'Shape_Length', 'Shape_Area

### Washoe County 

In [8]:
baseURL = "https://wcgisweb.washoecounty.us/arcgis/rest/services/OpenData/OpenData/FeatureServer/0"
fields = "*"
outfc = "Parcel_WA_Extracted"

# Get record extract limit
urlstring = baseURL + "?f=json"
j = urllib.request.urlopen(urlstring)
js = json.load(j)
maxrc = int(js["maxRecordCount"])
print("Record extract limit: %s" % maxrc)

# Get object ids of features
where = "1=1"
urlstring = baseURL + "/query?where={}&returnIdsOnly=true&f=json".format(where)
j = urllib.request.urlopen(urlstring)
js = json.load(j)
idfield = js["objectIdFieldName"]
idlist = js["objectIds"]
idlist.sort()
numrec = len(idlist)
print("Number of target records: %s" % numrec)

# Gather features
print ("Gathering records...")
fs = dict()
for i in range(0, numrec, maxrc):
    torec = i + (maxrc - 1)
    if torec > numrec:
        torec = numrec - 1
    fromid = idlist[i]
    toid = idlist[torec]
    where = "{} >= {} and {} <= {}".format(idfield, fromid, idfield, toid)
    print ("  {}".format(where))
    urlstring = baseURL + "/query?where={}&returnGeometry=true&outFields={}&f=json".format(where,fields)
    # build that feature set!
    fs[i] = arcpy.FeatureSet()
    fs[i].load(urlstring)

# Save features
print("Saving features...")
fslist = []
for key,value in fs.items():
    fslist.append(value)
arcpy.Merge_management(fslist, outfc)
print("Done")

Record extract limit: 1000
Number of target records: 188746
Gathering records...
  OBJECTID >= 1 and OBJECTID <= 1000
  OBJECTID >= 1001 and OBJECTID <= 2000
  OBJECTID >= 2001 and OBJECTID <= 3000
  OBJECTID >= 3001 and OBJECTID <= 4000
  OBJECTID >= 4001 and OBJECTID <= 5000
  OBJECTID >= 5001 and OBJECTID <= 6000
  OBJECTID >= 6001 and OBJECTID <= 7000
  OBJECTID >= 7001 and OBJECTID <= 8000
  OBJECTID >= 8001 and OBJECTID <= 9000
  OBJECTID >= 9001 and OBJECTID <= 10000
  OBJECTID >= 10001 and OBJECTID <= 11000
  OBJECTID >= 11001 and OBJECTID <= 12000
  OBJECTID >= 12001 and OBJECTID <= 13000
  OBJECTID >= 13001 and OBJECTID <= 14000
  OBJECTID >= 14001 and OBJECTID <= 15000
  OBJECTID >= 15001 and OBJECTID <= 16000
  OBJECTID >= 16001 and OBJECTID <= 17000
  OBJECTID >= 17001 and OBJECTID <= 18000
  OBJECTID >= 18001 and OBJECTID <= 19000
  OBJECTID >= 19001 and OBJECTID <= 20000
  OBJECTID >= 20001 and OBJECTID <= 21000
  OBJECTID >= 21001 and OBJECTID <= 22000
  OBJECTID >= 220

Done


### Select Parcels to Keep for each County

In [9]:
# list of parcel staging layers to trim
parcelLayers = ["Parcel_CC_Extracted",
                "Parcel_DG_Extracted",
                "Parcel_EL_Extracted",
                "Parcel_PL_Extracted",
                "Parcel_WA_Extracted"]

# delete BS Parcels
parcelDelete = "ParcelDelete"

for parcel in parcelLayers:
    # Run MakeFeatureLayer
    arcpy.management.MakeFeatureLayer(parcel, parcelDelete)
    # select within clementini 
    arcpy.management.SelectLayerByLocation(parcelDelete, 
                                           "INTERSECT", 
                                           # includes TRPA Boundary and Olympic Valley Wateshed
                                           parcelAOI, '0', 
                                           "NEW_SELECTION", "INVERT")

    # Run GetCount and if some features have been selected, then 
    #  run DeleteFeatures to remove the selected features.
    deleteCount=arcpy.management.GetCount(parcelDelete)[0]
    if int(deleteCount) > 0:
        arcpy.management.DeleteFeatures(parcelDelete)
    # delete feature layer
    arcpy.management.Delete(parcelDelete)
    print("{} features deleted".format(deleteCount))

21111 features deleted
23846 features deleted
84154 features deleted
176602 features deleted
179330 features deleted


## Transform

### Carson City County

In [10]:
# get staging feature class and name output transformed feature class
in_features = "Parcel_CC_Extracted"
parcel_out  = "Parcel_CC_Transformed"

# in-memory feature class
carsonParcel = r"in_memory/inMemoryFeatureClass"

# copy feature class into in-memory feature class to work on
arcpy.management.CopyFeatures(in_features, carsonParcel)

# Add TRPA base fields
arcpy.management.AddFields(carsonParcel, baseFields)


# Do work.
with arcpy.da.UpdateCursor(carsonParcel, [
                                        ## TRPA base schema ##
                                        'APN_TRPA',                 #0
                                        'PPNO_TRPA',                #1
                                        'JURISDICTION_TRPA',        #2
                                         # parcel address   
                                        'HSE_NUMBR_TRPA',           #3
                                        'STR_DIR_TRPA',             #4
                                        'STR_NAME_TRPA',            #5
                                        'STR_SUFFIX_TRPA',          #6
                                        'UNIT_NUMBR_TRPA',          #7
                                        'APO_ADDRESS_TRPA',         #8
                                        'PSTL_TOWN_TRPA',           #9
                                        'PSTL_STATE_TRPA',          #10
                                        'PSTL_ZIP5_TRPA',           #11
                                        # owner fields
                                            # no first and last fields
                                        'OWN_FULL_TRPA',            #12
                                        'MAIL_ADD1_TRPA',           #13
                                        'MAIL_CITY_TRPA',           #14
                                        'MAIL_STATE_TRPA',          #15
                                        'MAIL_ZIP5_TRPA',           #16
                                        # value fields  
                                        'AS_LANDVALUE_TRPA',        #17
                                        'AS_IMPROVALUE_TRPA',       #18
                                        'AS_SUM_TRPA',              #19
                                        'TAX_LANDVALUE_TRPA',       #20 
                                        'TAX_IMPROVALUE_TRPA',      #21
                                        'TAX_SUM_TRPA',             #22
                                        'TAX_YEAR_TRPA',            #23
                                        # land use fields 
                                        'COUNTY_LANDUSE_CODE_TRPA', #24
                                        'COUNTY_LANDUSE_TRPA',      #25
                                        # Fields for building info
                                        "YEAR_BUILT_TRPA",          #26
                                        'UNITS_TRPA',               #27
                                        'BEDROOMS_TRPA',            #28
                                        'BATHROOMS_TRPA',           #29
                                        'BUILDING_SQFT_TRPA',       #30
                                        'VHR_TRPA',                 #31
                                        'HOA_TRPA',                 #32
                                        ###-------------------------###
                                        # County Fields to get data from
                                        'APN',   # apn              #33
                                        'APN_NUM',   # ppno         #34
                                        'Phy_Addr', #full adr       #35
                                        'Loc1', # house number      #36
                                        'Dir',# street dir          #37
                                        'Street_Name',# street name #38
                                        'Unit',  # unite Number     #39
                                        'Legal_Owner',  # Owner     #40
                                        'Mail_Addr',# mail address1 #41
                                        'Mail2_Addr',#mail address2 #42
                                        'MCity', # Mailing City     #43
                                        'MZip',  # Mailing Zip      #44
                                        'Land_Value',# land value   #45
                                        'Improv_Val',# improvedvalue#46
                                        'LU',    # land use code    #47
                                        'Total_DWUnits',  # units   #48

]) as cursor:
    # loop through each record and transform the values
    for row in cursor:
        # Set APN
        apn = row[33]
        if not (apn is None or apn == "" or apn.isspace() == True):
            row[0] = (apn[:3] + "-" + apn[3:6] + "-" + apn[6:8])
        else:
            row[0] = ''
            
        #PPNO
        ppno = row[34]
        if not (ppno is None):
            row[1] = int(ppno)
        else:
            row[1] = ''
            
        # Jurisdiction
        row[2] = "CC"
        
        # APO Address
        full_address = row[35]
        if not (full_address is None or full_address=='' or full_address.isspace()==True):
            row[8] = full_address
        else:
            row[8] = ''
        
        # House Number
        house = row[36]
        if not (house is None):
            row[3] = str(house)
        else:
            row[3] = ''
        
        # Street Direction
        street_direction = row[37]
        if not (street_direction is None or street_direction=='' or street_direction.isspace()==True):
            row[4] = street_direction
        else:
            row[4] = ''
            
        # Street Name
        street_name = row[38]
        if not (street_name is None or street_name =='' or street_name.isspace()==True):
            row[5] = street_name.split(" ",-1)[0]
        else:
            row[5] = ''
            
        # Street Suffix
        street_suffix = row[38]
        if not (street_suffix is None or street_suffix =='' or street_suffix.isspace()==True):
            row[6] = street_suffix.split(" ")[-1]
        else:
            row[6] = ''
            
        # Unit Number
        unit= row[39]
        if not (unit is None or unit=='' or unit.isspace()==True):
            row[7] = unit
        else:
            row[7] = ''
                    
        # Postal Town - see Search/Update Cursor below
        
        # Postal State
        row[10] = 'NV'
        
        # Postal Zip - See Search/Update Cursor below
        row[11] = ''    
        
        # Owner Name
        owner = row[40]
        if not (owner is None or owner == '' or owner.isspace()==True):
            row[12] = owner.strip()
        else:
            row[12] = ''

        # Mailing Address
        address1 = row[41]
        address2 = row[42]
        if not (address2 is None or address2 == '' or address2.isspace()==True):
            row[13] = str(address2).strip()
        elif (address2 is None or address2 == '' or address2.isspace()==True and address1 is None or address1 == '' or address1.isspace()==True):
            row[13] = str(address1).strip()
        else:
            row[13] = '' 
           
        # Mailing City
        mail_city = row[43]        
#         mail_city.split(',',1)[0]
        if not (mail_city is None or mail_city=='' or mail_city.isspace()==True):
            row[14] = mail_city.strip().split(',',1)[0]
        else:
            row[14] = ''
            
        # Mailing State
        mail_state = row[43]
        if not (mail_state is None or mail_state=='' or mail_state.isspace()==True):
            row[15] = mail_state.strip().rsplit(',')[-1]
        else:
            row[15] = ''
        
        # Mailing Zipcode
        mail_zip = row[44] 
        if (mail_zip is not None and len(mail_zip)>=5):
            row[16] = mail_zip[:5]
        else:
            row[16] = ''
            
        # Assessed Land Value
        land_value = row[45]
        if not(land_value is None):
            row[17] = land_value
        else:
            row[17] = 0
        
        # Assessed Improved Value
        improved_value = row[46]
        if not (improved_value is None):
            row[18] = improved_value
        else:
            row[18] = 0
                
        # Assessed Sum
        if not (land_value is None or improved_value is None):
            assessed_sum = improved_value + land_value
            row[19] = assessed_sum
        else:
            row[19] = None
        
        # Tax  Land Value
        taxland_value = row[45]
        if not(taxland_value is None):
            row[20] = taxland_value/0.35
        else:
            row[20] = None
        
        # Tax Improved Value
        taximproved_value = row[46]
        if not (taximproved_value is None):
            row[21] = taximproved_value/0.35
        else:
            row[21] = None
        
        # Tax Sum
        if not (land_value is None or improved_value is None):
            tax_sum = row[20]+row[21]
            row[22] = tax_sum
        else:
            row[22] = None
        
        # Tax Year
        tax_year =  datetime.now().year

        if not (tax_year is None):
            row[23] = tax_year
        else:
            row[23] = ''
            
        # County Land Use Code
        county_luc = row[47]
        if not (county_luc is None):
            row[24] = str(county_luc)
        else:
            row[24] = '' 
            
        # Units
        units = row[48]
        if not (units is None):
            row[27] = units
        else:
            row[27] = None
        
#         Update the row.
        cursor.updateRow(row)
del cursor

#arcpy.management.CopyFeatures(carsonParcel, parcel_out)

out_coordinate_system = arcpy.SpatialReference('NAD 1983 UTM Zone 10N') 
arcpy.Project_management(carsonParcel, parcel_out, out_coordinate_system)

print('New Carson Parcels transformed')

New Carson Parcels transformed


### Douglas County

In [11]:
# get staging feature class and name output trnasformed feature class
in_features = "Parcel_DG_Extracted"
parcel_out  = "Parcel_DG_Transformed"

# in-memory feature class
douglasParcel = r"in_memory/inMemoryFeatureClass"

# copy feature class into in-memory feature class to work on
arcpy.management.CopyFeatures(in_features, douglasParcel)

# Add TRPA base fields
arcpy.management.AddFields(douglasParcel, baseFields)


# Do work.
with arcpy.da.UpdateCursor(douglasParcel, [
                                        ## TRPA base schema ##
                                        'APN_TRPA',                 #0
                                        'PPNO_TRPA',                #1
                                        'JURISDICTION_TRPA',        #2
                                         # parcel address   
                                        'HSE_NUMBR_TRPA',           #3
                                        'STR_DIR_TRPA',             #4
                                        'STR_NAME_TRPA',            #5
                                        'STR_SUFFIX_TRPA',          #6
                                        'UNIT_NUMBR_TRPA',          #7
                                        'APO_ADDRESS_TRPA',         #8
                                        'PSTL_TOWN_TRPA',           #9
                                        'PSTL_STATE_TRPA',          #10
                                        'PSTL_ZIP5_TRPA',           #11
                                        # owner fields
                                            # no own first and last for DG
                                        'OWN_FULL_TRPA',            #12
                                        'MAIL_ADD1_TRPA',           #13
                                        'MAIL_CITY_TRPA',           #14
                                        'MAIL_STATE_TRPA',          #15
                                        'MAIL_ZIP5_TRPA',           #16
                                        # value fields  
                                        'AS_LANDVALUE_TRPA',        #17
                                        'AS_IMPROVALUE_TRPA',       #18
                                        'AS_SUM_TRPA',              #19
                                        'TAX_LANDVALUE_TRPA',       #20 
                                        'TAX_IMPROVALUE_TRPA',      #21
                                        'TAX_SUM_TRPA',             #22
                                        'TAX_YEAR_TRPA',            #23
                                        # land use fields 
                                        'COUNTY_LANDUSE_CODE_TRPA', #24
                                        'COUNTY_LANDUSE_TRPA',      #25
                                        # Fields for building info
                                        "YEAR_BUILT_TRPA",          #26
                                        'UNITS_TRPA',               #27
                                        'BEDROOMS_TRPA',            #28
                                        'BATHROOMS_TRPA',           #29
                                        'BUILDING_SQFT_TRPA',       #30
                                        'VHR_TRPA',                 #31
                                        'HOA_TRPA',                 #32
                                        ###-------------------------###
                                        # County Fields to get data from
                                        'APN',   # apn,ppno         #33
                                        'PLOC_', # house number     #34
                                        'PLOCDR',# street dir       #35
                                        'PLOCNM',# street name      #36
                                        'PLOCTP',# street suffix    #37
                                        'PLOCU_',# unit number      #38
                                        'PANAME',# owner name       #39
                                        'PMADD1',# mailing addr1    #40
                                        'PMADD2',# mailing addr2    #41
                                        'PMCTST',# city,state       #42
                                        'PZIP',  # zip              #43
                                        'YYEAR', # tax year         #44
                                        'YLDUSE',# land use code    #45
                                        'YLANDV',# land value       #46
                                        'YIMPRV',# improved value   #47
                                        'YEXMP', # tax exempt value #48
                                        'YNETV', # tax net value    #49
                                        'PCONYR',# year built       #50
                                        'PBEDS', # bedrooms         #51
                                        'PBATHS',# bathrooms        #52

    ### These are missing from the new service
    #                                         'PBLDSF',# building sqft    #
    #                                         'STREETADDR', #full adr     #
    #                                         'P_DWEL',# units            #
    #                                         'VHR',   # vhr yes?         #
    #                                         'HOA',   # hoa name         #
]) as cursor:
    # loop through each record and transform the values
    for row in cursor:
        # APN field
        # Get County value
        apn = str(row[33])
        if not (apn is None or apn == ""):
            row[0] =(apn[:4] + "-" + apn[4:6] + "-" + apn[6:9] + "-" + apn[9:12])
        else:
            row[0] = ""
            
        #PPNO
        ppno = row[33]
        if not (ppno is None):
            row[1] = int(ppno)
        else:
            row[1] = ''
            
        # Jurisdiction
        row[2] = "DG"
        
        # APO Address
        house            = str(row[34]).strip()
        street_direction = str(row[35]).strip()
        street_name      = str(row[36]).strip()
        street_suffix    = str(row[37]).strip()
        unit             = str(row[38]).strip()
        if not (street_name is None or street_name=='' or street_name.isspace()==True):
            row[8] = re.sub(" +"," ", (house + " " + street_direction +" " + street_name+" " + street_suffix+" " + unit).strip())
        else:
            row[8] = ''
        
        # House Number
        house = row[34]
        if not (house is None):
            row[3] = str(house)
        else:
            row[3] = ''
        
        # Street Direction
        street_direction = row[35]
        if not (street_direction is None or street_direction=='' or street_direction.isspace()==True):
            row[4] = street_direction
        else:
            row[4] = ''
            
        # Street Name
        street_name = row[36]
        if not (street_name is None or street_name =='' or street_name.isspace()==True):
            row[5] = street_name
        else:
            row[5] = ''
            
        # Street Suffix
        street_suffix = row[37]
        if not (street_suffix is None or street_suffix =='' or street_suffix.isspace()==True):
            row[6] = street_suffix
        else:
            row[6] = ''
            
        # Unit Number
        unit= row[38]
        if not (unit is None or unit=='' or unit.isspace()==True):
            row[7] = unit
        else:
            row[7] = ''
                    
        # Postal Town - see Search/Update Cursor below
        
        # Postal State
        row[10] = 'NV'
        
        # Postal Zip - See Search/Update Cursor below
        row[11] = ''    
        
        # Owner Name
        owner = row[39]
        if not (owner is None or owner == '' or owner.isspace()==True):
            row[12] = owner.strip()
        else:
            row[12] = ""

        # Mailing Address
        address1 = row[40].strip()
        address2 = row[41].strip()
        if not (address1 is None or address1=='' or address1.isspace()==True):
            row[13] = str(address1 + " " + address2)
        elif (address2 is None):
            row[13] = address1
        else:
            row[13] = ''
                   
        # Mailing City
        mail_city = str(row[42]).split(',',1)[0].strip()
        
        if not (mail_city is None or mail_city=='' or mail_city.isspace()==True):
            row[14] = mail_city
        else:
            row[14] = ''
            
        # Mailing State - Added logic to set anything that isn't 2 characters long to '' 
        mail_state = str(row[42]).rsplit(',')[-1].strip().split(' ',1)[0].strip()
        if not (mail_state is None or mail_state=='' or mail_state.isspace()==True or len(mail_state)!=2):
            row[15] = mail_state
        else:
            row[15] = ''
        
        # Mailing Zipcode
        mail_zip = row[43].strip()
        if not (mail_zip is None or mail_zip=='' or mail_zip.isspace()==True):
            row[16] = mail_zip[:5]
        else:
            row[16] = ''
            
        # Assessed Land Value
        land_value = row[46]
        if not(land_value is None):
            row[17] = land_value
        else:
            row[17] = ''
        
        # Assessed Improved Value
        improved_value = row[47]
        if not (improved_value is None):
            row[18] = improved_value
        else:
            row[18] = None
                
        # Assessed Sum
        if not (land_value is None or improved_value is None):
            assessed_sum = improved_value + land_value
            row[19] = assessed_sum
        else:
            row[19] = None
        
        # Tax  Land Value
        taxland_value = row[46]
        if not(taxland_value is None):
            row[20] = taxland_value/0.35
        else:
            row[20] = None
        
        # Tax Improved Value
        taximproved_value = row[47]
        if not (taximproved_value is None):
            row[21] = taximproved_value/0.35
        else:
            row[21] = None
        
        # Tax Sum
        if not (land_value is None or improved_value is None):
            tax_sum = row[49]
            row[22] = tax_sum
        else:
            row[22] = None
        
        # Tax Year
        tax_year = row[44]
        if not (tax_year is None):
            row[23] = tax_year
        else:
            row[23] = ''
            
        # County Land Use Code
        county_luc = row[45]
        if not (county_luc is None):
            row[24] = str(county_luc)
        else:
            row[24] = '' 
        
        # Year Built
        year_built = row[50]
        if not (year_built is None or year_built==''):
            row[26] = year_built
        else:
            row[26] = None
        
        # Bedrooms
        bedrooms = row[51]
        if not (bedrooms is None or bedrooms==''):
            row[28] = bedrooms
        else:
            row[28] = None
        # Bathrooms
        baths = row[52]
        if not (baths is None or baths==''):
            row[29] = baths
        else:
            row[29] = None
            
#         Update the row.
        cursor.updateRow(row)
del cursor

out_coordinate_system = arcpy.SpatialReference('NAD 1983 UTM Zone 10N') 
arcpy.Project_management(douglasParcel, parcel_out, out_coordinate_system)

print('New Douglas Parcels transformed')

New Douglas Parcels transformed


### Eldorado County

In [12]:
# get staging feature class to transform
in_features = "Parcel_EL_Extracted"
parcel_out  = "Parcel_EL_Transformed"

# in-memory feature class
eldoradoParcel = r"in_memory/inMemoryFeatureClass"

# copy feature class into in-memory feature class to work on
arcpy.management.CopyFeatures(in_features, eldoradoParcel)

# Add TRPA base fields
arcpy.management.AddFields(eldoradoParcel, baseFields)

# Set up the regex queries for the data.
# cityStateZipRegex = r'(.+?)\s([A-Z]{1,2})\s(.+?)$' - Keep in case new one doesn't work out long term.
cityStateZipRegex = r'(.+?)\s([A-Z]{1,2})\s(?=\d)(.*)'
poBoxRegex = r'([^x]+)\W(P\s*O BOX\W*[0-9]{1,6})'
addressRegex = r'(\d{1,5}\D+.+)'
canadaRegex = r'(.+?)\s([A-Z]{1,2})\s(CANADA)\s(.*)'
brazilRegex = r'(.+?)\s(BRAZIL)\s(.*)'

# Set up list for addresses with a country name in the mail_addr4 column.
countriesList = ['japan','canada']

# Transform County data to TRPA Schema
with arcpy.da.UpdateCursor(eldoradoParcel, [
                                        ## TRPA base schema ##
                                        'APN_TRPA',                 #0
                                        'PPNO_TRPA',                #1
                                        'JURISDICTION_TRPA',        #2
                                         # parcel address   
                                        'HSE_NUMBR_TRPA',           #3
                                        'STR_DIR_TRPA',             #4
                                        'STR_NAME_TRPA',            #5
                                        'STR_SUFFIX_TRPA',          #6
                                        'UNIT_NUMBR_TRPA',          #7
                                        'APO_ADDRESS_TRPA',         #8
                                        'PSTL_TOWN_TRPA',           #9
                                        'PSTL_STATE_TRPA',          #10
                                        'PSTL_ZIP5_TRPA',           #11
                                        # owner fields
                                            # no first and last fields
                                        'OWN_FULL_TRPA',            #12
                                        'MAIL_ADD1_TRPA',           #13
                                        'MAIL_CITY_TRPA',           #14
                                        'MAIL_STATE_TRPA',          #15
                                        'MAIL_ZIP5_TRPA',           #16
                                        # value fields  
                                        'AS_LANDVALUE_TRPA',        #17
                                        'AS_IMPROVALUE_TRPA',       #18
                                        'AS_SUM_TRPA',              #19
                                        'TAX_LANDVALUE_TRPA',       #20 
                                        'TAX_IMPROVALUE_TRPA',      #21
                                        'TAX_SUM_TRPA',             #22
                                        'TAX_YEAR_TRPA',            #23
                                        # land use fields 
                                        'COUNTY_LANDUSE_CODE_TRPA', #24
                                        'COUNTY_LANDUSE_TRPA',      #25
                                        # Fields for building info
                                        "YEAR_BUILT_TRPA",          #26
                                        'UNITS_TRPA',               #27
                                        'BEDROOMS_TRPA',            #28
                                        'BATHROOMS_TRPA',           #29
                                        'BUILDING_SQFT_TRPA',       #30
                                        'VHR_TRPA',                 #31
                                        'HOA_TRPA',                 #32
                                        ###-------------------------###
                                        # County Fields to get data from
                                        'PRCL_ID',                  #33
                                        'OWNER_NAME',               #34
                                        'MAIL_ADDR1',               #35
                                        'MAIL_ADDR2',               #36
                                        'MAIL_ADDR3',               #37
                                        'MAIL_ADDR4',               #38
                                        'ADDRSTNBR',                #39
                                        'ADDRSTDIR',                #40
                                        'ADDRSTNAME',               #41
                                        'ADDRSTTYPE',               #42
                                        'ADDRUNITNB',               #43
                                        'PRCL_ADDR',                #44
                                        'USECD_1',                  #45
                                        'USECDLIT_1',               #46
                                        'STRUCT_VAL',               #47
                                        'LAND_VAL',                 #48
                                        'YR_BUILT',                 #49
                                        'DWELLUNITS',               #50
                                        'BEDROOMS',                 #51
                                        'ADDRSTPRFX'                 #52
]) as cursor:
    # transform each row
    for row in cursor:   
        # Set APN
        apn = row[33]
        if not (apn is None or apn == "" or apn.isspace() == True or 'UN' in apn):
            row[0] = (apn[:3] + "-" + apn[3:6] + "-" + apn[6:9])
        else:
            row[0] = ''
            
        # Set PPNO
        ppno = row[33]
        if not (apn is None or apn == "" or apn.isspace() == True or 'UN' in apn or 'NP' in apn):
            try:
                row[1] = float(ppno)
            except ValueError:
                row[1] = 0
        else:
            row[1] = 0
        # Set County
        row[2] = 'EL'
        
        # APO Address
        full_address = row[44]
        if not (full_address is None or full_address=='' or full_address.isspace()==True):
            
            row[8] = full_address
        else:
            row[8] = ''
        
        # House Number
        house = row[39]
        if not (house is None):
            # convert house number to integer type
            row[3] = str(int(house))
        else:
            row[3] = ''
        
        # Street Direction
        street_direction = row[40]
        if not (street_direction is None or street_direction=='' or street_direction.isspace()==True
                or street_direction == 'UNASSIGNED'):
            # get the first character
            row[4] = street_direction[0]
        else:
            row[4] = ''
            
        # Street Name
        street_name = row[41]
        street_prefix = row[52]
        if not (street_name is None or street_name =='' or street_name.isspace()==True):
            if not (street_prefix is None or street_prefix =='' or street_prefix.isspace()==True
                   or street_prefix == 'UNASSIGNED'):
                row[5]= street_prefix + ' ' + street_name
            else:
                row[5] = street_name
        else:
            row[5] = ''
            
        # Street Suffix
        street_suffix = row[42]
        if not (street_suffix is None or street_suffix =='' or street_suffix.isspace()==True               
                or street_direction == 'UNASSIGNED'):
            row[6] = street_suffix
        else:
            row[6] = ''
            
        # Unit Number
        unit= row[43]
        if not (unit is None or unit=='' or unit.isspace()==True):
            row[7] = ("#" + str(unit))
        else:
            row[7] = ''
                    
        # Postal Town - see Search/Update Cursor below
        row[9] = ''
        
        # Postal State
        row[10] = 'CA'
        
        # Postal Zip - See Search/Update Cursor below
        row[11] = ''    
        
        # Set Mailing Owner, Address, City, State, Zip
        if row[38] != ' ':
#             print("Working on MAIL_ADDR4")
            if row[38] != 'UNKNOWN' and row[38].lower() not in countriesList:
                # Parse out city, state, and zip code and assign variables.
                cityStateZip = re.search(cityStateZipRegex, str(row[38]))
                if cityStateZip is not None:
                    city = cityStateZip.group(1)
                    state = cityStateZip.group(2)
                    zipCode = cityStateZip.group(3)
                    country = ''
                else:
                    continue
                # Check to see if address starts with PO Box and assign variable.
                if str(row[37]).startswith('PO') or str(row[37]).startswith('P O'):
                    address = str(row[37])
                elif "PO BOX" in str(row[37]) or "P O BOX" in str(row[37]) or "P.O. BOX" in str(row[37]):
                    address = str(row[37])

                # Parse out address that doesn't have PO Box and assign variable.
                else:
                    add = re.search(addressRegex,str(row[37]))
                    address = add.group(1)

                # Assign owner variable.
                owner = str(row[34])+' '+str(row[35])+' '+str(row[36])
            elif row[38].lower() in countriesList:
                country = str(row[38])
                state = str(row[37])
                city = str(row[36])
                address = str(row[35])
                owner = str(row[34])
                zipCode = ''
            elif row[38] == "CANADA": # temporary patch for incorrectly entered Canadian address
                canadaZip = re.search(r'[ABCEGHJKLMNPRSTVXY][0-9][ABCEGHJKLMNPRSTVWXYZ] ?[0-9][ABCEGHJKLMNPRSTVWXYZ][0-9]', str(row[3]))
                canadaProvZip = re.search(r'(.*?)\s(N[BLSTU]|[AMN]B|[BQ]C|ON|PE|SK)',str(row[36]))
                if canadaZip != None:
                    zipCode = str(canadaZip.group(0))
                else:
                    zipCode =''
                    address = str(row[35])
                    city = str(canadaProvZip.group(1))
                    state = str(canadaProvZip.group(2))
                    country = str(row[38])
            else:
                owner = str(row[34])
                address = ''
                city = ''
                state = ''
                zipCode = ''
                country = ''

        # If mail_addr4 is "empty".
        elif row[37] != ' ':
#             print("Working on MAIL_ADDR3")
            # Parse out city, state, and zip code.
            cityStateZip = re.search(cityStateZipRegex, str(row[37]))

            # Foreign addresses won't parse so assign country, owner, address, and city variables. Set state and zip to blanks.
            if cityStateZip is None:
                country = str(row[37])
                owner = str(row[34])
                address = str(row[35])
                city = str(row[36])
                state = ''
                zipCode = ''
            else:
                country = ''
                row2 = str(row[2])

                # Sanitize rows that start with a space.
                if str(row[36]).startswith(' '):
                    row2 = str(row[36])[1:]

                # Parse out city, state, and zip code and assign variables.
                city = cityStateZip.group(1)
                state = cityStateZip.group(2)
                zipCode = cityStateZip.group(3)

                # Check to see if address starts with PO Box and assign variable.
                if row2.startswith('PO') or row2.startswith('P O') or row2.startswith('P.O.'):
                    address = row2

                # Sometimes there may be a word in front of PO Box and parse that out and assign variable.
                elif "PO BOX" in row2 or "P O BOX" in row2 or row2.startswith('ONE ') or row2.startswith('TWO '):
                    address = row2
                else:
                    # Parse out address that doesn't have PO Box and assign variable, sometimes there no address so set variable to None.
                    add = re.search(addressRegex,row2)
                    if add is None:
                        address = 'None'
                    else:
                        address = add.group(1)

                # Assign owner variable.
                owner = str(row[34])+' '+str(row[35])

        # Before moving to mail_addr2 must capture "blanks" and USA owned parcels and insert blanks.
        elif row[0] == 'UNITED STATES OF AMERICA':
            cityStateZip = re.search(cityStateZipRegex, str(row[36]))
            owner = str(row[34])
            address = str(row[35])
            if cityStateZip is None:
                city = ''
                state = ''
                zipCode = ''
            else:
                city = cityStateZip.group(1)
                state = cityStateZip.group(2)
                zipCode = cityStateZip.group(3)
            country = ''
        elif row[34] == ' ':
            owner = ''
            address = ''
            city = ''
            state = ''
            zipCode = ''
            country = ''
        elif row[35] == ' ':
            owner = str(row[34])
            address = ''
            city = ''
            state = ''
            zipCode = ''
            country = ''

        # Parse the rest of the address info.
        else:
#             print("Working on MAIL_ADDR2")
            if str(row[36]) == ' ':
                owner = str(row[34])
                address = str(row[35])
                city = ''
                state = ''
                zipCode = ''
                country = ''
            else:
                row2 = str(row[36])

                # Parse out city, state, and zip code and assign variables.
                cityStateZip = re.search(cityStateZipRegex, row2)

                # if it can't parse it's a foreign address and assign country variable.
                if cityStateZip is None:
                    if "CANADA" in row2:
                        cityStateZip = re.search(canadaRegex, row2)
                        city = cityStateZip.group(1)
                        state = cityStateZip.group(2)
                        zipCode = cityStateZip.group(4)
                        country = cityStateZip.group(3)
                    if "BRAZIL" in row2:
                        cityStateZip = re.search(brazilRegex, row2)
                        city = cityStateZip.group(1)
                        state = ''
                        zipCode = cityStateZip.group(3)
                        country = cityStateZip.group(2)
                else:
                    row1 = str(row[35])
                    country = ''
                    city = cityStateZip.group(1)
                    state = cityStateZip.group(2)
                    zipCode = cityStateZip.group(3)

                    # Sanitize rows that start with a space.
                    if row1.startswith(' '):
                        row1 = row1[1:]

                    # Check to see if address starts with PO Box and assign variable.
                    if row1.startswith('PO') or row1.startswith('P.O.') or row1.startswith('P O') or row1.startswith('P  O'):
                        address = str(row[35])

                    # Sometimes there may be a word in front of PO Box and parse that out and assign variable.
                    elif "PO BOX" in row1 or "P O BOX" in row1:
                        poBox = re.search(poBoxRegex,row1)

                        # If it can't be parsed assign variable.
                        if poBox is None:
                            address = row1
                        else:
                            address = poBox.group(2)
                    else:
                        # Parse out address that doesn't have PO Box and assign variable, sometimes there no address so set variable to None.
                        add = re.search(addressRegex,row1)

                        # Have exception for addresses that spell out 'one' instead of '1'.
                        if add is None or row1.startswith('ONE'):
                            address = row1
                        else:
                            address = add.group(1)

                # Set owner variable.
                owner = str(row[34])

        # Set Owner
        row[12] = owner
        
        # Set Mailing Address
        row[13] = address
        
        # Set Mailing City
        row[14] = city
        
        # Set Mailing State
        row[15] = state
        
        # Set Mailing ZIP
        row[16] = zipCode[:5]
#         row[10] = country

        # Assessed Land Value
        land_value = row[48]
        if not(land_value is None):
            row[17] = land_value
        else:
            row[17] = ''
        
        # Assessed Improved Value
        improved_value = row[47]
        if not (improved_value is None):
            row[18] = improved_value
        else:
            row[18] = None
                
        # Assessed Sum
        if not (land_value is None or improved_value is None):
            assessed_sum = improved_value + land_value
            row[19] = assessed_sum
        else:
            row[19] = None
        
        # Tax  Land Value
        taxland_value = row[48]
        if not(taxland_value is None):
            row[20] = taxland_value
        else:
            row[20] = None
        
        # Tax Improved Value
        taximproved_value = row[47]
        if not (taximproved_value is None):
            row[21] = taximproved_value
        else:
            row[21] = None
        
        # Tax Sum
        if not (land_value is None or improved_value is None):
            tax_sum = taximproved_value + taxland_value
            row[22] = tax_sum
        else:
            row[22] = None
        
        # Tax Year
        row[23] = datetime.now().year # get current year
            
        # County Land Use Code
        county_luc = row[45]
        if not (county_luc is None):
            row[24] = str(county_luc)
        else:
            row[24] = '' 
        
        # County Land Use - See Search/Update Cursor Below
        county_landuse = row[46]
        if not (county_landuse is None or county_landuse=='' or county_landuse.isspace()==True):
            row[25] = county_landuse
        else:
            row[25] = '' 
        
        # Year Built
        year_built = row[49]
        if not (year_built is None):
            row[26] = year_built
        else:
            row[26] = None
            
        # Units
        units = row[50]
        if not (units is None):
            row[27] = units
        else:
            row[27] = None
        
        # Bedrooms
        bedrooms = row[51]
        if not (bedrooms is None):
            row[28] = bedrooms
        else:
            row[28] = None

        # Update the row.
        cursor.updateRow(row)
del cursor

out_coordinate_system = arcpy.SpatialReference('NAD 1983 UTM Zone 10N') 



CombineAPNs(eldoradoParcel, 'APN_TRPA')


arcpy.Project_management(eldoradoParcel, parcel_out, out_coordinate_system)



print('New El Dorado Parcels transformed')

Started combining APNs: 2023-05-18 20:20:15
{'', '920-000-740', '029-630-003', '029-670-002', '027-010-016', '029-630-026', '022-312-017', '029-630-020', '029-630-023', '920-000-739', '029-630-025', '029-630-014', '022-333-002', '029-630-013', '029-630-001', '029-630-002', '029-630-015', '029-630-017', '029-630-008', '029-630-010', '029-630-004', '029-670-001', '029-630-028', '029-630-009', '029-630-007', '029-630-006', '029-630-019', '029-630-027', '029-630-022', '920-000-405', '029-630-021', '029-630-018', '029-630-012', '029-630-011', '029-630-024', '029-630-005', '029-630-016'}
920-000-740
Function DissolveGeoms took 0.015628337860107422 seconds to execute.
029-630-003
Function DissolveGeoms took 0.015587806701660156 seconds to execute.
029-670-002
Function DissolveGeoms took 0.0 seconds to execute.
027-010-016
Function DissolveGeoms took 0.015615463256835938 seconds to execute.
029-630-026
Function DissolveGeoms took 0.0 seconds to execute.
022-312-017
Function DissolveGeoms took 

### Placer County

In [21]:
in_features = "Parcel_PL_Extracted"
parcel_out  = "Parcel_PL_Transformed"

# in-memory feature class
placerParcel = r"in_memory/inMemoryFeatureClass"

# copy feature class into in-memory feature class to work on
arcpy.management.CopyFeatures(in_features, placerParcel)

# Add TRPA base fields
arcpy.management.AddFields(placerParcel, baseFields)

# Transform County data to TRPA data.
with arcpy.da.UpdateCursor(placerParcel, ['APN_TRPA',               #0
                                        'PPNO_TRPA',                #1
                                        'JURISDICTION_TRPA',        #2
                                         # parcel address   
                                        'HSE_NUMBR_TRPA',           #3
                                        'STR_DIR_TRPA',             #4
                                        'STR_NAME_TRPA',            #5
                                        'STR_SUFFIX_TRPA',          #6
                                        'UNIT_NUMBR_TRPA',          #7
                                        'APO_ADDRESS_TRPA',         #8
                                        'PSTL_TOWN_TRPA',           #9
                                        'PSTL_STATE_TRPA',          #10
                                        'PSTL_ZIP5_TRPA',           #11
                                        # owner fields
                                        'OWN_FIRST_TRPA',           #12
                                        'OWN_LAST_TRPA',            #13
                                        'OWN_FULL_TRPA',            #14
                                        'MAIL_ADD1_TRPA',           #15
                                        'MAIL_CITY_TRPA',           #16
                                        'MAIL_STATE_TRPA',          #17
                                        'MAIL_ZIP5_TRPA',           #18
                                        # value fields  
                                        'AS_LANDVALUE_TRPA',        #19
                                        'AS_IMPROVALUE_TRPA',       #20
                                        'AS_SUM_TRPA',              #21
                                        'TAX_LANDVALUE_TRPA',       #22 
                                        'TAX_IMPROVALUE_TRPA',      #23
                                        'TAX_SUM_TRPA',             #24
                                        'TAX_YEAR_TRPA',            #25
                                        # land use fields 
                                        'COUNTY_LANDUSE_CODE_TRPA', #26
                                        'COUNTY_LANDUSE_TRPA',      #27
                                        # Fields for building info
                                        "YEAR_BUILT_TRPA",          #28
                                        'UNITS_TRPA',               #29
                                        'BEDROOMS_TRPA',            #30
                                        'BATHROOMS_TRPA',           #31
                                        'BUILDING_SQFT_TRPA',       #32
                                        'VHR_TRPA',                 #33
                                        'HOA_TRPA',                 #34
                                        ###-------------------------###
                                        # County Fields to get data from
                                        'APN',   # apn                #35
                                        'FEEPARCEL',   # ppno            #36
                                        'STREETNUM', # house number   #37
                                        'STREETDIR',# street dir      #38
                                        'STREETNAME',# street name    #39
                                        'STREETTYPE',# street suffix  #40
                                        'SP_APT',  # unit number      #41
                                        'OWNER1',# owner name         #42
                                        'OWNER2',# owner 2            #43
                                        'ADR1',  # mailing addr1      #44
                                        'ADR2',  # mailing addr2      #45
                                        'CITY',  # city               #46 
                                        'STATE', # state              #47
                                        'ZIP',  # zip                 #48
                                        'USE_CD', # land use code     #49
                                        'USE_CD_N', # land use desc   #50
                                        'LANDVALUE',# land value      #51
                                        'STRUCTURE',# improved value  #52
                                        'EffectiveYr',# year built     #53
                                        'StructureSF'  # build sqft      #54
                                    
]) as cursor:   
    # loop through each record to transform values to TRPA schema values
    for row in cursor:
        # set APN
        apn = row[35]
        if not (apn is None or apn == "" or apn.isspace() == True or "ROW" in apn or len(apn) < 8):
            row[0] =apn[:11]
        else:
            row[0] = ""
            
        # set PPNO
        ppno = row[36]
        if not (ppno is None or ppno == "" or "ROW" in ppno or len(ppno) < 8):
            row[1] = ppno[:8]
        else:
            row[1] = 0
            
        # Jurisdiction
        row[2] = "PL"
        
        # House Number
        house = row[37]
        if not (house is None or house=='' or house.isspace()==True):
            row[3] = house
        else:
            row[3] = ''
        
        # Street Direction
        street_direction = row[38]
        if not (street_direction is None or street_direction=='' or street_direction.isspace()==True):
            row[4] = street_direction
        else:
            row[4] = ''
            
        # Street Name
        street_name = row[39]
        if not (street_name is None or street_name =='' or street_name.isspace()==True):
            row[5] = street_name
        else:
            row[5] = ''
            
        # Street Suffix
        street_suffix = row[40]
        if not (street_suffix is None or street_suffix =='' or street_suffix.isspace()==True):
            row[6] = street_suffix
        else:
            row[6] = ''
            
        # Unit Number
        unit= row[41]
        if not (unit is None or unit=='' or unit.isspace()==True):
            row[7] = str(unit)
        else:
            row[7] = ''
        
        # APO Address
        full_address = [house, street_direction, street_name, street_suffix, unit]
        adr = str(' '.join(filter(None, full_address))).strip()
        
        if not (adr is None or adr=='' or adr.isspace()==True):
            row[8] = adr
        else:
            row[8] = ''
            
        # Postal Town - See TRPA ATTRIBUTION section
            
        # Postal State
        row[10] = 'CA'

        # Postal City - See TRPA ATTRIBUTION section
        
        # Owner Name
        owner1 = row[42]
        owner2 = row[43]
        # own first
        if not (owner1 is None or owner1 == "" or owner1.isspace() == True):
            row[12] = owner1.strip()
        else:
            row[12] = ''
        # own last
        if not (owner2 is None or owner2 == "" or owner2.isspace() == True):
            row[13] = owner2.strip()
        else:
            row[13] = ''    
        # own full
        if not (owner2 is None or owner2 == "" or owner2.isspace() == True):
            row[14] = (owner1+" " + owner2).strip()
        elif not (owner1 is None or owner1 == ""):
            row[14] = owner1.strip()
        else:
            row[14] = ''
            
        # Mailing Address
        address1 = row[44]
        address2 = row[45]
        if not (address1 is None or address1=='' or address1.isspace()==True):
            row[15] = str(address1).strip()
        else:
            row[15] = ''
                  
        # Mailing City
        mail_city = row[46]
        
        if not (mail_city is None or mail_city=='' or mail_city.isspace()==True):
            row[16] = mail_city
        else:
            row[16] = ''
            
        # Mailing State
        mail_state = row[47]
        if not (mail_state is None or mail_state=='' or mail_state.isspace()==True):
            row[17] = mail_state
        else:
            row[17] = ''
        
        # Mailing Zipcode
        mail_zip = row[48]
        if not (mail_zip is None or mail_zip=='' or mail_zip.isspace()==True):
            row[18] = mail_zip[:5]
        else:
            row[18] = ''
            
        # Assessed Land Value
        land_value = row[51]
        if not(land_value is None or land_value==''):
            row[19] = int(land_value)
        else:
            row[19] = None
       
        # Assessed Improved Value    
        improved_value = row[52]
        if not (improved_value is None or improved_value==''):
            row[20] = int(improved_value)
        else:
            row[20] = None

        # Assessed Sum
        if not (row[19] is None and row[20] is None):
            assessed_sum = improved_value + land_value
            row[21] = assessed_sum
        else:
            row[21] = None
        
        # Tax Land Value
        taxland_value = row[51]
        if not(taxland_value is None):
            row[22] = int(taxland_value)
        else:
            row[22] = None
        
        # Tax Improved Value
        taximproved_value = row[52]
        if not (taximproved_value is None):
            row[23] = int(taximproved_value)
        else:
            row[23] = None
        
        # Tax Sum
        if not (row[22] is None and row[23] is None):
            tax_sum = taximproved_value + taxland_value
            row[24] = tax_sum
        else:
            row[24] = None
        
        # Tax Year
        row[25] = datetime.now().year # get current year
            
        # County Land Use Code
        county_luc = row[49]
        if not (county_luc is None or county_luc=='' or county_luc.isspace()==True):
            row[26] = county_luc
        else:
            row[26] = '' 
        
        # County Land Use
        county_landuse = row[50]
        if not (county_landuse is None or county_landuse=='' or county_landuse.isspace()==True):
            row[27] = county_landuse
        else:
            row[27] = ''
            
        # Year Built
        year_built = row[53]
        if not (year_built is None):
            row[28] = year_built
        else:
            row[28] = None
            
        # Building SQFT
        bldsqft = row[54]
        if not (bldsqft is None):
            row[32] = bldsqft
        else:
            row[32] = None
             
        # Update the row.
        cursor.updateRow(row)
del cursor

# combine duplicate APNs 
### some shoreline parcels are split by the highway and have two features for the same APN
CombineAPNs(placerParcel, 'APN_TRPA')

# project to our projected coordinate system
out_coordinate_system = arcpy.SpatialReference('NAD 1983 UTM Zone 10N') 
arcpy.Project_management(placerParcel, parcel_out, out_coordinate_system)

# done with the transormations for Placer
print('New Placer Parcels transformed')

Started combining APNs: 2023-05-18 21:10:22
{'', '083-061-011', '117-030-008', '112-250-029', '098-101-006', '094-410-001', '098-191-001', '112-210-005', '093-411-018', '098-051-001', '096-102-036', '084-231-009', '096-440-019', '094-221-004', '096-030-056', '111-190-039', '117-200-005', '096-440-013', '096-102-006', '084-191-002', '093-416-002', '117-120-036', '097-200-023', '111-100-026', '092-110-050', '083-162-017', '092-083-009', '102-060-008', '094-350-035', '102-080-007', '096-440-009', '112-290-017', '094-350-006', '095-430-004', '096-130-012', '093-042-040', '084-073-004', '117-240-010', '083-110-022', '092-024-004', '093-411-009', '096-671-001', '095-100-018', '093-260-019', '085-216-016', '093-060-011', '112-100-016', '097-170-005', '096-440-010', '090-164-010', '096-380-021', '115-040-027', '090-271-008', '111-070-030', '096-460-015', '117-190-008', '096-121-010', '083-151-010', '111-210-005', '093-540-015', '093-360-020', '102-130-022', '090-111-039', '094-380-008', '116-0

094-350-006
Function DissolveGeoms took 0.0 seconds to execute.
095-430-004
Function DissolveGeoms took 0.0 seconds to execute.
096-130-012
Function DissolveGeoms took 0.0 seconds to execute.
093-042-040
Function DissolveGeoms took 0.0 seconds to execute.
084-073-004
Function DissolveGeoms took 0.0 seconds to execute.
117-240-010
Function DissolveGeoms took 0.0 seconds to execute.
083-110-022
Function DissolveGeoms took 0.0 seconds to execute.
092-024-004
Function DissolveGeoms took 0.015559673309326172 seconds to execute.
093-411-009
Function DissolveGeoms took 0.0 seconds to execute.
096-671-001
Function DissolveGeoms took 0.0 seconds to execute.
095-100-018
Function DissolveGeoms took 0.0 seconds to execute.
093-260-019
Function DissolveGeoms took 0.0 seconds to execute.
085-216-016
Function DissolveGeoms took 0.0 seconds to execute.
093-060-011
Function DissolveGeoms took 0.0009963512420654297 seconds to execute.
112-100-016
Function DissolveGeoms took 0.0 seconds to execute.
097-1

096-410-001
Function DissolveGeoms took 0.0 seconds to execute.
090-202-023
Function DissolveGeoms took 0.0 seconds to execute.
085-270-001
Function DissolveGeoms took 0.0 seconds to execute.
094-172-006
Function DissolveGeoms took 0.0 seconds to execute.
116-050-044
Function DissolveGeoms took 0.0 seconds to execute.
083-010-037
Function DissolveGeoms took 0.015602588653564453 seconds to execute.
102-020-001
Function DissolveGeoms took 0.0 seconds to execute.
094-253-004
Function DissolveGeoms took 0.0 seconds to execute.
117-090-049
Function DissolveGeoms took 0.0 seconds to execute.
096-440-029
Function DissolveGeoms took 0.0 seconds to execute.
096-440-016
Function DissolveGeoms took 0.0 seconds to execute.
090-340-004
Function DissolveGeoms took 0.0 seconds to execute.
096-440-018
Function DissolveGeoms took 0.0 seconds to execute.
093-230-044
Function DissolveGeoms took 0.0 seconds to execute.
083-440-015
Function DissolveGeoms took 0.0 seconds to execute.
096-440-017
Function Di

098-252-004
Function DissolveGeoms took 0.0 seconds to execute.
098-060-036
Function DissolveGeoms took 0.0 seconds to execute.
098-169-007
Function DissolveGeoms took 0.0 seconds to execute.
084-140-026
Function DissolveGeoms took 0.0 seconds to execute.
117-170-007
Function DissolveGeoms took 0.0 seconds to execute.
084-083-006
Function DissolveGeoms took 0.0 seconds to execute.
112-180-035
Function DissolveGeoms took 0.0019974708557128906 seconds to execute.
084-232-004
Function DissolveGeoms took 0.0020072460174560547 seconds to execute.
096-651-027
Function DissolveGeoms took 0.001986980438232422 seconds to execute.
090-192-060
Function DissolveGeoms took 0.0 seconds to execute.
085-330-017
Function DissolveGeoms took 0.0 seconds to execute.
084-110-016
Function DissolveGeoms took 0.0 seconds to execute.
111-180-021
Function DissolveGeoms took 0.0 seconds to execute.
091-172-007
Function DissolveGeoms took 0.0 seconds to execute.
102-010-009
Function DissolveGeoms took 0.001000881

085-260-013
Function DissolveGeoms took 0.0 seconds to execute.
098-231-007
Function DissolveGeoms took 0.0 seconds to execute.
096-450-014
Function DissolveGeoms took 0.01565408706665039 seconds to execute.
090-222-040
Function DissolveGeoms took 0.0 seconds to execute.
115-020-035
Function DissolveGeoms took 0.0 seconds to execute.
095-030-001
Function DissolveGeoms took 0.015599727630615234 seconds to execute.
090-231-026
Function DissolveGeoms took 0.0 seconds to execute.
116-130-001
Function DissolveGeoms took 0.0156252384185791 seconds to execute.
093-243-006
Function DissolveGeoms took 0.0 seconds to execute.
Finished combining APNs: 2023-05-18 21:22:36
Function CombineAPNs took 733.4439570903778 seconds to execute.
New Placer Parcels transformed


### Washoe County

In [14]:
# input/output
in_features = "Parcel_WA_Extracted"
parcel_out  = "Parcel_WA_Transformed"

# in-memory feature class
washoeParcels = r"in_memory/inMemoryFeatureClass"

# copy features to in-memory feature class
arcpy.CopyFeatures_management(in_features, washoeParcels)

# Add TRPA base fields
arcpy.management.AddFields(washoeParcels,baseFields)

# Tansform County Data to TRPA Data.
with arcpy.da.UpdateCursor(washoeParcels, ['APN_TRPA',              #row[0]
                                        'PPNO_TRPA',                #row[1]
                                        'JURISDICTION_TRPA',        #row[2]
                                         # parcel address   
                                        'HSE_NUMBR_TRPA',           #3
                                        'STR_DIR_TRPA',             #4
                                        'STR_NAME_TRPA',            #5
                                        'STR_SUFFIX_TRPA',          #6
                                        'UNIT_NUMBR_TRPA',          #7
                                        'APO_ADDRESS_TRPA',         #8
                                        'PSTL_TOWN_TRPA',           #9
                                        'PSTL_STATE_TRPA',          #10
                                        'PSTL_ZIP5_TRPA',           #11
                                        # owner fields
                                        'OWN_FIRST_TRPA',           #12
                                        'OWN_LAST_TRPA',            #13
                                        'OWN_FULL_TRPA',            #14
                                        'MAIL_ADD1_TRPA',           #15
                                        'MAIL_CITY_TRPA',           #16
                                        'MAIL_STATE_TRPA',          #17
                                        'MAIL_ZIP5_TRPA',           #18
                                        # value fields  
                                        'AS_LANDVALUE_TRPA',        #19
                                        'AS_IMPROVALUE_TRPA',       #20
                                        'AS_SUM_TRPA',              #21
                                        'TAX_LANDVALUE_TRPA',       #22 
                                        'TAX_IMPROVALUE_TRPA',      #23
                                        'TAX_SUM_TRPA',             #24
                                        'TAX_YEAR_TRPA',            #25
                                        # land use fields 
                                        'COUNTY_LANDUSE_CODE_TRPA', #26
                                        'COUNTY_LANDUSE_TRPA',      #27
                                        # Fields for building info
                                        "YEAR_BUILT_TRPA",          #28
                                        'UNITS_TRPA',               #29
                                        'BEDROOMS_TRPA',            #30
                                        'BATHROOMS_TRPA',           #31
                                        'BUILDING_SQFT_TRPA',       #32
                                        'VHR_TRPA',                 #33
                                        'HOA_TRPA',                 #34
                                        ###-------------------------###
                                        # County Fields to get data from
                                        'PIN',   # apn              #35
                                        'APN',   # ppno             #36
                                        'FullAddress',#full adrress #37
                                        'STREETNUM', # house number #38
                                        'STREETDIR',# street dir    #39
                                        'STREET',# street name      #40
                                        'CITY',    # postal town    #41
                                        'SITUSZIP', # postal zip    #42
                                        'SQFEET',# building sqft    #43
                                        'FIRSTNAME',# first name    #44
                                        'LASTNAME', # last name     #45
                                        'MAILING1',# mailing addr1  #46
                                        'MAILING2',# mailing addr2  #47
                                        'MAILCITY',# city           #48
                                        'MAILSTATE', # mailing state#49
                                        'MAILZIP',  # zip           #50
                                        'TAXYEAR', # tax year       #51
                                        'LAND_USE',# land use code  #52
                                        'LANDASS',# land value      #53
                                        'BUILDASS',# improved value #54
                                        'TOTALASS', # total assesed #55
                                        'LANDAPR',  # land apr      #56
                                        'BUILDAPR', # building apr  #57
                                        'TOTALAPR', # total apr     #58
                                        'YEARBLT',# year built      #59
                                        'STORIES',# stories         #60
                                        'BEDROOMS', # bedrooms      #61      
                                        'BATHS',# bathrooms         #62
                                        'UNITS'   # units           #63
]) as cursor:
    # loop through each record and transform the values
    for row in cursor:
        # APN field
        # Get County value
        apn  = row[35]
        if not (apn is None or apn == "" or apn.isspace() == True):
            # set TRPA value
            row[0] = apn
        else:
            row[0] = ''
            
        #PPNO
        ppno = row[36]
        if not (ppno is None or ppno == ""):
            row[1] = int(ppno)
        else:
            row[1] = None
            
        # Jurisdiction
        row[2] = "WA"
                
        # APO Address
        fulladdress = row[37]
        if not (fulladdress is None or fulladdress=='' or fulladdress.isspace()==True):
            row[8] = fulladdress
        else:
            row[8] = ''
        
        # House Number
        house = row[38]
        if not (house is None or house=='' or house.isspace()==True):
            row[3] = house
        else:
            row[3] = ''
            
           
        # Unit Number
        if not (fulladdress is None or fulladdress == ""):
            if fulladdress.strip()[-1].isdigit():
                if not ('STATE ROUTE 28' in fulladdress):
                    row[7] = (fulladdress.rsplit(' ')[-1].strip())
                else:
                    if not (fulladdress.strip().rsplit(' ')[-1] == '28'):
                        row[7] = (fulladdress.rsplit(' ')[-1].strip())
                    else:
                        if not ('STATE ROUTE 28 28' in fulladdress): 
                            row[7] = ""
                        else:
                            row[7] = (fulladdress.rsplit(' ')[-1].strip())
            else:
                if not ('US HIGHWAY 395' in fulladdress):
                    if len(fulladdress.rsplit(' ')[-1]) == 1:
                        row[7] = (fulladdress.rsplit(' ')[-1].strip())
                    elif not (len(fulladdress.rsplit(' ')[-1]) == 1):
                        if fulladdress[-2].isdigit():
                            row[7] = (fulladdress.rsplit(' ')[-1].strip())
                        else:
                            row[7] = ""
                    else:
                        row[7] = ""
                else:
                    row[7] = ""
        else:
            row[7] = ""
            
        # Street Direction
        stdir = row[39]
        if not (stdir is None):
            row[4] = (stdir.strip())
        else:
            row[4] = ""
            
        # Street Name    
        stname = row[40]
        if not stname in ('CROSS BOW', 'ENTERPRISE', 'STATE ROUTE 28', 'UNSPECIFIED', 'US HIGHWAY 395', ''):
            if stname[:2] in ('N ', 'S ', 'E ', 'W '):
                row[5] = stname.rsplit(' ',1)[0].strip().split(' ',1)[1].strip()
            elif not (stname is None or stname == "" or stname.isspace() == True):
                row[5] = (stname.rsplit(' ',1)[0].strip())
            #Currently the only example of this is two blanks in Incline Village with no info
            elif stname is None or stname == "" or stname.isspace() == True:
                if fulladdress[0].isdigit():
                    row[5] = (fulladdress.rsplit(' ')[-1].strip())
                else:
                    row[5] = ""
            else:
                logging.info("Error parsing washoe street name")
        elif stname in ('CROSS BOW', 'ENTERPRISE', 'STATE ROUTE 28', 'UNSPECIFIED', ''):
                row[5] = (stname.strip())
        else:
            row[5] = ""
            
        # Street Suffix
        if not stname in ('CROSS BOW', 'ENTERPRISE', 'STATE ROUTE 28', 'UNSPECIFIED', 'US HIGHWAY 395', ''):
            if not (stname is None or stname == "" or stname.isspace() == True):
                row[6] = (stname.rsplit(' ')[-1].strip())
            elif stname is None or stname == "" or stname.isspace() == True:
                if fulladdress[0].isdigit():
                    row[6] = (fulladdress.rsplit(' ')[-1].strip())
                else:
                    row[6] = ""
            else:
                logging.info("Error parsing washoe street suffix")
        else:
            row[11] = ""

        # Postal Town
        postal_town = row[41]
        if not (postal_town is None or postal_town == '' or postal_town.isspace()==True):
            row[9] = postal_town
        else:
            row[9] = ''
            
        # Postal State
        row[10] = 'NV'
        
        # Postal Zip
        postal_zip = row[42]
        if not (postal_zip is None or postal_zip == '' or postal_zip.isspace()==True):
            row[11] = postal_zip
        else:
            row[11] = ''
            
        # Owner Name
        # set owner first name
        ownfirst = row[44]
        if not (ownfirst is None or ownfirst.isspace() == True):
            row[12] = ownfirst
        else:
            row[12] = ""
        
        # own last
        ownlast = row[45]
        if not (ownlast is None or ownlast.isspace() == True):
            row[13] = ownlast
        else:
            row[13] = ""
        
        # own full
        if not (ownfirst is None and ownlast is None):
            row[14] = (ownfirst + " " + ownlast).strip()
        else:
            row[14] = ""
            
        # Mailing Address  
        address1 = row[46].strip()
        address2 = row[47].strip()
        if not (address1 is None or address1=='' or address1.isspace()==True):
            row[15] = str((address1 + " " + address2).strip())
        elif (address2 is None):
            row[15] = address1
        else:
            row[15] = ''
           
        # Mailing City
        mail_city = row[48]
        if not (mail_city is None or mail_city=='' or mail_city.isspace()==True):
            row[16] = mail_city
        else:
            row[16] = ''
            
        # Mailing State
        mail_state = row[49]
        if not (mail_state is None or mail_state=='' or mail_state.isspace()==True):
            row[17] = mail_state
        else:
            row[17] = ''
        
        # Mailing Zipcode
        mail_zip = row[50].strip()
        if not (mail_zip is None or mail_zip=='' or mail_zip.isspace()==True):
            row[18] = mail_zip[:5]
        else:
            row[18] = ''
            
        # Assessement Value
        land_value = row[53]
        if not(land_value is None or land_value==''):
            row[19] = land_value
        else:
            row[19] = None
        
        improved_value = row[54]
        if not (improved_value is None or improved_value==''):
            row[20] = improved_value
        else:
            row[20] = None
                
        assessed_sum = row[55]
        if not (assessed_sum is None or assessed_sum==''):
            row[21] = assessed_sum
        else:
            row[21] = None
        
        # Tax Value
        taxland_value = row[56]
        if not(taxland_value is None or taxland_value==''):
            row[22] = taxland_value
        else:
            row[22] = None
        
        taximproved_value = row[57]
        if not (taximproved_value is None or taximproved_value==''):
            row[23] = taximproved_value
        else:
            row[23] = None
        
        tax_sum = row[58]
        if not (tax_sum is None or tax_sum==''):
            row[24] = tax_sum
        else:
            row[24] = None
        
        # Tax Year
        tax_year = row[51]
        if not (tax_year is None or tax_year=='' or tax_year.isspace()==True):
            row[25] = tax_year
        else:
            row[25] = None
            
        # County Land Use Code
        county_luc = row[52]
        if not (county_luc is None or county_luc=='' or county_luc.isspace()==True):
            row[26] = int(county_luc.split(",",1)[0].strip())
        else:
            row[26] = None 
        
        # Year Built
        year_built = row[59]
        if not (year_built is None or year_built==''):
            row[28] = year_built
        else:
            row[28] = None
            
        # Units
        units = row[63]
        if not (units is None or units==''):
            row[29] = int(units)
        else:
            row[29] = None
        
        # Bedrooms
        bedrooms = row[61]
        if not (bedrooms is None or bedrooms==''):
            row[30] = bedrooms
        else:
            row[30] = None
        
        # Bathrooms
        bathrooms = row[62]
        if not (bathrooms is None or bathrooms==''):
            row[31] = bathrooms
        else:
            row[31] = None
            
        # Building Square Feet
        building_sqft = row[43]
        if not (building_sqft is None or building_sqft==''):
            row[32] = building_sqft
        else:
            row[32] = None

        # Update the row.
        cursor.updateRow(row)
del cursor

# create a spatial reference object for the output coordinate system 
out_coordinate_system = arcpy.SpatialReference('NAD 1983 UTM Zone 10N') 
arcpy.Project_management(washoeParcels, parcel_out, out_coordinate_system)

print('New Washoe Parcels transformed')

New Washoe Parcels transformed


### Merge

In [22]:
# delete in-memory
arcpy.Delete_management("memory")
print("Deleted Memory Workspace: " + strftime("%Y-%m-%d %H:%M:%S"))

# out merge fc
parcel_out = "Parcel_Staging"

# input feature classes
ccParcel = "Parcel_CC_Transformed"
dgParcel = "Parcel_DG_Transformed"
elParcel = "Parcel_EL_Transformed"
plParcel = "Parcel_PL_Transformed"
waParcel = "Parcel_WA_Transformed"

# Create FieldMappings object to manage merge output fields
fieldMappings = arcpy.FieldMappings()
# Add all fields from all parcel staging layers
fieldMappings.addTable(ccParcel)
fieldMappings.addTable(dgParcel)
fieldMappings.addTable(elParcel)
fieldMappings.addTable(plParcel)
fieldMappings.addTable(waParcel)

# Remove all output fields from the field mappings, except fields in field_master list
for field in fieldMappings.fields:
    if field.name not in [  'OBJECTID',
                            'APN_TRPA',                 #0
                            'PPNO_TRPA',                #1
                            'JURISDICTION_TRPA',        #2
                            'COUNTY_TRPA',
                             # parcel address   
                            'HSE_NUMBR_TRPA',           #3
                            'STR_DIR_TRPA',             #4
                            'STR_NAME_TRPA',            #5
                            'STR_SUFFIX_TRPA',          #6
                            'UNIT_NUMBR_TRPA',          #7
                            'APO_ADDRESS_TRPA',         #8
                            'PSTL_TOWN_TRPA',           #9
                            'PSTL_STATE_TRPA',          #10
                            'PSTL_ZIP5_TRPA',           #11
                            # owner fields
                            'OWN_FIRST_TRPA',           #12
                            'OWN_LAST_TRPA',            #13
                            'OWN_FULL_TRPA',            #14
                            'MAIL_ADD1_TRPA',           #15
                            'MAIL_CITY_TRPA',           #16
                            'MAIL_STATE_TRPA',          #17
                            'MAIL_ZIP5_TRPA',           #18
                            # value fields  
                            'AS_LANDVALUE_TRPA',        #19
                            'AS_IMPROVALUE_TRPA',       #20
                            'AS_SUM_TRPA',              #21
                            'TAX_LANDVALUE_TRPA',       #22 
                            'TAX_IMPROVALUE_TRPA',      #23
                            'TAX_SUM_TRPA',             #24
                            'TAX_YEAR_TRPA',            #25
                            # land use fields 
                            'COUNTY_LANDUSE_CODE_TRPA', #26
                            'COUNTY_LANDUSE_TRPA',      #27
                            # Fields for building info
                            "YEAR_BUILT_TRPA",          #28
                            'UNITS_TRPA',               #29
                            'BEDROOMS_TRPA',            #30
                            'BATHROOMS_TRPA',           #31
                            'BUILDING_SQFT_TRPA',       #32
                            'VHR_TRPA',                 #33
                            'HOA_TRPA',                 #34
                            'SHAPE@']:
        # remove everything else
        fieldMappings.removeFieldMap(fieldMappings.findFieldMapIndex(field.name)) 
    
# Use Merge tool to move features into single dataset
arcpy.management.Merge([ccParcel, dgParcel, elParcel, plParcel, waParcel ], parcel_out, fieldMappings)
print("Transformed Parcel Datasets Merged")

# out merge fc
parcel_out = "Parcel_Staging"
result = arcpy.GetCount_management(parcel_out)
print('{} has {} records'.format(parcel_out, result[0]))

# out merge fc
parcel_out = "Parcel_Staging"

# delete unneccesary parcels
parcelDelete = "ParcelDelete"

# Run MakeFeatureLayer
arcpy.management.MakeFeatureLayer(parcel_out, parcelDelete)
 
arcpy.management.SelectLayerByAttribute(parcelDelete, 'NEW_SELECTION', 
                                        "APN_TRPA = '' Or APN_TRPA LIKE '920%' Or APN_TRPA LIKE '910%' OR APN_TRPA LIKE '%NP%' OR APN_TRPA LIKE '%ROW%' OR APN_TRPA LIKE '%UN%'")

# Run GetCount and if some features have been selected, then 
#  run DeleteFeatures to remove the selected features.
deleteCount = arcpy.management.GetCount(parcelDelete)[0]
if int(deleteCount) > 0:
    arcpy.management.DeleteFeatures(parcelDelete)
    print('{} records deleted'.format(deleteCount))
    
result = arcpy.GetCount_management(parcel_out)
print('{} has {} records now.'.format(parcel_out, result[0]))

Deleted Memory Workspace: 2023-05-18 21:36:04
Transformed Parcel Datasets Merged
Parcel_Staging has 67118 records
2451 records deleted
Parcel_Staging has 64667 records now.


### Additional Transformation of County Data

In [23]:


suffix_dict = {
    'CI':'CIR',
    'BL':'BLVD',
    'TR':'TRL',
    'WY':'WAY',
    'E': '',
    'L':'',
    'AV':'AVE',
    'LP':'LOOP',
    'HY':'HWY',
    'PY':'PKWY',
    'PKY':'PKWY',
    'DRIVE': 'DR'
}

suffix_field = ['STR_SUFFIX_TRPA']

UpdateFieldFromDictionary('Parcel_Staging', suffix_field, suffix_dict)

replacement_values = ['UNIT','SUITE','SPACE','NULL']
set_to_blank_values = ['0', '0 NULL']

with arcpy.da.UpdateCursor('Parcel_Staging', ["STR_NAME_TRPA"]) as cursor:
    for row in cursor:
        row[0]=row[0].upper()        
        for replacement_value in replacement_values:
            row[0] = row[0].replace(replacement_value, '')
        row[0] = row[0].replace('  ', ' ')
        if row[0] in set_to_blank_values:
            row[0]=''
        if row[0].startswith('0 '):
            row[0]=row[0][2:]
        if (row[0] == '0 NO ADDRESS ON FILE')| (row[0] == 'NO ADDRESS ON FILE'):
            NewStreet='NO ADDRESS ON FILE'
            
        cursor.updateRow(row)



0 rows were updated
Function UpdateFieldFromDictionary took 2.059204578399658 seconds to execute.


### TRPA Attribution

In [24]:
print("Starting TRPA Attribution: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Starting TRPA Attribution: " + strftime("%Y-%m-%d %H:%M:%S"))

# in and out with the same name overwrite == True
ParcelStaging = "Parcel_Staging"
ParcelPoint   = "Parcel_Point"
ParcelNew     = 'Parcel_Staging_Attributed'

# copy data into an in_memory feature class for warp speed.
ParcelLayer = r"memory/ParcelLayer"
arcpy.CopyFeatures_management(ParcelStaging, ParcelLayer)

# Add TRPA fields.
arcpy.management.AddFields(ParcelLayer,trpaFields)

# ### County Atribute Update -------------------------------------------------------------------------------------###

print("Starting the County attribute update: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Starting the County attribute update: " + strftime("%Y-%m-%d %H:%M:%S"))

with arcpy.da.UpdateCursor(ParcelLayer, ["JURISDICTION_TRPA", "COUNTY_TRPA"]) as cursor:
    for row in cursor:
        # set county field before changing EL to CSLT in Jurisdiction field
        row[1] = row[0] 
        cursor.updateRow(row)
del cursor
print("County Attribute Updated")

#### Featurs to Points to use in speedy spatial joins
# copy shapes to points in new parcel point layer
arcpy.FeatureToPoint_management(ParcelLayer, ParcelPoint, "INSIDE")
print("Copied features to points: "+ strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Copied features to points: "+ strftime("%Y-%m-%d %H:%M:%S"))

### Ownership Type Attribute Update ------------------------------------------------------------------------------###
print("Starting the Ownership Type attribute update: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Starting the Ownership Type attribute update: " + strftime("%Y-%m-%d %H:%M:%S"))

# owner name lists - this is a stupid way to figure this out
fedOwnList = ("UNITED STATES OF AMERICA C/O USDA FOREST SERVICE","USA FOREST SERVICE", 
              "USDA FOREST SERVICE", "USDA - FOREST SERVICE", "UNITED STATES POSTAL", 
              "UNITED STATES OF AMERICA", "UNITED STATES FOREST SERVICE", "U S POSTAL SERVICE", "U S COAST GUARD",
              "U S A FOREST SERVICE``", "U S A FOREST SERVICE", "LAKE VALLEY RANGER STA", "DEPT OF VETRANS AFFAIRS%", 
              "DEPT OF VETERANS AFFAIRS%", "DEPT OF VETERANS AFFAIRS %", "BUREAU OF LAND MANAGEMENT", "U S FOREST SERVICE",
              "DEPT OF VETERANS AFFAIRS  & ERSKINE NEIL H TR", "DEPT OF VETERANS AFFAIRS  & MASTERS DANE C", 
              "DEPT OF VETERANS AFFAIRS  & SLEZAK FRANK J CO TR", "DEPT OF VETERANS AFFAIRS & RIVES DONALD E JR"
              "DEPT OF VETERANS AFFAIRS & WILLIAMS MATTHEW G DBA WILLIAMS VACATION HOME", "DEPT OF VETRANS AFFAIRS & WILSON VIVIAN M",
              "DEPARTMENT OF TRANSPORTATION", "USA FOREST SERVICE & OWNERSHIP UNVERIFIED", "U S A FOREST SERVICE & LAKE TAHOE BASIN MNGMT UNIT",
              "U S A FOREST SERVICE & OWNERSHIP UNVERIFIED", "U S D A FOREST SERVICE", "UNITED STATES & DEPT OF AGRICULTURE",
              "UNITED STATES OF AMERICA & ATTN RICHARD T FLYNN", "UNITED STATES OF AMERICA & DEPARTMENT OF AGRICULTU", "UNITED STATES OF AMERICA & F/S DEPT OF AGRICULTURE",
              "UNITED STATES OF AMERICA & FOREST SER. DEPT OF AG.", "UNITED STATES OF AMERICA & FOREST SERVICE", "UNITED STATES OF AMERICA & FOREST SERVICE (USDA)",
              "UNITED STATES OF AMERICA & FOREST SERVICE DEPT OF", "UNITED STATES OF AMERICA & FOREST SERVICE TAHOE BA", "UNITED STATES OF AMERICA & FOREST SERVICE USDA",
              "UNITED STATES OF AMERICA & FOREST SVC/DEPT OF AGRI", "UNITED STATES OF AMERICA & LAKE TAHOE BASIN MANAGM",
              "UNITED STATES OF AMERICA & LAKE TAHOE BASIN MGT UN", "UNITED STATES OF AMERICA & REGIONAL LAND ADJUSTMEN",
              "UNITED STATES OF AMERICA & U S FOREST SERVICE", "UNITED STATES OF AMERICA & U S FOREST SERVIE",
              "UNITED STATES OF AMERICA & USDA FOREST SER LAKE TA", "UNITED STATES OF AMERICA & USDA FOREST SERVICE",
              "USDA - FOREST SERVICE & LAKE TAHOE BASIN MGMT UNIT")

stateOwnList = ("TAHOE CONSERVANCY", "STATE OF NEVADA FOREST SERVICE", "STATE OF NEVADA", "STATE OF CALIFORNIA THE", 
                "STATE OF CALIFORNIA (EASEMENT)", "STATE OF CALIFORNIA", "STATE OF CA", "REGENTS OF UNIV OF CALIF",
                "UNIVERSITY CALIFORNIA REGENTS", "UNIVERSITY OF NEVADA RENO", "NEVADA, STATE OF", "NEVADA STATE OF", 
                "CALIFORNIA TAHOE CONSERVANCY ET AL", "CALIFORNIA TAHOE CONSERVANCY", "CALIFORNIA STATE OF THE", 
                "CALIFORNIA STATE OF ET AL", "CALIFORNIA STATE OF", "CA STATE DEPT TRANSPORTATION", 
                "CA TAHOE CONSERVANCY", "CALIFORNIA STATE OF TAHOE CONSERVANCY", "CALIFORNIA STATE OF THE", 
                "NEVADA DEPT OF TRANSPORTATION", "STATE OF CALIFORNIA & CALIFORNIA TAHOE CONSERVANCY", "STATE OF CALIFORIA & CALIFORNIA TAHOE CONSERVANCY",
                "STATE OF CALIFORNIA & CA TAHOE CONSERVANCY", "STATE OF CALIFORNIA & CALIFORNIA TAHOE CONSERVANCY CALIFORNIA TAHOE CONSERVANCY",
                "STATE OF CALIFORNIA & CALIFORNIA TAHOE CONSEVANCY", "STATE OF CALIFORNIA & DEPART OF TRANSPORTATION", "STATE OF CALIFORNIA & DEPARTMENT OF GENERAL SERVIC", 
                "STATE OF CALIFORNIA & DEPARTMENT OF TRANSPORTATION", "STATE OF CALIFORNIA & DEPT OF GEN SRVS R E DIV", "STATE OF CALIFORNIA & DEPT OF GENERAL SERVICES",
                "STATE OF CALIFORNIA & DEPT OF PARKS & RECREATION", "STATE OF CALIFORNIA & DEPT OF TRANSPORTATION", "STATE OF CALIFORNIA & PARKS & RECREATION",
                "STATE OF CALIFORNIA (EASEMENT) & CALIFORNIA TAHOE", "CALIFORNIA STATE PARKS AND RECREATION",
                "CALIFORNIA STATE OF & DEPT GEN SERVICES REAL ESTAT")

localOwnList = ("ZEPHYR COVE GENERAL IMP DIST", "WASHOE COUNTY SCHOOL DISTRICT BOARD", "WASHOE COUNTY", "WASHOE TRIBE OF NV & CA", 
                "TALMONT RESORT IMPROVEMENT DISTRICT", "TALMONT RESORT IMPR DIST", "TALMONT RESORT IMP DISTRICT",
                "TALMONT RESORT IMP DIST", "TAHOE PARADISE RESORT IMP DIST", "TAHOE PARADISE RES IMP DST",
                "TAHOE FOREST HOSPITAL DISTRICT", "TAHOE TRUCKEE UNIFIED SCHOOL DISTRICT", "TAHOE TRUCKEE UNIFIED SCH DIST", 
                "TAHOE DOUGLAS FIRE PROTECT DIST", "TAHOE DOUGLAS SEWER DIST", "TAHOE DOUGLAS DISTRICT", 
                "TAHOE CITY PUBLIC UTILITY DISTRICT", "TAHOE CITY PUBLIC UTILITY DIST", "TAHOE CITY PUBLIC UTILDIST", 
                "TAHOE CITY PUB UTILITY DST", "TAHOE CITY PUB UTILITY DIS", "TAHOE CITY P U D", "TAHOE CITY CEMETERY DIST", 
                "SOUTH TAHOE REDEVELP AGENCY", "SOUTH TAHOE REFUSE CO", "SOUTH TAHOE PUD", "SOUTH TAHOE PUBLIC UTL DST",
                "SOUTH TAHOE PUBLIC UTILITYDIST", "SOUTH TAHOE PUBLIC UTILITY DST", "SOUTH TAHOE PUBLIC UTILITY DIS", 
                "SOUTH TAHOE PUBLIC UTILITY", "SOUTH TAHOE PUBLIC UTIL DT", "SOUTH TAHOE PUBLIC UTIL DIST", 
                "SOUTH TAHOE PUBLIC", "SOUTH TAHOE PUB UTIL DIST", "SOUTH LAKE TAHOE CTYOF 1/3", "SOUTH LAKE TAHOE CITY OF", 
                "SO TAHOE PUBLIC UTILITY DIST", "SO TAHOE PUB UTIL DIST", "SIERRA NEVADA COLLEGE", "ROUND HILL GEN IMP DIST",
                "PLACER COUNTY REDEVELOPMENT AGENCY", "PLACER COUNTY OF", "PLACER COUNTY", "NORTH TAHOE PUBLIC UTL DIST",
                "NORTH TAHOE PUBLIC UTILITY DISTRICT", "NORTH TAHOE PUBLIC UTILITY DIST", "NORTH TAHOE PUBLIC UTILITY DIS", 
                "NORTH TAHOE PUBLIC UTILITIES DIST", "NORTH TAHOE PUBLIC UTILIITY DISTRICT", "NORTH TAHOE P U D",
                "NORTH TAHOE FIRE PROTECTION DISTRICT", "NORTH TAHOE FIRE PROTECTION", "NORTH TAHOE FIRE DIST",
                "NORTH LAKE TAHOE FIRE PROTECTION DIST", "N TAHOE FIRE PROTECTION DIST", "MEEKS BAY FIRE PROT DIST", 
                "LAKERIDGE GENERAL IMP DIST", "LAKE VALLEY FIRE PROTECTION", "LAKE VALLEY FIRE PROT DST", "LAKE VALLEY FIRE PROT DIST", 
                "LAKE VALLEY FIRE DISTRICT", "LAKE TAHOE UNIFIED SCHOOL DIST", "LAKE TAHOE SCHOOL", "LAKERIDGE GENERAL IMP DIST", 
                "LAKE TAHOE FIRE PROTECTION DIST", "LAKE TAHOE FIRE PROTECT DIST", "LAKE TAHOE COMM COLLEGE DIST",
                "LAKE TAHOE COMM COL DIST", "KINGSBURY GENERAL IMP DISTRICT", "KINGSBURY GENERAL IMP DIST",
                "INCLINE VILLAGE GENERAL IMPROVEMENT DISTRICT", "INCLINE VILLAGE GENERAL IMPROVEMENT DIST", 
                "DOUGLAS COUNTY SEWER DIST", "DOUGLAS COUNTY SCHOOL DIST", "DOUGLAS COUNTY", "DOUGLAS CO SEWER IMP DIST #1", 
                "COUNTY OF EL DORADO", "CITY OF SOUTH LAKE TAHOE", "EL DORADO IRRIGATION DISTRICT", 
                "HAPPY HOMESTEAD CEMETERY DIST", "WASHOE TRIBE", "SOUTHTAHOE PUBLIC UTILITY DIST", "DOUGLAS COUNTY TRUSTEE", 
                "DOUGLAS COUNTY TRUSTEE (HOLD)", "WASHOE TRIBE OF NEVADA AND CALIFORNIA", "ALPINE SPRINGS CO WATER DIST", 
                "ALPINE SPRINGS COUNTY WATER DISTRICT", "ALPINE SPRINGS WATER DISTRICT", "NORTHSTAR COMMUNITY SERVICE DISTRICT",
                "SQUAW VALLEY CO WATER DIST", "SQUAW VALLEY PUBLIC SERVICE DISTRICT", "TRUCKEE TAHOE AIRPORT DISTRICT", 
                "COUNTY OF EL DORADO & ATTEN: PAUL MCINTOSH", "COUNTY OF EL DORADO & BOARD OF SUPERVISORS", 
                "COUNTY OF EL DORADO & BOARD OF SUPERVISORS", "COUNTY OF EL DORADO & C/O BOARD OF SUPERVISORS", "COUNTY OF EL DORADO & COUNSEL",
                "COUNTY OF EL DORADO & COUNSEL'S OFFICE", "COUNTY OF EL DORADO & DEPARTMENT OF PUBLIC WORKS", "COUNTY OF EL DORADO & DEPARTMENT OF TRANSPORTATION",
                "COUNTY OF EL DORADO & DEPT OF PUBLIC WORKS", "COUNTY OF EL DORADO & DEPT OF TRANSPORTATION", "COUNTY OF EL DORADO & GENERAL SERVICES DEPARTMENT",
                "COUNTY OF EL DORADO & OF EL DORADO", "COUNTY OF EL DORADO & PUBLIC WORKS DEPARTMENT", "EL DORADO CO OFFICE EDUCATION", "EL DORADO COUNTY & SUPERINTENDENT OF SCHOOLS",
                "EL DORADO COUNTY & BOARD OF SUPERVISORS", "LAKE TAHOE COMMUNITY COLLEGE DIST", "LAKE VALLEY RANGER STA & U S FOREST SERVICE",
                "LAKE VALLEY FIRE PROTECTION & DISTRICT POLITICAL S", "SOUTH TAHOE PUBLIC & UTILITY DISTRICT",
                "SOUTH TAHOE PUBLIC UTILITY &  DISTRIC", "SOUTH TAHOE PUBLIC UTIL DIST & CA MUNICIPAL CORP", "TAHOE CITY PUBLIC UTIL DST",
                "TAHOE RESOURCE CONSERVATION &  DISTRIC", "TAHOE RESOURCE CONSERVATION DIST  C/O DISTRICT MANAGER", "FALLEN LEAF COMM SERVICES DIST",
                "FALLEN LEAF LAKE COMM SERVDIST", "TAHOE DOUGLAS VISITORS AUTH", "KINGSBURY GENARAL IMP DIST", "ALPINE SPRINGS CO WTR DIST FIN CORP",
                "COUNTY OF PLACER", "MCKINNEY WATER DISTRICT", "NORTHSTAR COMMUNITY SERVICES DISTRICT", "PLACER COUNTY PUBLIC WORKS",
                "REDEVELOPMENT AGENCY OF THE COUNTY OF PL", "TRUCKEE DONNER PUBLIC UTILITY DISTRICT", "TRUCKEE SANITARY DISTRICT",
                "TAHOE TRANSPORTATION DISTRICT") 

with arcpy.da.UpdateCursor(ParcelLayer, ["OWN_FULL_TRPA", "OWNERSHIP_TYPE_TRPA"]) as cursor:
    for row in cursor:
        # set ownership type
        own = row[0]
        if not (own is None or own == "" or own.isspace() == True):
            if own in fedOwnList:
                row[1] = "Federal"
            elif own in localOwnList:
                row[1] = "Local"
            elif own in stateOwnList:
                row[1] = "State"
            elif not own in (fedOwnList, localOwnList, stateOwnList):
                row[1] = "Private" 
            cursor.updateRow(row)
del cursor
print ("The 'OWNERSHIP_TYPE' field in the parcel data has been updated")
# log.info("The 'Owernshipe Type' field in the parcel data has been updated")

### Existing Landuse Attribute Update ----------------------------------------------------------------------------###
fields = ("COUNTY_LANDUSE_CODE_TRPA",
          "COUNTY_LANDUSE_TRPA",  
          "EXISTING_LANDUSE_TRPA", 
          'JURISDICTION_TRPA')

with arcpy.da.UpdateCursor(ParcelLayer, fields) as cursor:
    for row in cursor:
        ctyluc = row[0]
        cty = row[3]
        # set Washoe county land use
        # set TRPA Land Use Description
        if (row[0] != None or row[0] != "") and (row[3] == 'WA'):
            if ctyluc in ('400', '410', '440', '500', '510', '520', '630', '640', '670', '720'):
                row[2] = "Commercial"
            elif ctyluc in ('210', '250'):
                row[2] = "Condominium"
            elif ctyluc in ('240'):
                row[2] = "Condominium Common Area"
            elif ctyluc in ('220', '230', '300', '310', '320', '330', '340', '350', '360'):
                row[2] = "Multi-Family Residential"
            elif ctyluc in ('600', '620'):
                row[2] = "Open Space"
            elif ctyluc in ('700', '710', 'PBRD'):
                row[2] = "Public Service"
            elif ctyluc in ('190'):
                row[2] = "Recreation"
            elif ctyluc in ('200'):
                row[2] = "Single Family Residential"     
            elif ctyluc in ('420', '430'):
                row[2] = "Tourist Accommodation"
            elif ctyluc in ('100', '110', '120', '130', '140', '150', '160', '170', '180'):
                row[2] = "Vacant"            
            elif ctyluc is None:
                row[2] == ''
        if (row[0] != None or row[0] != "") and (row[3] == 'WA'):
            if ctyluc in ('710'):
                row[1] = "Intracounty public utility"
            elif ctyluc == '700':
                row[1] = 'Centrally assessed public utility'
            elif ctyluc == '510':
                row[1] = 'Commercial Industrial: retail or office with Indus'
            elif ctyluc == '500':
                row[1] = 'General industrial: light indust, trucking, warehs'
            elif ctyluc == '440':
                row[1] = 'Resort commercial: ski, golf, sports, etc.'
            elif ctyluc == '430':
                row[1] = 'Commercial hotel or motel'
            elif ctyluc == '420':
                row[1] = 'Casino or hotel casino'
            elif ctyluc == '410':
                row[1] = 'Offices, professional and business, banks, etc.'
            elif ctyluc == '400':
                row[1] = 'General Commercial: retail, mixed, parking, school'
            elif ctyluc == '340':
                row[1] = 'Ten or more units'
            elif ctyluc == '330':
                row[1] = 'Five to Nine Units'
            elif ctyluc == '320':
                row[1] = 'Three or four Units'
            elif ctyluc == '310':
                row[1] = 'Two Single Family Units'
            elif ctyluc == '300':
                row[1] = 'Duplex'
            elif ctyluc == '250':
                row[1] = 'Condo or Townhouse valued as apartment use'
            elif ctyluc == '240':
                row[1] = 'Common Area'
            elif ctyluc == '210':
                row[1] = 'Condominium or Townhouse'
            elif ctyluc == '200':
                row[1] = 'Single Family Residence'
            elif ctyluc == '190':
                row[1] = 'Public Parks: vacant or improved'
            elif ctyluc == '170':
                row[1] = 'Other, unbuildable: roads, restrictions, terrain'
            elif ctyluc == '160':
                row[1] = 'Splinter, unbuildable: small size or shape'
            elif ctyluc == '140':
                row[1] = 'Vacant, commercial'
            elif ctyluc == '130':
                row[1] = 'Vacant, multi-residential'
            elif ctyluc == '120':
                row[1] = 'Vacant, single family'
            elif ctyluc == '110':
                row[1] = 'Vacant, under development'
            elif ctyluc == '100':
                row[1] = 'Vacant, other or unknown'            
            elif ctyluc is None:
                row[1] == ''
            cursor.updateRow(row)
        # Set Carson City County Land Use Descriptions
        if (row[0] != None or row[0] != "") and (row[3] == 'CC'):
            if ctyluc in ('400', '401', '402', '403', '404', '408', '410', '411', 
                          '412', '440', '441', '460', '470', '480', '482', '490', 
                          '500', '501', '510', '511', '512', '513', '520', '521', 
                          '560', '570', '580', '582', '590', '624', '625', '694', 
                          '800', '820', '830', '840', '880', '882', '890', '920', 
                          '921', '930', '960', '980', '990'):
                row[2] = "Commercial"
            elif ctyluc in ('210', '211'):
                row[2] = "Condominium"
            elif ctyluc == '970':
                row[2] = "Condominium Common Area"
            elif ctyluc in ('240', '241', '300', '301', '310', '311', '313', '320', 
                            '321', '330', '331', '333', '340', '341', '350', '360', 
                            '370', '380', '382', '390', '698'):
                row[2] = "Multi-Family Residential"
            elif ctyluc in ('190', '600', '610', '612', '613', '614', '615', '616', 
                            '618', '620', '695', '696', '697', '810'):
                row[2] = "Open Space"
            elif ctyluc in ('190', '700', '710', '711', '720', '731', '732', '733', '780', 
                            '790', '910', '922'):
                row[2] = "Public Service"
            elif ctyluc in ('450', '900'):
                row[2] = "Recreation"
            elif ctyluc in ('200', '201', '220', '222', '230', '231', '232', '260', 
                            '270', '280', '282', '290', '622', '692', '693'):
                row[2] = "Single Family Residential"     
            elif ctyluc in ('420', '421', '430', '431', '432', '514'):
                row[2] = "Tourist Accommodation"
            elif ctyluc in ('100', '108', '110', '117', '120', '130', '140', '150', '160'):
                row[2] = "Vacant"
            elif ctyluc is None:
                row[2] == ''
        if (row[0] != None or row[0] != "") and (row[3] == 'CC'):
            if ctyluc == '980':
                row[1] = 'Special Purpose with Minor Improvements'
            elif ctyluc == '320':
                row[1] = 'Three to Four Units'
            elif ctyluc == '280':
                row[1] = 'Single Family Residential with Minor Improvements'
            elif ctyluc == '190':
                row[1] = 'Vacant - Public Use Lands'
            elif ctyluc == '120':
                row[1] = 'Vacant - Single Family Residential' 
            elif ctyluc is None:
                row[1] == ''
            cursor.updateRow(row)           
        # update Douglas Land Use descriptions        
        if (row[0] != None or row[0] != "") and (row[3] == 'DG'):
            if ctyluc in ('400', '402', '410', '411', '412', 
                          '440', '460', '470', '480', '500', 
                          '510', '560', '580', '582'):
                row[2] = "Commercial"
            elif ctyluc in ('210', '211'):
                row[2] = "Condominium"
            elif ctyluc == '270':
                row[2] = "Condominium Common Area"
            elif ctyluc in ('300', '310', '320', '330', '350', '390'):
                row[2] = "Multi-Family Residential"
            elif ctyluc == '190':
                row[2] = "Open Space"
            elif ctyluc in ('700', '710', '711', '910', '980', '970'):
                row[2] = "Public Service"
            elif ctyluc in ('450', '900', '970'):
                row[2] = "Recreation"
            elif ctyluc in ('200', '220', '230', '236', '240', '280', '282'):
                row[2] = "Single Family Residential"     
            elif ctyluc in ('420', '430'):
                row[2] = "Tourist Accommodation"
            elif ctyluc in ('100', '110', '117', '120', '130', '140'):
                row[2] = "Vacant"
            elif ctyluc is None:
                row[2] == ''
        if (row[0] != None or row[0] != "" or ctyluc.isspace() != True) and (row[3] == 'DG'):
            if ctyluc == '980':
                row[1] = 'Special Purpose with Minor Improvements'
            elif ctyluc == '970':
                row[1] = 'Special Purpose Common Area'
            elif ctyluc == '910':
                row[1] = 'Cemeteries'
            elif ctyluc == '900':
                row[1] = 'Parks for Public Use'
            elif ctyluc == '711':
                row[1] = 'Communication, Transportation, and Utility Property of a Local Nature Under Construction'
            elif ctyluc == '710':
                row[1] = 'Communication, Transportation, and Utility Property of a Local Nature'
            elif ctyluc == '700':
                row[1] = 'Operating Communication, Transportation, and Utility Property of an Interstate or Intercounty Nature'
            elif ctyluc == '582':
                row[1] = 'Industrial with Minor Improvements - with structures insufficient to determine intended use'
            elif ctyluc == '580':
                row[1] = 'Industrial with Minor Improvements'
            elif ctyluc == '560':
                row[1] = 'Industrial Auxiliary Area'
            elif ctyluc == '510':
                row[1] = 'Commercial Industrial - retail or office use combined with Industrial use'
            elif ctyluc == '500':
                row[1] = 'General Industrial - light industry, trucking and warehousing, service, repair, etc.'
            elif ctyluc == '480':
                row[1] = 'Commercial with Minor Improvements'
            elif ctyluc == '470':
                row[1] = 'Commercial Common Area'
            elif ctyluc == '460':
                row[1] = 'Commercial Auxiliary Area'
            elif ctyluc == '450':
                row[1] = 'Golf Course'
            elif ctyluc == '440':
                row[1] = 'Commercial Recreation'
            elif ctyluc == '430':
                row[1] = 'Commercial Living Accommodations'
            elif ctyluc == '420':
                row[1] = 'Casino or Hotel Casino'
            elif ctyluc == '410':
                row[1] = 'Offices, Professional and Business Services'
            elif ctyluc == '402':
                row[1] = 'Parking and/or Parking Structures'
            elif ctyluc == '400':
                row[1] = 'General Commercial'
            elif ctyluc == '390':
                row[1] = 'Mixed Use with Multi-Family Residential as primary use'
            elif ctyluc == '382':
                row[1] = 'Multi-Family Residential with Minor Improvements - No livable structures'
            elif ctyluc == '380':
                row[1] = 'Multi-Family Residential with Minor Improvements'
            elif ctyluc == '370':
                row[1] = 'Multi-Family Residential Common Area'
            elif ctyluc == '360':
                row[1] = 'Multi-Family Residential Auxiliary Area'
            elif ctyluc == '350':
                row[1] = 'Manufactured Home Park - Ten or More Manufactured Home Units'
            elif ctyluc == '341':
                row[1] = 'Five or More Units - High Rise Under Construction'
            elif ctyluc == '340':
                row[1] = 'Five or More Units - High Rise'
            elif ctyluc == '333':
                row[1] = 'Exempt or Partially Exempt Apartment Building'
            elif ctyluc == '331':
                row[1] = 'Five or More Units - Low Rise Under Construction'
            elif ctyluc == '330':
                row[1] = 'Five or More Units - Low Rise'
            elif ctyluc == '321':
                row[1] = 'Three to Four Units Under Construction'
            elif ctyluc == '320':
                row[1] = 'Three to Four Units'
            elif ctyluc == '313':
                row[1] = 'Multi-Family Residence with Manufactured Home Conversion'
            elif ctyluc == '311':
                row[1] = 'Two Single Family Units Under Construction'
            elif ctyluc == '310':
                row[1] = 'Two Single Family Units'
            elif ctyluc == '301':
                row[1] = 'Duplex Under Construction'
            elif ctyluc == '300':
                row[1] = 'Duplex'
            elif ctyluc == '290':
                row[1] = 'Mixed Use with Single Family Residential as primary use'
            elif ctyluc == '282':
                row[1] = 'Single Family Residential with Minor Improvements - No livable structures'
            elif ctyluc == '280':
                row[1] = 'Single Family Residential with Minor Improvements'
            elif ctyluc == '270':
                row[1] = 'Single Family Residential Common Area'
            elif ctyluc == '260':
                row[1] = 'Single Family Residential Auxiliary Area'
            elif ctyluc == '240':
                row[1] = 'Individual Residential Unit - Townhouse or Row House'
            elif ctyluc == '236':
                row[1] = 'Personal Property Manufactured Home Secured'
            elif ctyluc == '233':
                row[1] = 'Secured Manufactured Home with Site Built Additions (Not Converted)'
            elif ctyluc == '232':
                row[1] = 'Manufactured Home - Unsecured with Site Built Additions'
            elif ctyluc == '231':
                row[1] = 'Manufacture Home Conversions Pending'
            elif ctyluc == '230':
                row[1] = 'Personal Property Manufactured Home on the Unsecured Roll'
            elif ctyluc == '222':
                row[1] = 'Manufactured Home (Converted) with Site Built Additions'
            elif ctyluc == '220':
                row[1] = 'Manufactured Home Converted to Real Property'
            elif ctyluc == '211':
                row[1] = 'Individual Unit in a Multiple Unit Building Under Construction'
            elif ctyluc == '210':
                row[1] = 'Individual Unit in a Multiple Unit Building'
            elif ctyluc == '201':
                row[1] = 'Single Family Residence Under Construction'
            elif ctyluc == '200':
                row[1] = 'Single Family Residence'
            elif ctyluc == '190':
                row[1] = 'Vacant - Public Use Lands'
            elif ctyluc == '150':
                row[1] = 'Vacant - Industrial'
            elif ctyluc == '140':
                row[1] = 'Vacant - Commercial'
            elif ctyluc == '130':
                row[1] = 'Vacant - Multi-Residential'
            elif ctyluc == '120':
                row[1] = 'Vacant - Single Family Residential'
            elif ctyluc == '117':
                row[1] = 'Vacant - Roads/Easements'
            elif ctyluc == '110':
                row[1] = 'Vacant - Splinter and Other Unbuildable'
            elif ctyluc == '108':
                row[1] = 'Vacant - Patented Mining Claim, Not Mined'
            elif ctyluc == '100':
                row[1] = 'Vacant - Unknown/Other'
            elif ctyluc is None:
                row[1] == ''
            cursor.updateRow(row)        
        # Set El Dorado County Land Use Description fields
        if (row[0] != None or row[0] != "") and (row[3] == 'EL'):
            if ctyluc in ('03', '29', '31', '32', '34', '36', '37', '38', '39', '41', '42', '43', '44', '45', '46', '47', '48', 
                          '65', '67', '68', '82', '91', '93'):
                row[2] = "Commercial"
            elif ctyluc == '14':
                row[2] = "Condominium"
            elif ctyluc == '89':
                row[2] = "Condominium Common Area"
            elif ctyluc in ('01', '07', '12', '13', '16', '18', '19', '28', '35'):
                row[2] = "Multi-Family Residential"
            elif ctyluc in ('25', '26', '50', '51', '52', '55', '56', '60', '70', '75', '79'):
                row[2] = "Open Space"
            elif ctyluc in ('90', '92', '94', '96', '97', '98', '99'):
                row[2] = "Public Service"
            elif ctyluc in ('61', '62', '63', '64'):
                row[2] = "Recreation"
            elif ctyluc in ('06', '11', '15', '22', '23'):
                row[2] = "Single Family Residential"     
            elif ctyluc in ('33', '80', '81'):
                row[2] = "Tourist Accommodation"
            elif ctyluc in ('00', '02', '05', '17', '21', '24', '30', '40'):
                row[2] = "Vacant"
            elif ctyluc is None:
                row[2] == ''
        if (row[0] != None or row[0] != "") and (row[3] == 'EL'):
            if ctyluc == '98':
                row[1] = 'DEV MSC FIRE SUPPRESSION FACILITIES'
            elif ctyluc == '96':
                row[1] = 'DEV MSC CEMETERIES'
            elif ctyluc == '94':
                row[1] = 'DEV MSC SCHOOLS - LARGE (101+ STUDENTS)'
            elif ctyluc == '93':
                row[1] = 'DEV MSC SCHOOLS - MEDIUM (13-100 STUDENTS)'
            elif ctyluc == '92':
                row[1] = 'DEV MSC SCHOOLS - SMALL (1-12 STUDENTS)'
            elif ctyluc == '90':
                row[1] = 'UTL IND PUBLIC UTILITY (ON STATE ASSESSED ROLL)'
            elif ctyluc == '84':
                row[1] = 'DEV MSC TEMPORARY USE CODE FOR PROJECT 184'
            elif ctyluc == '82':
                row[1] = 'DEV COM PARKING LOT'
            elif ctyluc == '81':
                row[1] = 'DEV MSC UNDERLYING INTEREST IN TIME SHARE PROJ'
            elif ctyluc == '79':
                row[1] = 'RLU MSC ENV. SENSITIVE LAND - RESTRICTED USE'
            elif ctyluc == '68':
                row[1] = 'DEV COM MARINAS'
            elif ctyluc == '65':
                row[1] = 'DEV COM RESTAURANT'
            elif ctyluc == '64':
                row[1] = 'DEV MSC SKI RESORTS'
            elif ctyluc == '63':
                row[1] = 'DEV MSC CAMPGROUNDS'
            elif ctyluc == '62':
                row[1] = 'DEV MSC COMMUNITY ORIENTED FACILITIES'
            elif ctyluc == '61':
                row[1] = 'DEV MSC MISC. IMPROVED RECREATIONAL'
            elif ctyluc == '60':
                row[1] = 'VAC MSC VACANT RECREATIONAL LAND'
            elif ctyluc == '50':
                row[1] = 'TPZ MSC TIMBER PRESERVE ZONING - ACTIVE'
            elif ctyluc == '48':
                row[1] = 'DEV IND OFFICES'
            elif ctyluc == '47':
                row[1] = 'DEV IND HOSPITALS & CONVALESCENT HOSPITALS'
            elif ctyluc == '46':
                row[1] = 'DEV IND MEDICAL/DENTAL/VET OFFICES'
            elif ctyluc == '45':
                row[1] = 'DEV IND LIGHT MANUFACTURING'
            elif ctyluc == '43':
                row[1] = 'DEV IND WAREHOUSES'
            elif ctyluc == '42':
                row[1] = 'DEV IND MINI-WAREHOUSES (MINI-STORAGE)'
            elif ctyluc == '41':
                row[1] = 'DEV IND MISC. IMPROVED INDUSTRIAL PROPERTY'
            elif ctyluc == '40':
                row[1] = 'VAC IND VACANT INDUSTRIAL LAND'
            elif ctyluc == '39':
                row[1] = 'DEV COM SUPERMARKETS'
            elif ctyluc == '38':
                row[1] = 'DEV COM RETAIL STORES >15,000 SQ. FT.'
            elif ctyluc == '37':
                row[1] = 'DEV COM RETAIL STORES 5,001-15,000 SQ. FT.'
            elif ctyluc == '36':
                row[1] = 'DEV COM RETAIL STORES <=5,000 SQ. FT.'
            elif ctyluc == '35':
                row[1] = 'DEV COM MOBILE HOME PARKS'
            elif ctyluc == '34':
                row[1] = 'DEV COM SERVICE STATION'
            elif ctyluc == '33':
                row[1] = 'DEV COM MOTEL, HOTEL'
            elif ctyluc == '31':
                row[1] = 'DEV COM MISC. IMPROVED COMMERCIAL'
            elif ctyluc == '30':
                row[1] = 'VAC COM VACANT COMMERCIAL LAND'
            elif ctyluc == '29':
                row[1] = 'DEV MSC RURAL NON-RES. IMPROVEMENT 2.51-20.0 AC.'
            elif ctyluc == '26':
                row[1] = 'AGP MSC RURAL RESTRICTIVE ZONING - NON-RENEWAL'
            elif ctyluc == '25':
                row[1] = 'AGP MSC RURAL RESTRICTIVE ZONING - CLCA (ACTIVE)'
            elif ctyluc == '24':
                row[1] = 'VAC RES RURAL RES. LAND 20+ MINOR NON-RES IMPR'
            elif ctyluc == '23':
                row[1] = 'DEV RES RURAL RES. 20+ AC. 1 RES. UNIT'
            elif ctyluc == '22':
                row[1] = 'DEV RES RURAL RES. 2.51-20.0 AC. 1 SF UNIT'
            elif ctyluc == '21':
                row[1] = 'VAC RES VAC RURAL RES LAND 2.51-20.0 AC. 1 UNIT'
            elif ctyluc == '17':
                row[1] = 'VAC MSC SUBJ. TO OPEN SPACE CONTRACT (NOT CLCA)'
            elif ctyluc == '16':
                row[1] = 'DEV RES MOBILE HOME ON RENTED LAND'
            elif ctyluc == '15':
                row[1] = 'DEV RES RESIDENCE ON LEASED LAND'
            elif ctyluc == '14':
                row[1] = 'DEV MFR CONDOMINIUMS & TOWNHOUSES'
            elif ctyluc == '13':
                row[1] = 'DEV MFR MULTI-RESIDENTIAL 4+ UNITS'
            elif ctyluc == '12':
                row[1] = 'DEV MFR MULTI-RESIDENTIAL 2-3 UNITS'
            elif ctyluc == '11':
                row[1] = 'DEV RES SINGLE FAM. RES. <=2.5 AC.(INC. MAN. HMS'
            elif ctyluc == '07':
                row[1] = 'DEV MFR RETIREMENT HOUSING'
            elif ctyluc == '05':
                row[1] = 'VAC MFR VACANT MULTI-RES. LAND 4+ UNITS ALLOWED'
            elif ctyluc == '03':
                row[1] = 'DEV COM PLACE OF WORSHIP'
            elif ctyluc == '02':
                row[1] = 'VAC RES NON-RES. IMPROVEMENTS <=2.5 AC.'
            elif ctyluc == '00':
                row[1] = 'VAC RES VACANT RES. LAND <=2.5 AC. 1-3 UNITS'
            elif ctyluc is None:
                row[1] == ''
            cursor.updateRow(row)
        # set Placer TRPA land use description
        if (row[0] != None or row[0] != "") and (row[3] == 'PL'):
            if ctyluc in ('07', '11', '12', '13', '14', '15', '17', '19', '21', '22', '23', 
                          '24', '25', '26', '27', '29', '31', '32', '36', '37', '38', 
                          '39', '62', '63', '71', '88'):
                row[2] = "Commercial"
            elif ctyluc == ('04'):
                row[2] = "Condominium"
            elif ctyluc == '89':
                row[2] = "Condominium Common Area"
            elif ctyluc in ('02', '03', '04', '05', '09', '28'):
                row[2] = "Multi-Family Residential"
            elif ctyluc in ('56', '55', '60', '61', '87', '90'):
                row[2] = "Open Space"
            elif ctyluc in ('72', '76', '77', '81'):
                row[2] = "Public Service"
            elif ctyluc in ('65', '66', '67', '68', '69'):
                row[2] = "Recreation"
            elif ctyluc in ('01', '08', '16'):
                row[2] = "Single Family Residential"     
            elif ctyluc in ('06', '18', '64'):
                row[2] = "Tourist Accommodation"
            elif ctyluc in ('00', '10', '20', '30'):
                row[2] = "Vacant"
            elif ctyluc is None:
                row[2] == ''
        if (row[0] != None or row[0] != "") and (row[3] == 'PL'):
            if ctyluc == '90':
                row[1] = 'GREENBELT'
            elif ctyluc == '89':
                row[1] = 'COMMON AREA'
            elif ctyluc == '88':
                row[1] = 'HIGHWAYS, ROADS, STREETS'
            elif ctyluc == '87':
                row[1] = 'RIVERS, LAKES, RESERVOIR, CANAL'
            elif ctyluc == '81':
                row[1] = 'UTILITIES, PUBLIC & PRIVATE'
            elif ctyluc == '77':
                row[1] = 'CEMETERIES'
            elif ctyluc == '76':
                row[1] = 'MISC. PUBLIC BUILDINGS'
            elif ctyluc == '72':
                row[1] = 'SCHOOLS'
            elif ctyluc == '71':
                row[1] = 'CHURCHES'
            elif ctyluc == '69':
                row[1] = 'MISCELLANEOUS RECREATIONAL'
            elif ctyluc == '68':
                row[1] = 'CAMPS & PARKS, GENERAL'
            elif ctyluc == '67':
                row[1] = 'SKI FACILITY'
            elif ctyluc == '66':
                row[1] = 'GOLF COURSE'
            elif ctyluc == '65':
                row[1] = 'TENNIS, SWIMMING CLUBS'
            elif ctyluc == '64':
                row[1] = 'LODGES, HALLS'
            elif ctyluc == '63':
                row[1] = 'MARINA, PIER'
            elif ctyluc == '62':
                row[1] = 'THEATER, BOWLING ALLEY'
            elif ctyluc == '61':
                row[1] = 'NON-PROFIT CAMPS/PARKS'
            elif ctyluc == '60':
                row[1] = 'CONSERVATION EASEMENT RESTRICTIONS'
            elif ctyluc == '56':
                row[1] = 'TIMBERLAND, ZONED TPZ'
            elif ctyluc == '55':
                row[1] = 'TIMBERLAND, UNRESTRICTED'
            elif ctyluc == '39':
                row[1] = 'MISCELLANEOUS INDUSTRIAL'
            elif ctyluc == '38':
                row[1] = 'WAREHOUSE'
            elif ctyluc == '37':
                row[1] = 'MINI-STORAGE, COVERED STORAGE'
            elif ctyluc == '36':
                row[1] = 'UNCOVERED STORAGE, WRECKING YARD'
            elif ctyluc == '32':
                row[1] = 'HEAVY INDUSTRIAL'
            elif ctyluc == '31':
                row[1] = 'LIGHT INDUSTRIAL'
            elif ctyluc == '30':
                row[1] = 'VACANT INDUSTRIAL'
            elif ctyluc == '29':
                row[1] = "MISCELLANEOUS COMM'L"
            elif ctyluc == '28':
                row[1] = 'MOBILE HOME PARK'
            elif ctyluc == '27':
                row[1] = 'PARKING LOTS'
            elif ctyluc == '26':
                row[1] = 'AUTO SALES, REPAIR'
            elif ctyluc == '25':
                row[1] = 'SERVICE STATION'
            elif ctyluc == '24':
                row[1] = 'MINI-MARKET WITH GAS'
            elif ctyluc == '23':
                row[1] = "BANKS, S&L'S, CREDIT UNION"
            elif ctyluc == '22':
                row[1] = 'FAST FOOD RESTAURANT'
            elif ctyluc == '21':
                row[1] = 'RESTAURANTS, COCKTAIL LOUNGES'
            elif ctyluc == '20':
                row[1] = 'VACANT, COMMERCIAL'
            elif ctyluc == '19':
                row[1] = 'OFFICE MEDICAL/DENTAL'
            elif ctyluc == '18':
                row[1] = 'HOTELS, MOTELS, RESORTS'
            elif ctyluc == '17':
                row[1] = 'OFFICE GENERAL'
            elif ctyluc == '16':
                row[1] = 'RESIDENCE ON COMMERCIAL LAND'
            elif ctyluc == '15':
                row[1] = 'SHOPPING CENTER'
            elif ctyluc == '14':
                row[1] = 'OFFICE CONDO'
            elif ctyluc == '13':
                row[1] = 'MINI-MARKETS, NO GAS'
            elif ctyluc == '12':
                row[1] = 'SUBURBAN STORE'
            elif ctyluc == '11':
                row[1] = 'COMMERCIAL STORE'
            elif ctyluc == '10':
                row[1] = 'VACANT, SUBDIVIDED RESIDENTIAL'
            elif ctyluc == '09':
                row[1] = 'MOBILE HOME IN M H PARK'
            elif ctyluc == '08':
                row[1] = 'MOBILE HOME OUTSIDE OF PARK'
            elif ctyluc == '07':
                row[1] = 'RESIDENTIAL, AUXILIARY IMP'
            elif ctyluc == '06':
                row[1] = 'TIMESHARES'
            elif ctyluc == '05':
                row[1] = 'APARTMENTS, 4 UNITS OR MORE'
            elif ctyluc == '04':
                row[1] = 'SINGLE FAM RES, CONDO'
            elif ctyluc == '03':
                row[1] = '3 SINGLE FAM RES, TRIPLEX'
            elif ctyluc == '02':
                row[1] = '2 SINGLE FAM RES, DUPLEX'
            elif ctyluc == '01':
                row[1] = 'SINGLE FAM RES, HALF PLEX'
            elif ctyluc == '00':
                row[1] = 'VACANT, ALL TYPES-NOT ASGND'
            elif ctyluc is None:
                row[1] == ''
            cursor.updateRow(row)
# delete cursor
del cursor
print ("The 'EXISTING_LANDUSE' field in the parcel data has been updated")
# log.info("The 'EXISTING_LANDUSE' field in the parcel data has been updated")
result = arcpy.GetCount_management(ParcelLayer)
print('{} has {} records now.'.format(ParcelLayer, result[0]))

### Regional Landuse Update --------------------------------------------------------------------------------------###
print("Starting the Regional Land Use Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Starting the Regional Land Use Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))

# Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_RegionalLandUse, ParcelPoint_RegionalLandUse, 
                           "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print ("Finished the Regional Land Use Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Finished the Regional Land Use Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))

# transfer attributes to Parcel Layer
fieldJoinCalc_multikey(ParcelLayer, ['APN_TRPA', 'COUNTY_TRPA'],['REGIONAL_LANDUSE_TRPA'], 
              ParcelPoint_RegionalLandUse, ['APN_TRPA', 'COUNTY_TRPA'],['Description'])
print ("The 'REGIONAL_LANDUSE' field in the parcel data has been updated")
# log.info("The 'REGIONAL_LANDUSE' field in the parcel data has been updated")

## Estimated Coverage Allowed Attirbute Update ------------------------------------------------------------------###
print("Starting the Estimated Coverage Allowed Identity Overlay: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Starting the Estimated Coverage Allowed Identity Overlay: " + strftime("%Y-%m-%d %H:%M:%S"))

# create out table for the stats sum
outTable =  memory + "id_Parcel_Bailey_Table"

# Create Identity Output Layer
id_ParcelLyr_BaileyLyr = memory + "id_Parcel_Bailey"

# Create Impervious Layer
Bailey_lyr = memory + "Bailey_lyr"

# Create Identity Layer
identity_layer = memory + "bailey_identity_layer"

# Make a layer from the feature class Impervious that only passes Ftype = 'building' and 'other'
arcpy.MakeFeatureLayer_management(sde_Bailey, Bailey_lyr)
print ("Created feature layer of Bailey Soils")

# Process: Use the Identity function
print ("Starting Identity: "+ strftime("%Y-%m-%d %H:%M:%S"))
arcpy.Identity_analysis (ParcelLayer, Bailey_lyr, id_ParcelLyr_BaileyLyr)
print ("Finished Identity: "+ strftime("%Y-%m-%d %H:%M:%S"))

# Add SqFt field
arcpy.management.AddField(id_ParcelLyr_BaileyLyr, "SqFt", "DOUBLE", "", "", "", 
                          "Square Feet", "NULLABLE", "NON_REQUIRED", "")

# Make a layer from the feature class Impervious that only passes Ftype = 'building' and 'other'
arcpy.MakeFeatureLayer_management(id_ParcelLyr_BaileyLyr, identity_layer, 
                                  where_clause = "NOT CAPABILITY in ('WB', '-1', '0')")

# calculate geometry of output identity
arcpy.CalculateField_management(identity_layer, "SqFt", "!shape.area@SQUAREFEET!", "PYTHON3", "")

# multiply square footage by bailey coefficents
with arcpy.da.UpdateCursor(identity_layer, ['CAPABILITY', 'SqFt', 'PERCENT_COVERAGE_ALLOWED']) as cur:
    for row in cur:
        if row[0] != ('','WB'):
            row[1] = row[1]*row[2]
        else:
            row[1] == 0
        cur.updateRow(row)
del cur    
# Sum the square footage
arcpy.Statistics_analysis(identity_layer, outTable, [["SqFt", "SUM"]], ["APN_TRPA","COUNTY_TRPA"])

print("Finsished the Estimated Coverage Allowed Identity Overlay: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Finsished the Estimated Coverage Allowed Identity Overlay: " + strftime("%Y-%m-%d %H:%M:%S"))

## Join parcel sums back to parcel layer and calculate field

# transfer attributes to Parcel Layer
fieldJoinCalc_multikey(ParcelLayer, ['APN_TRPA', 'COUNTY_TRPA'],['ESTIMATED_COVERAGE_ALLOWED_TRPA'], 
              outTable, ['APN_TRPA', 'COUNTY_TRPA'],['SUM_SqFt'])
print ("The 'ESTIMATED_COVERAGE_ALLOWED' field in the parcel data has been updated")

### Impervious Surface Attrigute Update --------------------------------------------------------------------------###
# create out table for the stats sum
outTable =  memory +"id_Parcel_Imp_Table"

# Create Identity Output Layer
id_ParcelLyr_ImperviousLyr = memory + "id_Parcel_Impervious"

# Create Impervious Layer
Impervious_lyr = memory + "Impervious_lyr"

# Create Identity Layer
identity_layer = memory + "identity_layer"
    
# Make a layer from the feature class Impervious that only passes Ftype = 'building' and 'other'
arcpy.MakeFeatureLayer_management(sde_Impervious, Impervious_lyr)

# Process: Use the Identity function
print ("Starting Identity of Imperviuos Surface by parcel: "+ strftime("%Y-%m-%d %H:%M:%S"))
arcpy.Identity_analysis (ParcelLayer, Impervious_lyr, id_ParcelLyr_ImperviousLyr)
print ("Finished Identity of Imperviuos Surface by parcel:: "+ strftime("%Y-%m-%d %H:%M:%S"))

# Add SqFt field
arcpy.management.AddField(id_ParcelLyr_ImperviousLyr, 
                          "SqFt", "DOUBLE", "", "", "", "Square Feet", "NULLABLE", "NON_REQUIRED", "")

# Make a layer from the feature class Impervious that only passes Ftype = 'building' and 'other'
arcpy.MakeFeatureLayer_management(id_ParcelLyr_ImperviousLyr, identity_layer, 
                                  where_clause = "Feature IN ('Building', 'Road', 'Other', 'Driveway')")

# calculate geometry of output identity
arcpy.CalculateField_management(identity_layer, "SqFt", "!shape.area@SQUAREFEET!", "PYTHON3", "")
                                                           
# Sum the square footage of buildings and other by APN
arcpy.Statistics_analysis(identity_layer, outTable, [["SqFt", "SUM"]], ["APN_TRPA","COUNTY_TRPA"])

# Join parcel sums back to parcel layer and calculate field "Impervious Surface Sq Ft"
# transfer attributes to Parcel Layer
fieldJoinCalc_multikey(ParcelLayer, ['APN_TRPA', 'COUNTY_TRPA'],['IMPERVIOUS_SURFACE_SQFT_TRPA'], 
              outTable, ['APN_TRPA', 'COUNTY_TRPA'],['SUM_SqFt'])
print ("The 'ImperviousCoverage_SqFt' field in the parcel data has been updated")

### Fire District Attribute Update -------------------------------------------------------------------------------###
print("Starting the Fire District Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Starting the Fire District Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# Process Fire District Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_FireDistrict, ParcelPoint_FireDistrict, 
                           "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print ("Finished the Fire District Spatial Join: "  + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Finished the Fire District Spatial Join: "  + strftime("%Y-%m-%d %H:%M:%S"))

# transfer attributes to Parcel Layer
fieldJoinCalc_multikey(ParcelLayer, ['APN_TRPA', 'COUNTY_TRPA'],['FIREPD_TRPA'], 
              ParcelPoint_FireDistrict, ['APN_TRPA', 'COUNTY_TRPA'],['DISTRICT'])
print ("The 'FIRE_PD' field has been updated")
# log.info("The 'FIRE_PD' field has been updated")

### Soil 1974 Attribute Update ------------------------------------------------------------------------------------### 
print("Starting the SOIL_1974 Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Starting the SOIL_1974 Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_NRCSSoils1974, ParcelPoint_Soils74, 
                           "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print ("Finished the SOIL_1974 Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Finished the SOIL_1974 Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))

# transfer attributes to Parcel Layer
fieldJoinCalc_multikey(ParcelLayer, ['APN_TRPA', 'COUNTY_TRPA'],['SOIL_1974_TRPA'], 
              ParcelPoint_Soils74, ['APN_TRPA', 'COUNTY_TRPA'],['MUSYM_74'])
print ("The 'SOIL_1974' field in the parcel data has been updated")
# log.info("The 'SOIL_1974' field in the parcel data has been updated")

### Soil 2003 Attribute Update -----------------------------------------------------------------------------------###
print("Starting the SOIL_2003 Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Starting the SOIL_2003 Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))

# Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_NRCSSoils2003, ParcelPoint_Soils03, 
                           "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print ("Finished the SOIL_2003 Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Finished the SOIL_2003 Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))

# transfer attributes to Parcel Layer
fieldJoinCalc_multikey(ParcelLayer, ['APN_TRPA', 'COUNTY_TRPA'],['SOIL_2003_TRPA'], 
              ParcelPoint_Soils03, ['APN_TRPA', 'COUNTY_TRPA'],['MUSYM_03'])
print ("The 'SOIL_2003' field in the parcel data has been updated.")
# log.info("The 'SOIL_2003' field in the parcel data has been updated: "  + strftime("%Y-%m-%d %H:%M:%S"))

### HRA Attribute Upate -------------------------------------------------------------------------------------###
print("Starting the Hydrologic Area Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Starting the Hydrologic Area Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))

# Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_HydroArea, ParcelPoint_HydroArea, 
                           "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print ("Finished the Hydrologic Area Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Finished the Hydrologic Area Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))

# transfer attributes to Parcel Layer
fieldJoinCalc_multikey(ParcelLayer, ['APN_TRPA', 'COUNTY_TRPA'],['HRA_NAME_TRPA'], 
              ParcelPoint_HydroArea, ['APN_TRPA', 'COUNTY_TRPA'],['HRA_NAME'])
print ("The 'HRA_NAME' field in the parcel data has been updated")
# log.info("The 'HRA_NAME' field in the parcel data has been updated")

### Watshed Attribute Update -------------------------------------------------------------------------------###
print("Starting the Watershed Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Starting the Watershed Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))

# Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_Watershed, ParcelPoint_Watershed, 
                           "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print ("Finished the Watershed Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Finished the Watershed Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))

# transfer attributes to Parcel Layer
fieldJoinCalc_multikey(ParcelLayer, ['APN_TRPA', 'COUNTY_TRPA'],['WATERSHED_NUMBER_TRPA'], 
              ParcelPoint_Watershed, ['APN_TRPA', 'COUNTY_TRPA'],['NUMBER'])
# replace null with 0
with arcpy.da.UpdateCursor(ParcelLayer, ['WATERSHED_NUMBER_TRPA']) as cursor:
    for row in cursor:
        if row[0] is None:
            # If the value is null, replace it with 0
            row[0] = 0
            cursor.updateRow(row)
del cursor   
print ("The 'WATERSHED_NUMBER' field in the parcel data has been updated")
# log.info("The 'WATERSHED_NUMBER' field in the parcel data has been updated")

# transfer attributes to Parcel Layer
fieldJoinCalc_multikey(ParcelLayer, ['APN_TRPA', 'COUNTY_TRPA'],['WATERSHED_NAME_TRPA'], 
              ParcelPoint_Watershed, ['APN_TRPA', 'COUNTY_TRPA'],['NAME'])
print ("The 'WATERSHED_NAME' field in the parcel data has been updated")
# log.info("The 'WATERSHED_NAME' field in the parcel data has been updated")

# transfer attributes to Parcel Layer
fieldJoinCalc_multikey(ParcelLayer, ['APN_TRPA', 'COUNTY_TRPA'],['PRIORITY_WATERSHED_TRPA'], 
              ParcelPoint_Watershed, ['APN_TRPA', 'COUNTY_TRPA'],['PRIORITY'])
# replace null with 0
with arcpy.da.UpdateCursor(ParcelLayer, ['PRIORITY_WATERSHED_TRPA']) as cursor:
    for row in cursor:
        if row[0] is None:
            # If the value is null, replace it with 0
            row[0] = 0
            cursor.updateRow(row)
del cursor
print ("The 'PRIORITY_WATERSHED' field in the parcel data has been updated")
# log.info("The 'PRIORITY_WATERSHED' field in the parcel data has been updated")

### Local Plan Attribute Update -----------------------------------------------------------------------------###
print("Starting the Local Plan Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Starting the Local Plan Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_LocalPlan, ParcelPoint_LocalPlan, 
                           "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print ("Finished the Local Plan Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Finished the Local Plan Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))

# transfer attributes to Parcel Layer
fieldJoinCalc_multikey(ParcelLayer, ['APN_TRPA', 'COUNTY_TRPA'],['PLAN_ID_TRPA'], 
              ParcelPoint_LocalPlan, ['APN_TRPA', 'COUNTY_TRPA'],['PLAN_ID'])
print ("The 'PLAN_ID' field in the parcel data has been updated")
# log.info("The 'PLAN_ID' field in the parcel data has been updated")

# transfer attributes to Parcel Layer
fieldJoinCalc_multikey(ParcelLayer, ['APN_TRPA', 'COUNTY_TRPA'],['PLAN_NAME_TRPA'], 
              ParcelPoint_LocalPlan, ['APN_TRPA', 'COUNTY_TRPA'],['PLAN_NAME'])
print ("The 'PLAN_NAME' field in the parcel data has been updated")
# log.info("The 'PLAN_NAME' field in the parcel data has been updated")

# transfer attributes to Parcel Layer
fieldJoinCalc_multikey(ParcelLayer, ['APN_TRPA', 'COUNTY_TRPA'],['PLAN_TYPE_TRPA'], 
              ParcelPoint_LocalPlan, ['APN_TRPA', 'COUNTY_TRPA'],['PLAN_TYPE'])
print ("The 'PLAN_TYPE' field in the parcel data has been updated")
# log.info("The 'PLAN_NAME' field in the parcel data has been updated")

# transfer attributes to Parcel Layer
fieldJoinCalc_multikey(ParcelLayer, ['APN_TRPA', 'COUNTY_TRPA'],['LOCAL_PLAN_HYPERLINK_TRPA'], 
              ParcelPoint_LocalPlan, ['APN_TRPA', 'COUNTY_TRPA'],['File_URL'])
print ("The 'LOCAL_PLAN_HYPERLINK' field in the parcel data has been updated")
# log.info("The 'LOCAL_PLAN_HYPERLINK' field in the parcel data has been updated")

### Town Center Attribute Update --------------------------------------------------------------------------------### 
print("Starting the Town Center Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Starting the Town Center Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_TownCenter, ParcelPoint_TownCenter, 
                           "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")

print("Finished the Town Center Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Finished the Town Center Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))

# transfer attributes to Parcel Layer
fieldJoinCalc_multikey(ParcelLayer, ['APN_TRPA', 'COUNTY_TRPA'],['TOWN_CENTER_TRPA'], 
              ParcelPoint_TownCenter, ['APN_TRPA', 'COUNTY_TRPA'],['NAME'])
print("The 'TOWN_CENTER' field in the parcel data has been updated")
# log.info("The 'TOWN_CENTER' field in the parcel data has been updated")

### Town Center Buffer Attribute Update --------------------------------------------------------------------------###
print("Starting the Town Center Buffer Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Starting the Town Center Buffer Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))

# Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_TownCenterBuffer, ParcelPoint_TownCenterBuffer, 
                           "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print ("Finished the Town Center Buffer Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Finished the Town Center Buffer Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))

# transfer attributes to Parcel Layer
fieldJoinCalc_multikey(ParcelLayer, ['APN_TRPA', 'COUNTY_TRPA'],['LOCATION_TO_TOWNCENTER_TRPA'], 
              ParcelPoint_TownCenterBuffer, ['APN_TRPA', 'COUNTY_TRPA'],['BUFFER_NAME'])
print ("The 'LOCATION_TO_TOWNCENTER' field in the parcel data has been updated")
# log.info("The 'LOCATION_TO_TOWNCENTER' field in the parcel data has been updated")

### Catchment Attribute Update ------------------------------------------------------------------------------------### 
print("Starting the Catchment Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Starting the Catchment Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_Catchment, ParcelPoint_Catchment, 
                           "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print("Finished the Catchment Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Finished the Catchment Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))

# transfer attributes to Parcel Layer
fieldJoinCalc_multikey(ParcelLayer, ['APN_TRPA', 'COUNTY_TRPA'],['CATCHMENT_TRPA'], 
              ParcelPoint_Catchment, ['APN_TRPA', 'COUNTY_TRPA'],['Name'])
print ("The 'Catchment' field in the parcel data has been updated")
# log.info("The 'Catchment' field in the parcel data has been updated")

### Littoral Parcel Attribute Update -----------------------------------------------------------------------------###
print("Identifying Littoral parcels: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info('Identifying Littoral parcels: ' + strftime("%Y-%m-%d %H:%M:%S"))

# Select parcels that have their center within
littoralSelect = arcpy.SelectLayerByLocation_management(ParcelLayer, 
                                                          'HAVE_THEIR_CENTER_IN', 
                                                           sde_Littoral, 
                                                           0, 
                                                          'NEW_SELECTION')
# Update field 1= yes 0 = no
with arcpy.da.UpdateCursor(littoralSelect, ['LITTORAL_TRPA']) as cursor:
    for row in cursor:
        row[0] = '1'
        cursor.updateRow(row) 
del cursor 

# switch the selection
litSelect = arcpy.SelectLayerByAttribute_management(littoralSelect,'SWITCH_SELECTION')

# update other parcels
with arcpy.da.UpdateCursor(litSelect, ['LITTORAL_TRPA']) as cursor:
    for row in cursor:
        row[0] = '0'
        cursor.updateRow(row)
del cursor
print("Littoral parcels updated: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Littoral parcels Updated: " + strftime("%Y-%m-%d %H:%M:%S"))

### Tolerance ID -------------------------------------------------------------------------------------------------###
print("Starting the Tolerance District Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_Tolerance, ParcelPoint_Tolerance, 
                           "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "INTERSECT", "", "")
print ("Finished the Tolerance District Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))

# transfer attributes to Parcel Layer
fieldJoinCalc_multikey(ParcelLayer, ['APN_TRPA', 'COUNTY_TRPA'],['TOLERANCE_ID_TRPA'], 
              ParcelPoint_Tolerance, ['APN_TRPA', 'COUNTY_TRPA'],['DISTRICT'])
print ("The Tolerance ID field in the parcel data has been updated")

### Index 1987 Attribute Update ----------------------------------------------------------------------------------###
print("Starting the 1987 Index Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Starting the 1987 Index Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))

# Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_Index1987, ParcelPoint_Index1987, 
                           "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print ("Finished the 1987 Index Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Finished the 1987 Index Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))

# transfer attributes to Parcel Layer
fieldJoinCalc_multikey(ParcelLayer, ['APN_TRPA', 'COUNTY_TRPA'],['INDEX_1987_TRPA'], 
              ParcelPoint_Index1987, ['APN_TRPA', 'COUNTY_TRPA'],['MAP_NUMBER'])
print("The 'INDEX_1987' field in the parcel data has been updated")
# log.info("The 'INDEX_1987' field in the parcel data has been updated")

# transfer attributes to Parcel Layer
fieldJoinCalc_multikey(ParcelLayer, ['APN_TRPA', 'COUNTY_TRPA'],['INDEX_1987_HYPERLINK_TRPA'], 
              ParcelPoint_Index1987, ['APN_TRPA', 'COUNTY_TRPA'],['MAP_PATH'])
print ("The 'INDEX_1987_HYPERLINK' field in the parcel data has been updated")
# log.info("The 'INDEX_1987_HYPERLINK' field in the parcel data has been updated")

### Postal Town Field --------------------------------------------------------------------------------------------### 
print("Starting the Postal Town Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Starting the Postal Town Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))

# Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_Zip, ParcelPoint_PstlTown, 
                           "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print ("Finished the Postal Town Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Finished the Postal Town Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))

# transfer attributes to Parcel Layer
fieldJoinCalc_multikey(ParcelLayer, ['APN_TRPA', 'COUNTY_TRPA'],['PSTL_TOWN_TRPA'], 
              ParcelPoint_PstlTown, ['APN_TRPA', 'COUNTY_TRPA'],['PO_NAME'])
print ("The 'PSTL_TOWN' field in the parcel data has been updated")
# log.info("The 'PSTL_TOWN' field in the parcel data has been updated")

### Postal ZIP ---------------------------------------------------------------------------------------------------###

# transfer attributes to Parcel Layer
fieldJoinCalc_multikey(ParcelLayer, ['APN_TRPA', 'COUNTY_TRPA'],['PSTL_ZIP5_TRPA'], 
              ParcelPoint_PstlTown, ['APN_TRPA', 'COUNTY_TRPA'],['ZIP_CODE'])
print("The 'PSTL_ZIP5' field in the parcel data has been updated")
# log.info("The 'PSTL_ZIP5' field in the parcel data has been updated")

### CSLT Jurisdiction Update -------------------------------------------------------------------------------------###
print("Starting to select parcels within City of South Lake Tahoe: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Starting to select parcels within City of South Lake Tahoe: " + strftime("%Y-%m-%d %H:%M:%S"))

# select by location
csltParcels = arcpy.SelectLayerByLocation_management(ParcelLayer, "HAVE_THEIR_CENTER_IN", sde_CSLT, 0,   
                                                     "NEW_SELECTION")
# update jurisdcition field
with arcpy.da.UpdateCursor(csltParcels, ["JURISDICTION_TRPA"]) as cursor:
    for row in cursor:
        row[0] = "CSLT"
        # update all rows
        cursor.updateRow(row)
del cursor 
print("Finished updating parcels within City of South Lake Tahoe: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Finished updating parcels within City of South Lake Tahoe:  " + strftime("%Y-%m-%d %H:%M:%S"))
print("JURISDCITION field update with 'CSLT' values ")
# log.info("JURISDCITION field update with 'CSLT' values ")

### Zoning Attribute Update --------------------------------------------------------------------------------------###
# Spatial Join
print("Starting the Zoning Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Starting the Zoning Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
arcpy.SpatialJoin_analysis(ParcelPoint, sde_Zoning, ParcelPoint_Zoning, 
                           "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print("Finished the Zoning Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Finished the Zoning Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))

# transfer attributes to Parcel Layer
fieldJoinCalc_multikey(ParcelLayer, ['APN_TRPA', 'COUNTY_TRPA'],['ZONING_ID_TRPA'], 
              ParcelPoint_Zoning, ['APN_TRPA', 'COUNTY_TRPA'],['ZONING_ID'])
print("The Zoning ID field in the parcel data has been updated")
# log.info("The Zoning ID field in the parcel data has been updated")

# transfer attributes to Parcel Layer
fieldJoinCalc_multikey(ParcelLayer, ['APN_TRPA', 'COUNTY_TRPA'],['ZONING_DESCRIPTION_TRPA'], 
              ParcelPoint_Zoning, ['APN_TRPA', 'COUNTY_TRPA'],['ZONING_DESCRIPTION'])
print("The Zoning Description field in the parcel data has been updated")
# log.info("The Zoning Description field in the parcel data has been updated")

# transfer attributes to Parcel Layer
fieldJoinCalc_multikey(ParcelLayer, ['APN_TRPA', 'COUNTY_TRPA'],["DESIGN_GUIDELINES_HYPERLINK_TRPA"], 
              ParcelPoint_Zoning, ['APN_TRPA', 'COUNTY_TRPA'],["DESIGN_GUIDELINES_HYPERLINK"])
print("The DESIGN_GUIDELINES_HYPERLINK field in the parcel data has been updated")
# log.info("The DESIGN_GUIDELINES_HYPERLINK_TRPA field in the parcel data has been updated")

### TAZ Attirbute Update --------------------------------------------------------------------------------------###
# Spatial Join
print("Starting the TAZ Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Starting the TAZ Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
arcpy.SpatialJoin_analysis(ParcelPoint, sde_TAZ, ParcelPoint_TAZ, 
                           "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print("Finished the TAZ Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Finished the TAZ Spatial Join: " + strftime("%Y-%m-%d %H:%M:%S"))

# transfer attributes to Parcel Layer
fieldJoinCalc_multikey(ParcelLayer, ['APN_TRPA', 'COUNTY_TRPA'],['TAZ_TRPA'], 
              ParcelPoint_TAZ, ['APN_TRPA', 'COUNTY_TRPA'],["TAZ"])
print("The TAZ field in the parcel data has been updated")

### LTinfo Parcel Details Hyperlink Attribute Update -------------------------------------------------------------###
print("Creating LTinfo Hyperlinks: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Creating LTinfo Hyperlinks: " + strftime("%Y-%m-%d %H:%M:%S"))

# create ltinfo hyper link
with arcpy.da.UpdateCursor(ParcelLayer, ["APN_TRPA","LTINFO_HYPERLINK_TRPA"]) as cursor:
    for row in cursor:
        if not (row[0] == None):
            row[1] = 'https://parcels.laketahoeinfo.org/Parcel/Detail/'+ row[0]
        else:
            row[1] = ''
        cursor.updateRow(row)
del cursor
print("The LTINFO_HYPERLINK field in the parcel data has been updated")

### set within TRPA boundary -------------------------------------------------------------------------------------###
print("Identifying parcels within TRPA Boundary: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info('Identifying parcels within TRPA Boundary: ' + strftime("%Y-%m-%d %H:%M:%S"))

# Select all new parcels that have their center within
parcelSelect = arcpy.SelectLayerByLocation_management(ParcelLayer, 
                                                          'INTERSECT', 
                                                           sde_TRPAboundary, 
                                                           0, 
                                                          'NEW_SELECTION')

# Update field 1= yes 0 = no
with arcpy.da.UpdateCursor(parcelSelect, ['WITHIN_TRPA_BNDY_TRPA']) as cursor:
    for row in cursor:
        row[0] = '1'
        cursor.updateRow(row) 
del cursor        
# switch the selection
parcelSelect = arcpy.SelectLayerByAttribute_management(parcelSelect,'SWITCH_SELECTION')

# update other parcels
with arcpy.da.UpdateCursor(parcelSelect, ['WITHIN_TRPA_BNDY_TRPA']) as cursor:
    for row in cursor:
        row[0] = '0'
        cursor.updateRow(row)
del cursor
print("Within TRPA Boundary Updated: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Within TRPA Boundary Updated: " + strftime("%Y-%m-%d %H:%M:%S"))

### set within Bonus Unit Boundary -------------------------------------------------------------------------------###
print("Identifying parcels within bonus unit boundary: "  + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Identifying parcels within bonus unit boundary: " + strftime("%Y-%m-%d %H:%M:%S"))

# Select all new parcels that have their center within
parcelSelect = arcpy.SelectLayerByLocation_management(ParcelLayer, 
                                                          'HAVE_THEIR_CENTER_IN', 
                                                           sde_BonusUnitboundary, 
                                                           0, 
                                                          'NEW_SELECTION')

with arcpy.da.UpdateCursor(parcelSelect, ['WITHIN_BONUSUNIT_BNDY_TRPA']) as cursor:
    for row in cursor:
        row[0] = '1'
        cursor.updateRow(row) 
del cursor   
# switch the selection
parcelSelect = arcpy.SelectLayerByAttribute_management(parcelSelect,'SWITCH_SELECTION')

with arcpy.da.UpdateCursor(parcelSelect, ['WITHIN_BONUSUNIT_BNDY_TRPA']) as cursor:
    for row in cursor:
        row[0] = '0'
        cursor.updateRow(row)
del cursor     
print("Bonus Unit Boundary Updated: " + strftime("%Y-%m-%d %H:%M:%S"))
# log.info("Bonus Unit Boundary Updated: " + strftime("%Y-%m-%d %H:%M:%S"))
result = arcpy.GetCount_management(ParcelLayer)
print('{} has {} records now.'.format(ParcelLayer, result[0]))
### Calculate Area Field------------------------------------------------------------------------------------------###
print("Calculating Acres..." + strftime("%Y-%m-%d %H:%M:%S"))
with arcpy.da.UpdateCursor(ParcelLayer, ['PARCEL_ACRES_TRPA', 'SHAPE@']) as cursor:
    for row in cursor:
        row[0] = row[1].getArea('PLANAR', 'ACRES')
        cursor.updateRow(row)
del cursor

# calculate square feet
print("Calculating Square Feet..." + strftime("%Y-%m-%d %H:%M:%S"))
with arcpy.da.UpdateCursor(ParcelLayer, ['PARCEL_SQFT_TRPA', 'SHAPE@']) as cursor:
    for row in cursor:
        row[0] = row[1].getArea('PLANAR', 'SquareFeetUS')
        cursor.updateRow(row)
del cursor

### Copy to Feature Class ----------------------------------------------------------------------------------------###
print("Copying memory features to staging: " + strftime("%Y-%m-%d %H:%M:%S"))
# copy in-memory features to staging feature class
arcpy.CopyFeatures_management(ParcelLayer, ParcelNew)
print("Copied in-memory features, parcel staging new is set: " + strftime("%Y-%m-%d %H:%M:%S"))

arcpy.Delete_management("memory")
print("Deleted Memory Workspace: " + strftime("%Y-%m-%d %H:%M:%S"))

Starting TRPA Attribution: 2023-05-18 21:37:23
Starting the County attribute update: 2023-05-18 21:37:35
County Attribute Updated
Copied features to points: 2023-05-18 21:37:53
Starting the Ownership Type attribute update: 2023-05-18 21:37:53
The 'OWNERSHIP_TYPE' field in the parcel data has been updated
The 'EXISTING_LANDUSE' field in the parcel data has been updated
memory/ParcelLayer has 64667 records now.
Starting the Regional Land Use Spatial Join: 2023-05-18 21:38:01
Finished the Regional Land Use Spatial Join: 2023-05-18 21:38:26
Started data transfer: 2023-05-18 21:38:26
Updating row 1000
Updating row 2000
Updating row 3000
Updating row 4000
Updating row 5000
Updating row 6000
Updating row 7000
Updating row 8000
Updating row 9000
Updating row 10000
Updating row 11000
Updating row 12000
Updating row 13000
Updating row 14000
Updating row 15000
Updating row 16000
Updating row 17000
Updating row 18000
Updating row 19000
Updating row 20000
Updating row 21000
Updating row 22000
Updat

Updating row 1000
Updating row 2000
Updating row 3000
Updating row 4000
Updating row 5000
Updating row 6000
Updating row 7000
Updating row 8000
Updating row 9000
Updating row 10000
Updating row 11000
Updating row 12000
Updating row 13000
Updating row 14000
Updating row 15000
Updating row 16000
Updating row 17000
Updating row 18000
Updating row 19000
Updating row 20000
Updating row 21000
Updating row 22000
Updating row 23000
Updating row 24000
Updating row 25000
Updating row 26000
Updating row 27000
Updating row 28000
Updating row 29000
Updating row 30000
Updating row 31000
Updating row 32000
Updating row 33000
Updating row 34000
Updating row 35000
Updating row 36000
Updating row 37000
Updating row 38000
Updating row 39000
Updating row 40000
Updating row 41000
Updating row 42000
Updating row 43000
Updating row 44000
Updating row 45000
Updating row 46000
Updating row 47000
Updating row 48000
Updating row 49000
Updating row 50000
Updating row 51000
Updating row 52000
Updating row 53000
Up

Updating row 41000
Updating row 42000
Updating row 43000
Updating row 44000
Updating row 45000
Updating row 46000
Updating row 47000
Updating row 48000
Updating row 49000
Updating row 50000
Updating row 51000
Updating row 52000
Updating row 53000
Updating row 54000
Updating row 55000
Updating row 56000
Updating row 57000
Updating row 58000
Updating row 59000
Updating row 60000
Updating row 61000
Updating row 62000
Updating row 63000
Updating row 64000
Finished data transfer: 2023-05-18 21:48:12
Function fieldJoinCalc_multikey took 5.547964334487915 seconds to execute.
The 'PLAN_ID' field in the parcel data has been updated
Started data transfer: 2023-05-18 21:48:12
Updating row 1000
Updating row 2000
Updating row 3000
Updating row 4000
Updating row 5000
Updating row 6000
Updating row 7000
Updating row 8000
Updating row 9000
Updating row 10000
Updating row 11000
Updating row 12000
Updating row 13000
Updating row 14000
Updating row 15000
Updating row 16000
Updating row 17000
Updating row

Updating row 1000
Updating row 2000
Updating row 3000
Updating row 4000
Updating row 5000
Updating row 6000
Updating row 7000
Updating row 8000
Updating row 9000
Updating row 10000
Updating row 11000
Updating row 12000
Updating row 13000
Updating row 14000
Updating row 15000
Updating row 16000
Updating row 17000
Updating row 18000
Updating row 19000
Updating row 20000
Updating row 21000
Updating row 22000
Updating row 23000
Updating row 24000
Updating row 25000
Updating row 26000
Updating row 27000
Updating row 28000
Updating row 29000
Updating row 30000
Updating row 31000
Updating row 32000
Updating row 33000
Updating row 34000
Updating row 35000
Updating row 36000
Updating row 37000
Updating row 38000
Updating row 39000
Updating row 40000
Updating row 41000
Updating row 42000
Updating row 43000
Updating row 44000
Updating row 45000
Updating row 46000
Updating row 47000
Updating row 48000
Updating row 49000
Updating row 50000
Updating row 51000
Updating row 52000
Updating row 53000
Up

Updating row 35000
Updating row 36000
Updating row 37000
Updating row 38000
Updating row 39000
Updating row 40000
Updating row 41000
Updating row 42000
Updating row 43000
Updating row 44000
Updating row 45000
Updating row 46000
Updating row 47000
Updating row 48000
Updating row 49000
Updating row 50000
Updating row 51000
Updating row 52000
Updating row 53000
Updating row 54000
Updating row 55000
Updating row 56000
Updating row 57000
Updating row 58000
Updating row 59000
Updating row 60000
Updating row 61000
Updating row 62000
Updating row 63000
Updating row 64000
Finished data transfer: 2023-05-18 21:51:40
Function fieldJoinCalc_multikey took 3.2194976806640625 seconds to execute.
The 'PSTL_ZIP5' field in the parcel data has been updated
Starting to select parcels within City of South Lake Tahoe: 2023-05-18 21:51:40
Finished updating parcels within City of South Lake Tahoe: 2023-05-18 21:51:43
JURISDCITION field update with 'CSLT' values 
Starting the Zoning Spatial Join: 2023-05-18 21

In [33]:
# replace null with ''
replace_null_values_with_blank(ParcelNew)
result = arcpy.GetCount_management(ParcelNew)
print('{} has {} records.'.format(ParcelNew, result[0]))

Function replace_null_values_with_blank took 16.029126167297363 seconds to execute.
Parcel_Staging_Attributed has 64667 records.


In [34]:
result = arcpy.GetCount_management("Parcel_Staging")
print('{} has {} records'.format("Parcel_Staging", result[0]))
result = arcpy.GetCount_management("Parcel_Staging_Attributed")
print('{} has {} records'.format("Parcel_Staging_Attributed", result[0]))

Parcel_Staging has 64667 records
Parcel_Staging_Attributed has 64667 records


In [35]:
# Describe the feature class and get its spatial reference   
desc = arcpy.Describe("Parcel_Staging_Attributed")
spatialRef = desc.spatialReference
 
# Print the spatial reference name
print (spatialRef.Name)

NAD_1983_UTM_Zone_10N


### Create Parcel County Staging Feature Class

In [36]:
staging_fc     = "Parcel_Staging_Attributed"
new_fc         = "Parcel_County_Staging" 

# Create FieldMappings object to manage merge output fields
fieldMappings = arcpy.FieldMappings()
# # Add all fields from all parcel staging layers
# fieldMappings.addTable(fc)

for field in arcpy.ListFields(staging_fc):
    if not field.name == "OBJECTID" and not field.name == "Shape":
        old_name = field.name

        #Rename if necessary
        if old_name.endswith("_TRPA"):
            new_name = old_name[:-5]
        else:
            new_name = old_name

        #Create new FieldMap object    
        new_f = arcpy.FieldMap()
        new_f.addInputField(staging_fc, old_name) # Specify the input field to use

        #Rename output field
        new_f_name = new_f.outputField
        new_f_name.name = new_name
        new_f_name.aliasName = new_name
        new_f.outputField = new_f_name

        #Add field to FieldMappings object
        fieldMappings.addFieldMap(new_f)

#Convert fc using new field names
arcpy.FeatureClassToFeatureClass_conversion(staging_fc, 
                                            os.path.dirname(new_fc), 
                                            os.path.basename(new_fc), 
                                            field_mapping=fieldMappings)

arcpy.DeleteField_management("Parcel_County_Staging", 
                             ["OBJECTID_1"])

<Result '//Trpa-fs01/GIS/PARCELUPDATE/Workspace/ParcelStaging.gdb\\Parcel_County_Staging'>

## QA QC

In [12]:
fc = "Parcel_County_Staging"

apn = 'APN'


# Create an expression with proper delimiters
expression = u"{} = '034-402-001'".format(arcpy.AddFieldDelimiters(fc, apn))

# Create a search cursor using an SQL expression
with arcpy.da.SearchCursor(fc, ['APO_ADDRESS'],
                           where_clause=expression) as cursor:
    for row in cursor:
        # Print the name of the residential road
        print(row)

('2950 US HWY 50',)


In [13]:
fc = "Parcel_County_Staging"

apn = 'APN'


# Create an expression with proper delimiters
expression = u"{} = '094-520-001'".format(arcpy.AddFieldDelimiters(fc, apn))

# Create a search cursor using an SQL expression
with arcpy.da.SearchCursor(fc, ['APO_ADDRESS'],
                           where_clause=expression) as cursor:
    for row in cursor:
        # Print the name of the residential road
        print(row)

('NO ADDRESS ON FILE',)


## Load

In [5]:
## TRPA_ADMIN credentials 
portal_user = "TRPA_PORTAL_ADMIN"
portal_pwd = "@dmin6224"
portal_url = "https://maps.trpa.org/portal/"
# sign in
arcpy.SignInToPortal(portal_url, portal_user, portal_pwd)
#Generate old new apn lists
#Currently no sde.county_parcel_staging
parcelMaster   = sdeBase + "\\sde.SDE.Parcels\\sde.SDE.Parcel_Master"
parcelBase     = sdeBase + "\\sde.SDE.Parcels\\sde.SDE.Parcels_Base"
parcelNew      = "Parcel_County_Staging"

#This needs to be shifted over to SQL Table
df_special_parcels= pd.read_excel("//Trpa-fs01/GIS/PARCELUPDATE/Workspace/special_parcels.xlsx")

# parcelSpecial = sdeBase+"\\sde.SDE." 
# parcelUpdated = "ParcelUpdated"

### Version Management

#### Create New Version

In [2]:
## CREATE NEW VERSION - maybe do for each parcel update
# parent version
workspace_parent = r"https://maps.trpa.org/server/rest/services/Parcel_Edits/FeatureServer"
parent_version = "SDE.DEFAULT"
# Define the name of the new branch version and the access level

version_name = "Parcel_Update_" + strftime("%Y-%m-%d")
version_name_full = portal_user + "." + version_name
access = "PUBLIC"

#Create the new branch version
arcpy.CreateVersion_management(workspace_parent, parent_version, version_name, access)

<Result 'https://maps.trpa.org/server/rest/services/Parcel_Edits/FeatureServer'>

#### Change Version

In [3]:


# parcel master branch versioned feature service
parcelMasterVersion = r"https://maps.trpa.org/server/rest/services/Parcel_Edits/FeatureServer/0"
# feature layer name
featureLayer= 'parcelMasterVersion'
# make feature layer
arcpy.management.MakeFeatureLayer(parcelMasterVersion, featureLayer)
# change to version to edit
arcpy.management.ChangeVersion(featureLayer, "BRANCH", version_name_full)

parcelBaseVersion = r"https://maps.trpa.org/server/rest/services/Parcel_Edits/FeatureServer/1"
featureLayer_Base = 'parcelBaseVersion'
arcpy.management.MakeFeatureLayer(parcelBaseVersion, featureLayer_Base)
# change to version to edit
arcpy.management.ChangeVersion(featureLayer_Base, "BRANCH", version_name_full)


<Result 'parcelBaseVersion'>

### Obsolete and New Parcels

In [7]:
# prefixxes to remove...these we keep or ignore
prefix_remove = ('880','881','910','920')
# parcel master old/new
parcel_master_new_apn = old_new_parcels_list(featureLayer, parcelNew, 'Yes', 
                                             prefix_remove,'New APN')
parcel_master_old_apn = old_new_parcels_list(featureLayer, parcelNew, 'No', 
                                             prefix_remove,'Old APN')
# parcel base old/new
parcel_base_new_apn   = old_new_parcels_list(featureLayer_Base, parcelNew, 'Yes', 
                                           prefix_remove,'New APN')
parcel_base_old_apn   = old_new_parcels_list(featureLayer_Base, parcelNew, 'No', 
                                           prefix_remove,'Old APN')

C:\Users\amcclary\AppData\Local\Temp\2\ipykernel_5088\4040263664.py:299: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_merge['DiscoveryDate'] = date
C:\Users\amcclary\AppData\Local\Temp\2\ipykernel_5088\4040263664.py:301: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_merge['TRPA_Boundary'] = df_merge[TRPA_BNDY_Fields].sum(axis=1)


Function make_old_new_dataframe took 59.71883726119995 seconds to execute.
Function old_new_parcels_list took 59.71883726119995 seconds to execute.


C:\Users\amcclary\AppData\Local\Temp\2\ipykernel_5088\4040263664.py:299: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_merge['DiscoveryDate'] = date
C:\Users\amcclary\AppData\Local\Temp\2\ipykernel_5088\4040263664.py:301: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_merge['TRPA_Boundary'] = df_merge[TRPA_BNDY_Fields].sum(axis=1)


Function make_old_new_dataframe took 54.6669864654541 seconds to execute.
Function old_new_parcels_list took 54.6669864654541 seconds to execute.


C:\Users\amcclary\AppData\Local\Temp\2\ipykernel_5088\4040263664.py:299: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_merge['DiscoveryDate'] = date
C:\Users\amcclary\AppData\Local\Temp\2\ipykernel_5088\4040263664.py:301: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_merge['TRPA_Boundary'] = df_merge[TRPA_BNDY_Fields].sum(axis=1)


Function make_old_new_dataframe took 21.850532054901123 seconds to execute.
Function old_new_parcels_list took 21.850532054901123 seconds to execute.
Function make_old_new_dataframe took 21.77422261238098 seconds to execute.
Function old_new_parcels_list took 21.77422261238098 seconds to execute.


C:\Users\amcclary\AppData\Local\Temp\2\ipykernel_5088\4040263664.py:299: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_merge['DiscoveryDate'] = date
C:\Users\amcclary\AppData\Local\Temp\2\ipykernel_5088\4040263664.py:301: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_merge['TRPA_Boundary'] = df_merge[TRPA_BNDY_Fields].sum(axis=1)


In [6]:
#Update ParcelNew_Old
#Need to generate this list as a one off from the old data and make a stable system for updates going forward
parcel_base_new_apn   = old_new_parcels_list(featureLayer_Base, parcelNew, 'Yes', 
                                           prefix_remove,'New APN')
parcel_base_old_apn   = old_new_parcels_list(featureLayer_Base, parcelNew, 'No', 
                                           prefix_remove,'Old APN')
print(parcel_base_new_apn)


NameError: name 'prefix_remove' is not defined

In [9]:
print(parcel_base_new_apn)

['090-282-018', '097-140-043', '116-080-003', '097-130-030', '092-100-030', '090-153-011', '112-010-012', '112-010-011', '117-160-010', '117-190-060', '094-440-010', '085-343-022', '112-190-048', '094-350-021', '094-470-026', '094-233-004', '097-091-009', '117-200-054', '117-180-008', '117-180-055', '090-074-021', '094-070-002', '094-300-011', '098-180-016', '097-200-014', '097-092-011', '098-180-015', '117-230-004', '097-193-004', '097-060-040', '084-010-007', '097-140-046', '097-192-010', '023-960-008', '023-960-002', '023-960-003', '023-960-004', '023-960-005', '023-960-006', '023-960-007', '090-282-019', '1418-34-401-030', '1418-34-111-040', '1418-34-111-041', '1418-34-111-042', '1318-09-810-118', '1318-09-810-119', '1318-15-110-014', '110-070-021', '1319-00-001-001', '1419-00-001-002', '1319-30-516-050']


### Update_Parcel_Master

In [74]:
# set up mapping for conversion from feature class to data frame
data_type_mapping = {
    "String": str,
    "Integer": int,
    "SmallInteger": int,
    "Single": float,
    "Double": float,
    "Date": pd.to_datetime
}

fields_to_exclude = ['SHAPE','OBJECTID', 'Shape']
dfparcelNew = generate_spatial_dataframe(parcelNew, data_type_mapping, fields_to_exclude)
dfparcelMaster = generate_spatial_dataframe(featureLayer, data_type_mapping, fields_to_exclude)

# filter to new parcels within TRPA boundary
dfparcelNew=dfparcelNew.loc[dfparcelNew['WITHIN_TRPA_BNDY']==1]
df_special_parcels = pd.read_excel("//Trpa-fs01/GIS/PARCELUPDATE/Workspace/special_parcels.xlsx")
matching_apns_parcel_master = return_matching_apns(featureLayer, parcelNew, df_special_parcels)

# filter parcel master to matching APNs
dfparcelMaster = dfparcelMaster[dfparcelMaster['APN'].isin(matching_apns_parcel_master)]
# filter staging data to matching APNs
dfparcelNew    = dfparcelNew[dfparcelNew['APN'].isin(matching_apns_parcel_master)]


fields_to_ignore = ['PARCEL_SQFT', 'PPNO', 'ESTIMATED_COVERAGE_ALLOWED', 'IMPERVIOUS_SURFACE_SQFT', 
                    'LOCATION_TO_TOWNCENTER', 'UNITS', 'PARCEL_ACRES','YEAR_BUILT', 'BEDROOMS',
                   'BUILDING_SQFT', 'BATHROOMS']

# get the differences as dictionaries
differences_master = differenceDictionary(dfparcelMaster, dfparcelNew, 'APN', fields_to_ignore)





Function generate_spatial_dataframe took 4.2861552238464355 seconds to execute.
Function generate_spatial_dataframe took 158.5700855255127 seconds to execute.
Function return_matching_apns took 60.14953112602234 seconds to execute.
Function differenceDictionary took 7.624205589294434 seconds to execute.


In [77]:
dfDifferences = pd.DataFrame.from_dict(differences_master)
dfDifferences.to_csv('dfDifferences.csv')

In [78]:
len(dfDifferences.columns)

0

In [76]:
# function to update attributes
apn_issues = update_fc_from_dict(differences_master, 'APN', featureLayer)
if len(apn_issues)==0:
    print ("No issues with executing updates")
else:
    print("There were issues with the following APNs")
    print(apn_issues)

Updating Attributes started: 2023-05-19 12:22:11
Updating Attributes Finished: 2023-05-19 12:24:08
Total updated0
Function update_fc_from_dict took 117.28276133537292 seconds to execute.
No issues with executing updates


### Log
* Error updating 028-081-014: ERROR: code:500, Unable to complete operation., GraphicFeatureServer:PerformEdit() returned no result., Failure to access the DBMS server [[Microsoft][ODBC Driver 17 for SQL Server]Communication link failure], Internal server error.
* Error updating 025-341-013: Client tried to access password-protected page without proper authorization.
* Updating Attributes Finished: 2023-05-15 22:30:28
*  Total updated24832
* There were issues with the following APNs
* ['1319-19-714-026', '1318-23-710-062', '1318-23-812-013', '1319-19-111-008', '1318-23-710-097', '1319-30-625-001', '1319-30-612-004', '1319-30-519-022', '1319-30-519-021', '1319-30-519-020', '1319-30-519-016', '1319-30-519-010', '1319-30-519-015', '1319-30-519-009', '1319-30-519-014', '1319-30-519-008', '1319-30-519-004', '1319-30-519-003', '1319-30-519-002', '1319-30-101-006', '1319-30-101-005', '1319-30-301-003', '1319-30-310-026', '1319-30-101-007', '1319-30-711-002', '1319-19-810-008', '1319-19-810-017', '1319-19-411-019', '1319-19-411-015', '1319-19-411-021', '1319-19-810-001', '1319-19-810-002', '1319-19-411-020', '1319-19-810-003', '1319-19-411-018', '1319-19-411-023', '1319-19-411-006', '1319-19-810-005', '1319-19-411-005', '1319-19-411-008', '1319-19-411-009', '1319-19-411-010', '1319-19-810-004', '1319-19-411-012', '1319-19-411-013', '1319-19-411-029', '1319-30-110-005', '1319-30-110-002', '1319-19-810-013', '1319-19-810-014', '1319-19-801-002', '1319-19-410-013', '1319-19-411-004', '1319-19-410-009', '1319-19-410-014', '1319-19-810-016', '1319-30-610-001', '1418-27-810-050', '1418-27-410-004', '1418-27-410-006', '1418-27-810-047', '1418-27-404-001', '1418-27-411-023', '1418-27-410-007', '1418-27-410-001', '1418-27-810-042', '1418-27-810-051', '1418-27-812-003', '1418-27-810-029', '1418-27-812-006', '1418-27-812-007', '1418-27-810-033', '1418-27-810-034', '1418-27-810-027', '1418-27-810-016', '1418-27-810-009', '1418-27-810-008', '1418-27-810-010', '1418-27-810-005', '1418-27-810-003', '1418-27-810-020', '1418-27-810-015', '1418-27-810-017', '1418-27-810-023', '1418-27-810-024', '1418-27-811-003', '1418-27-712-008', '1418-27-811-006', '1418-27-710-007', '1418-27-711-001', '1418-27-711-002', '1418-27-712-010', '1418-27-712-009', '1418-27-710-004', '1418-27-710-001', '1418-27-811-005', '1418-27-811-001', '1418-27-811-002', '1418-27-710-011', '1418-34-112-011', '1418-34-112-029', '1418-34-211-020', '1418-34-211-029', '1418-34-211-006', '1418-34-211-045', '1418-34-112-006', '1418-34-112-013', '1418-34-112-007', '1418-34-112-014', '1418-34-112-012', '1418-34-112-010', '1418-34-211-008', '1418-34-112-009', '1418-34-211-007', '1418-34-211-003', '1418-34-211-038', '1418-34-211-002', '1418-34-211-012', '1418-34-211-044', '1418-34-211-040', '1418-34-211-043', '1418-34-211-042', '1418-34-211-041', '1418-34-211-047', '1418-34-211-015', '1418-34-211-037', '1418-34-211-035', '1418-34-610-017', '1418-34-610-008', '1418-34-610-010', '1418-34-610-004', '1418-34-610-001', '1418-34-610-016', '1418-34-211-005', '1418-34-402-006', '1418-34-202-006', '1418-34-202-007', '1418-34-303-001', '1418-34-202-005', '1418-27-810-036', '1418-34-601-007', '1418-27-810-052', '1418-34-210-002', '1418-34-110-047', '1418-34-110-037', '1418-34-110-052', '1418-34-110-027', '1418-34-110-026', '1418-34-210-015', '1418-34-110-043', '1418-34-110-051', '1418-34-210-023', '1418-34-210-022', '1418-34-210-021', '1418-34-210-024', '1418-34-210-027', '1418-34-210-028', '1418-34-210-016', '1418-34-210-029', '1418-34-210-031', '1418-34-210-032', '1418-34-210-004', '1418-34-201-007', '1418-34-201-009', '1418-34-301-007', '1418-34-301-005', '1418-34-304-003', '1418-34-304-006', '1418-34-310-013', '1418-34-302-001', '1418-34-201-008', '1418-34-304-007', '1318-03-210-011', '1318-03-210-012', '1318-03-110-003', '1318-03-110-002', '1318-03-110-008', '1318-03-210-036', '1318-03-111-015', '1318-03-110-031', '1318-03-111-016', '1318-03-111-028', '1318-03-111-029', '1318-03-110-021', '1318-03-110-034', '1318-03-111-013', '1318-03-110-030', '1318-03-111-019', '1318-03-111-023', '1318-03-111-025', '1318-03-111-027', '1318-03-111-030', '1318-03-210-038', '1318-03-211-002', '1318-03-211-001', '1318-03-111-039', '1318-03-111-045', '1318-03-111-042', '1318-03-111-044', '1318-03-111-051', '1318-03-111-035', '1318-03-111-053', '1318-03-211-013', '1318-03-211-005', '1318-03-211-009', '1318-03-111-034', '1318-03-211-012', '1318-03-211-014', '1318-03-211-007', '1318-03-211-008', '1318-03-212-036', '1318-03-212-033', '1318-03-212-040', '1318-03-212-039', '1318-03-212-041', '1318-03-212-044', '1318-03-212-037', '1318-03-212-051', '1318-03-212-056', '1318-03-212-048', '1318-03-212-052', '1318-03-210-035', '1318-03-210-031', '1318-03-212-029', '1318-03-212-024', '1318-03-212-015', '1318-03-212-014', '1318-03-212-006', '1318-03-212-007', '1318-03-212-028', '1318-03-210-034', '1318-03-212-030', '1318-03-212-018', '1318-03-212-016', '1318-03-210-030', '1318-03-212-025', '1318-03-212-017', '1318-03-210-022', '1318-03-210-029', '1318-03-212-065', '1318-03-212-060', '1318-03-212-077', '1318-03-212-067', '1318-03-212-003', '1318-03-212-002', '1318-03-212-093', '1318-26-101-093', '1318-27-001-004', '1318-26-101-002', '1318-26-101-003', '1318-27-001-010', '1318-27-001-006', '1318-27-001-011', '1318-27-001-009', '1318-27-001-013', '1318-27-001-017', '1318-23-401-036', '1318-23-401-042', '1318-16-710-026', '1318-16-710-004', '1318-16-810-052', '1318-16-710-013', '1318-16-710-012', '1318-16-710-017', '1318-16-710-011', '1318-16-710-019', '1318-16-710-010', '1318-16-710-020', '1318-16-710-021', '1318-16-710-009', '1318-16-710-008', '1318-16-710-007', '1318-16-710-023', '1318-16-710-003', '1318-16-710-002', '1318-16-710-024', '1318-16-710-001', '1318-16-810-051', '1318-16-810-050', '1318-16-810-036', '1318-16-810-002', '1318-16-810-047', '1318-16-810-035', '1318-16-810-034', '1318-15-801-002', '1318-22-001-004', '1318-23-401-015', '1318-23-401-016', '1318-23-401-044', '1318-23-401-048', '1318-23-401-049', '1318-23-401-050', '1318-23-401-043', '1318-23-314-001', '1318-23-314-021', '1318-23-314-004', '1318-23-314-020', '1318-23-314-005', '1318-23-314-016', '1318-23-314-014', '1318-23-214-001', '1318-23-310-061', '1318-23-310-055', '1318-23-310-059', '1318-23-310-029', '1318-23-310-034', '1318-23-310-035', '1318-23-310-037', '1318-23-310-040', '1318-23-310-052', '1318-23-310-049', '1318-23-310-038', '1318-23-310-044', '1318-23-310-042', '1318-23-310-060', '1318-23-211-001', '1318-23-310-063', '1318-23-310-070', '1318-23-310-068', '1318-23-310-067', '1318-15-611-037', '1318-15-611-038', '1318-15-611-039', '1318-15-712-007', '1318-15-712-008', '1318-15-803-003', '1318-15-803-008', '1318-15-611-012', '1318-15-611-014', '1318-15-611-030', '1318-15-611-020', '1318-15-611-021', '1318-15-802-009', '1318-15-703-002', '1318-15-703-003', '1318-15-310-001', '1318-15-613-001', '1318-15-210-005', '1318-15-710-004', '1318-15-311-018', '1318-15-311-016', '1318-15-311-021', '1318-15-210-003', '1318-15-210-004', '1318-15-610-024', '1318-15-613-002', '1318-15-311-020', '1318-15-311-009', '1318-15-311-017', '1318-15-311-010', '1318-15-311-015', '1318-15-311-014', '1318-15-311-013', '1318-15-311-023', '1318-15-311-024', '1318-15-310-002', '1318-15-310-003', '1318-15-714-041', '1318-15-715-019', '1318-15-715-022', '1318-15-711-002', '1318-15-612-001', '1318-15-612-002', '1318-15-612-007', '1318-15-613-005', '1318-15-612-005', '1318-15-612-006', '1318-15-613-003', '1318-15-711-005', '1318-15-710-006', '1318-15-712-001', '1318-15-611-048', '1318-15-611-045', '1318-15-712-003', '1318-15-611-047', '1318-15-611-046', '1318-15-712-004', '1318-15-612-024', '1318-15-612-025', '1318-15-612-026', '1318-15-611-052', '1318-15-611-053', '1318-15-611-056', '1318-15-612-023', '1318-15-612-020', '1318-15-612-011', '1318-15-612-014', '1318-15-612-019', '1318-15-612-018', '1318-15-711-006', '1318-15-711-023', '1318-15-711-013', '1318-15-711-021', '1318-15-711-027', '1318-15-711-014', '1318-15-711-030', '1318-15-711-019', '1318-15-711-017', '1318-15-611-058', '1318-15-611-076', '1318-15-611-001', '1318-15-611-073', '1318-15-611-004', '1318-15-611-063', '1318-15-611-008', '1318-15-611-064', '1318-15-610-045', '1318-15-511-003', '1318-15-511-004', '1318-15-611-075', '1318-15-611-074', '1318-15-611-009', '1318-15-611-071', '1318-15-611-011', '1318-15-611-010', '1318-15-611-007', '1318-15-611-060', '1318-15-611-065', '1318-15-611-059', '1318-15-611-069', '1318-15-511-001', '1318-15-610-010', '1318-15-610-011', '1318-15-610-047', '1318-15-610-029', '1318-15-610-028', '1318-15-610-022', '1318-15-610-035', '1318-15-311-004', '1318-15-511-002', '1318-15-610-008', '1318-15-610-015', '1318-15-610-009', '1318-15-610-013', '1318-15-610-039', '1318-15-610-002', '1318-15-610-012', '1318-15-610-021', '1318-15-610-031', '1318-15-210-001', '1318-15-311-003', '1318-15-311-002', '1318-15-610-025', '1318-23-315-045', '1318-27-001-015', '1318-15-802-005', '1318-15-703-001', '1318-15-802-010', '1318-10-310-058', '1318-10-310-059', '1318-10-310-076', '1318-10-310-014', '1318-10-310-086', '1318-09-701-003', '1318-09-701-002', '1318-09-701-001', '1318-09-810-117', '1318-09-810-028', '1318-22-001-003', '1318-15-101-009', '1318-09-811-014', '1318-09-811-018', '1318-09-811-017', '1318-10-417-052', '1318-09-811-019', '1318-10-417-048', '1318-10-417-053', '1318-10-417-051', '1318-10-417-040', '1318-10-417-035', '1318-10-417-034', '1318-10-417-036', '1318-10-417-037', '1318-10-417-045', '1318-10-312-020', '1318-10-301-001', '1318-10-312-015', '1318-10-312-016', '1318-10-312-011', '1318-10-312-013', '1318-10-312-006', '1318-10-313-005', '1318-10-313-009', '1318-10-312-024', '1318-10-416-013', '1318-27-002-009', '1318-00-002-006', '1318-10-416-003', '1318-10-416-019', '1318-10-416-014', '1318-10-416-020', '1318-10-416-024', '1318-10-416-015', '1318-10-416-029', '1318-10-416-030', '1318-10-415-020', '1318-10-415-025', '1318-10-312-034', '1318-10-312-036', '1318-10-411-014', '1318-10-316-021', '1318-10-315-006', '1318-10-411-004', '1318-10-412-014', '1318-10-412-004', '1318-10-412-015', '1318-10-411-012', '1318-10-410-008', '1318-10-316-022', '1318-10-315-005', '1318-10-411-005', '1318-10-412-002', '1318-10-411-013', '1318-10-412-007', '1318-10-412-012', '1318-10-412-016', '1318-10-411-008', '1318-10-411-010', '1318-10-417-020', '1318-10-417-006', '1318-09-811-002', '1318-15-711-007', '1318-15-711-004', '1318-09-811-001', '1318-10-417-014', '1318-10-417-005', '1318-10-417-009', '1318-10-417-031', '1318-10-417-029', '1318-10-417-017', '1318-10-417-015', '1318-10-316-012', '1318-09-811-008', '1318-10-417-032', '1318-10-417-007', '1318-10-417-011', '1318-10-417-023', '1318-10-416-054', '1318-10-414-001', '1318-10-416-056', '1318-10-314-020', '1318-10-415-007', '1318-10-416-057', '1318-10-413-017', '1318-10-416-049', '1318-10-415-063', '1318-10-312-030', '1318-10-311-007', '1318-10-301-008', '1318-10-314-003', '1318-10-314-007', '1318-10-301-009', '1318-10-413-005', '1318-10-416-047', '1318-10-416-045', '1318-10-416-042', '1318-10-416-053', '1318-10-416-040', '1318-10-313-020', '1318-10-314-013', '1318-10-416-033', '1318-10-416-031', '1318-10-416-038', '1318-10-312-031', '1318-10-312-032', '1318-10-312-027', '1318-10-313-025', '1318-10-314-012', '1318-10-313-010', '1318-10-311-006', '1318-10-301-005', '1318-10-313-011', '1318-10-314-011', '1318-10-413-006', '1318-10-413-001', '1318-10-413-007', '1318-10-314-018', '1318-10-413-013', '1318-10-413-012', '1318-10-416-048', '1318-10-416-062', '1318-10-416-035', '1318-10-416-036', '1318-10-415-072', '1318-10-416-034', '1318-10-416-063', '1318-15-102-003', '1318-03-212-078', '1418-34-210-011', '1318-23-411-013', '1318-23-410-068', '1318-23-410-033', '1318-23-410-069', '1418-34-210-010', '1318-23-410-070', '1318-23-410-038', '1318-23-410-040', '1318-23-410-063', '1318-23-401-026', '1318-23-410-016', '1318-23-410-026', '1318-23-410-015', '1318-23-410-014', '1318-23-410-013', '1318-23-411-012', '1318-23-410-012', '1318-23-410-028', '1318-23-410-029', '1318-23-410-011', '1318-23-410-031', '1318-23-410-010', '1318-23-410-032', '1318-23-410-009', '1318-23-410-008', '1318-23-410-036', '1318-23-410-066', '1318-23-410-065', '1318-23-410-005', '1318-23-410-004', '1318-23-410-071', '1318-23-410-003', '1318-23-410-002', '1318-23-410-041', '1318-23-410-042', '1318-23-410-059', '1318-23-410-058', '1318-23-410-001', '1318-23-810-018', '1318-23-811-012', '1318-23-810-084', '1318-23-810-102', '1318-23-811-022', '1318-23-811-023', '1318-23-811-006', '1318-23-810-073', '1318-23-810-109', '1318-23-411-028', '1318-23-411-029', '1318-23-810-075', '1318-23-811-028', '1318-23-810-042', '1318-26-510-005', '1318-26-511-011', '1318-26-510-009', '1318-26-511-004', '1318-26-511-010', '1318-23-810-020', '1318-23-811-014', '1318-23-810-024', '1318-23-810-025', '1318-23-810-056', '1318-23-810-096', '1318-23-810-063', '1318-23-810-086', '1318-23-811-031', '1318-23-810-107', '1318-23-811-027', '1318-26-510-003', '1318-26-511-009', '1318-26-510-001', '1318-26-510-010', '1318-26-510-007', '1318-23-812-009', '1318-23-812-008', '1318-23-812-006', '1318-23-812-002', '1318-26-511-012', '1318-23-811-052', '1318-26-512-003', '1318-26-510-014', '1318-26-510-013', '1318-26-512-004', '1318-26-512-005', '1318-26-513-002', '1318-23-812-020', '1318-23-813-004', '1318-23-813-003', '1318-23-812-025', '1318-23-813-001', '1318-23-813-024', '1318-23-814-003', '1318-23-814-002', '1318-23-814-001', '1318-23-813-022', '1318-24-410-013', '1318-24-401-007', '1318-24-401-003', '1318-24-401-004', '1318-23-814-006', '1318-24-404-013', '1318-24-404-023', '1318-24-404-001', '1318-25-101-001', '1318-25-111-019', '1318-25-111-018', '1318-25-110-001', '1318-24-403-001', '1318-24-403-005', '1318-25-101-005', '1318-25-111-003', '1318-25-110-007', '1318-25-111-025', '1318-25-111-010', '1318-25-111-011', '1318-25-110-004', '1318-25-111-021', '1318-25-111-020', '1318-25-111-016', '1318-26-501-005', '1318-26-515-033', '1318-26-515-038', '1318-26-515-037', '1318-26-515-034', '1318-26-515-026', '1318-26-515-009', '1318-26-514-019', '1318-26-515-015', '1318-26-515-002', '1318-26-514-021', '1318-26-515-008', '1318-26-514-017', '1318-26-515-010', '1318-26-514-018', '1318-26-515-017', '1318-26-515-020', '1318-26-515-005', '1318-26-515-004', '1318-26-515-019', '1318-26-515-013', '1318-26-515-001', '1318-26-515-007', '1318-26-514-010', '1318-26-514-014', '1318-26-514-011', '1318-26-514-016', '1318-26-101-038', '1318-26-101-033', '1318-26-101-066', '1318-26-101-072', '1318-26-101-037', '1318-26-101-068', '1318-26-101-070', '1318-26-514-002', '1318-25-111-004', '1318-24-403-004', '1318-23-410-007', '1318-23-410-006', '1318-23-810-095', '1318-23-811-013', '1318-23-202-001', '1318-27-001-016', '1318-23-411-014', '1318-15-610-020', '1318-23-810-019', '1318-23-411-015', '1318-23-811-015', '1318-10-314-016', '1318-15-810-001', '1318-22-001-006', '1318-23-212-063', '1318-23-217-010', '1318-23-217-015', '1318-23-311-020', '1318-23-212-074', '1318-23-312-001', '1318-23-212-076', '1318-23-212-078', '1318-23-312-003', '1318-23-313-003', '1318-23-313-000', '1318-23-315-005', '1318-23-315-006', '1318-23-315-008', '1318-23-315-011', '1318-23-315-012', '1318-23-315-015', '1318-23-315-014', '1318-23-315-016', '1318-23-315-017', '1318-23-315-023', '1318-23-315-026', '1318-23-315-027', '1318-23-315-028', '1318-23-315-037', '1318-23-315-038', '1318-23-315-039', '1318-23-315-040', '1318-23-315-041', '1318-23-315-043', '1318-22-312-022', '1318-22-710-008', '1318-10-310-079', '1318-22-311-001', '1318-22-311-002', '1318-22-311-003', '1318-22-311-006', '1318-22-311-007', '1318-22-311-009', '1318-22-311-013', '1318-22-311-014', '1318-22-311-015', '1318-22-311-017', '1318-22-311-016', '1318-22-311-018', '1318-22-311-019', '1318-22-311-020', '1318-22-311-025', '1318-22-311-024', '1318-22-312-002', '1318-22-312-001', '1318-22-312-021', '1318-22-312-003', '1318-22-312-005', '1318-22-312-006', '1318-22-312-007', '1318-22-312-008', '1318-22-312-009', '1318-22-312-010', '1318-22-312-012', '1318-22-312-013', '1318-22-312-014', '1318-22-312-015', '1318-10-310-087', '1318-10-310-092', '1318-10-310-093', '1318-27-001-022', '1418-27-403-001', '1418-27-410-003', '1418-27-810-046', '1418-27-810-030', '1418-27-810-040', '1418-27-810-019', '1418-27-810-004', '1418-27-810-022', '1418-27-810-025', '1418-27-810-026', '1418-27-810-001', '1418-27-710-003', '1418-34-112-002', '1418-34-112-001', '1418-34-211-004', '1418-34-211-011', '1418-34-211-017', '1418-34-211-030', '1418-34-101-002', '1418-34-112-028', '1418-34-112-008', '1418-34-211-010', '1418-34-211-001', '1418-34-211-034', '1418-34-610-003', '1418-34-610-002', '1418-34-610-014', '1418-34-610-015', '1418-34-610-012', '1418-34-610-011', '1418-34-303-008', '1418-34-202-001', '1418-27-810-039', '1418-34-110-044', '1418-34-110-025', '1418-27-402-002', '1418-34-110-048', '1418-34-210-006', '1418-34-210-003', '1418-34-301-006', '1418-34-304-004', '1418-34-304-005', '1418-34-201-002', '1418-34-201-003', '1418-34-301-001', '1418-34-301-004', '1318-03-110-001', '1318-03-110-010', '1318-03-110-019', '1318-03-111-012', '1318-03-111-017', '1318-03-110-032', '1318-03-111-018', '1318-03-111-024', '1318-03-210-037', '1318-03-111-043', '1318-03-111-046', '1318-03-111-038', '1318-03-111-052', '1318-03-111-031', '1318-03-111-032', '1318-03-212-045', '1318-03-212-050', '1318-03-212-031', '1318-03-212-066', '1318-03-212-011', '1318-03-212-032', '1318-03-210-032', '1318-03-212-020', '1318-03-212-057', '1318-03-212-061', '1318-22-002-107', '1318-26-101-005', '1318-27-001-005', '1318-27-001-014', '1318-16-810-003', '1318-16-710-018', '1318-16-710-025', '1318-16-710-022', '1318-16-710-005', '1318-16-710-006', '1318-16-810-001', '1318-16-810-053', '1318-22-001-007', '1318-22-001-005', '1318-23-401-014', '1318-23-401-007', '1318-23-401-008', '1318-23-314-012', '1318-23-314-017', '1318-22-001-012', '1318-23-310-033', '1318-23-310-043', '1318-23-310-031', '1318-23-210-036', '1318-23-211-022', '1318-23-310-066', '1318-23-310-062', '1318-23-310-064', '1318-15-611-025', '1318-15-611-015', '1318-15-611-024', '1318-15-611-031', '1318-15-802-008', '1318-15-702-001', '1318-15-702-002', '1318-15-802-001', '1318-15-311-012', '1318-15-311-011', '1318-15-710-005', '1318-15-311-026', '1318-15-311-025', '1318-15-310-004', '1318-15-710-001', '1318-15-613-004', '1318-15-711-001', '1318-15-612-009', '1318-15-612-004', '1318-15-710-007', '1318-15-611-049', '1318-15-611-044', '1318-15-712-002', '1318-15-611-055', '1318-15-711-018', '1318-15-612-015', '1318-15-711-008', '1318-15-711-026', '1318-15-711-020', '1318-15-711-031', '1318-15-711-015', '1318-15-511-007', '1318-15-501-001', '1318-15-610-041', '1318-15-611-002', '1318-15-610-044', '1318-15-611-070', '1318-15-611-061', '1318-15-610-046', '1318-15-611-068', '1318-15-610-048', '1318-15-610-034', '1318-15-611-078', '1318-15-610-006', '1318-15-610-014', '1318-15-610-040', '1318-15-610-003', '1318-15-601-005', '1318-15-610-037', '1318-15-610-019', '1318-15-610-027', '1318-15-210-006', '1318-27-001-021', '1318-10-310-075', '1318-09-701-004', '1318-16-710-015', '1318-15-802-007', '1318-15-802-006', '1318-15-201-003', '1318-10-000-007', '1318-10-417-046', '1318-09-811-020', '1318-09-811-015', '1318-10-312-003', '1318-10-312-009', '1318-10-312-007', '1318-10-312-023', '1318-10-312-022', '1318-27-002-005', '1318-10-416-026', '1318-10-416-027', '1318-10-416-012', '1318-10-416-011', '1318-10-411-009', '1318-10-410-003', '1318-10-411-002', '1318-10-411-006', '1318-10-412-005', '1318-10-410-007', '1318-10-417-004', '1318-10-316-011', '1318-10-417-001', '1318-15-711-003', '1318-15-711-011', '1318-10-417-030', '1318-10-417-016', '1318-10-316-014', '1318-09-811-003', '1318-10-417-019', '1318-10-415-010', '1318-10-413-014', '1318-10-416-061', '1318-10-416-060', '1318-15-102-002', '1318-10-413-015', '1318-10-416-050', '1318-10-416-051', '1318-10-312-033', '1318-10-312-025', '1318-10-313-016', '1318-10-314-014', '1318-10-314-015', '1318-10-413-011', '1318-10-416-044', '1418-34-201-010', '1318-23-410-067', '1318-23-410-072', '1318-23-410-017', '1318-23-410-020', '1318-23-410-030', '1318-23-410-034', '1318-23-410-035', '1318-23-410-037', '1318-23-410-064', '1318-23-410-039', '1318-23-810-099', '1318-23-810-085', '1318-23-810-101', '1318-23-811-024', '1318-23-810-080', '1318-23-810-074', '1318-23-810-110', '1318-26-510-011', '1318-26-511-007', '1318-23-810-100', '1318-23-810-108', '1318-26-510-004', '1318-26-510-006', '1318-26-510-008', '1318-23-812-007', '1318-23-812-003', '1318-26-512-002', '1318-26-512-001', '1318-26-513-001', '1318-23-812-024', '1318-23-812-017', '1318-23-813-023', '1318-24-401-008', '1318-25-111-006', '1318-25-110-008', '1318-25-111-022', '1318-25-110-006', '1318-25-110-005', '1318-25-111-024', '1318-26-514-023', '1318-26-515-032', '1318-26-515-027', '1318-26-515-035', '1318-26-514-022', '1318-26-515-016', '1318-26-514-020', '1318-26-515-003', '1318-26-515-018', '1318-26-515-014', '1318-26-514-004', '1318-26-514-015', '1318-26-101-069', '1318-26-101-067', '1318-25-111-017', '1318-24-404-002', '1318-24-411-016', '1318-24-404-022', '1318-24-403-002', '1318-10-313-001', '1318-23-212-017', '1318-23-213-037', '1318-23-315-007', '1318-23-315-019', '1318-23-315-021', '1318-23-315-030', '1318-23-315-029', '1318-23-315-031', '1318-23-315-032', '1318-23-315-034', '1318-23-315-036', '1318-22-311-004', '1318-22-311-005', '1318-22-311-008', '1318-22-311-012', '1318-22-311-021', '1318-22-311-023', '1318-22-312-000', '1318-22-312-004', '1318-22-312-011', '1318-22-312-018', '1318-22-312-019', '1318-22-312-017', '1318-22-312-016', '1318-22-311-011', '1318-22-311-010', '1418-34-601-008', '1418-27-210-036', '1418-27-210-002', '1418-27-210-032', '1418-27-210-037', '1418-27-210-001', '1418-27-601-007', '1418-27-712-001', '1418-27-712-003', '1418-27-712-015', '1418-27-712-002', '1418-27-710-009', '1418-27-710-010', '1418-34-610-009', '1418-34-601-003', '1418-34-601-004', '1418-34-601-010', '1418-34-601-006', '1318-03-110-004', '1318-03-110-035', '1318-03-110-006', '1318-03-110-007', '1418-27-210-013', '1418-00-002-003', '1418-00-002-004', '1418-22-510-001', '1418-22-501-009', '1418-22-510-003', '1418-22-510-002', '1418-22-511-013', '1418-22-511-014', '1418-22-610-006', '1418-22-511-001', '1418-22-610-011', '1418-22-610-009', '1418-22-611-028', '1418-22-511-003', '1418-22-511-002', '1418-22-511-006', '1418-22-511-005', '1418-22-511-007', '1418-22-610-001', '1418-22-610-007', '1418-22-610-008', '1418-22-610-005', '1418-22-501-004', '1418-22-501-005', '1418-22-511-010', '1418-22-511-011', '1418-22-511-008', '1418-03-711-012', '1418-03-811-012', '1418-03-811-001', '1418-03-711-011', '1418-03-711-002', '1418-03-711-003', '1418-03-711-008', '1418-03-711-007', '1418-03-711-004', '1418-03-711-009', '1418-03-711-001', '1418-03-401-003', '1418-03-711-010', '1418-03-811-002', '1418-03-811-031', '1418-03-811-032', '1418-03-811-028', '1418-03-811-008', '1418-03-811-013', '1418-03-811-007', '1418-03-811-010', '1418-03-811-006', '1418-03-811-005', '1418-03-811-014', '1418-03-811-015', '1418-03-811-016', '1418-03-811-026', '1418-03-811-025', '1418-03-811-024', '1418-03-811-022', '1418-03-811-023', '1418-11-402-001', '1418-03-812-001', '1418-03-812-002', '1418-02-410-002', '1418-11-110-001', '1418-10-511-001', '1418-10-511-020', '1418-10-510-003', '1418-10-511-006', '1418-10-511-010', '1418-02-410-003', '1418-11-110-002', '1418-11-110-003', '1418-10-511-002', '1418-11-110-004', '1418-10-511-003', '1418-10-511-022', '1418-10-511-004', '1418-10-511-005', '1418-11-110-005', '1418-11-110-014', '1418-10-511-007', '1418-10-511-019', '1418-11-110-012', '1418-11-110-007', '1418-10-511-011', '1418-10-511-015', '1418-10-511-016', '1418-10-610-003', '1418-15-702-004', '1418-15-702-005', '1418-15-702-006', '1418-03-811-018', '1418-03-301-009', '1418-03-301-008', '1418-03-301-010', '1418-03-301-011', '1418-10-710-070', '1418-10-810-010', '1418-10-710-012', '1418-10-710-011', '1418-10-710-013', '1418-10-710-035', '1418-11-302-001', '1418-10-710-048', '1418-10-710-010', '1418-10-702-008', '1418-10-702-010', '1418-11-412-028', '1418-10-802-010', '1418-10-802-004', '1418-10-810-019', '1418-10-810-011', '1418-15-510-005', '1418-11-410-002', '1418-15-510-014', '1418-15-510-002', '1418-10-710-005', '1418-10-710-014', '1418-10-710-015', '1418-10-710-016', '1418-10-710-069', '1418-10-710-017', '1418-10-710-018', '1418-10-710-019', '1418-11-311-002', '1418-10-710-068', '1418-10-710-067', '1418-10-710-020', '1418-10-710-066', '1418-10-710-063', '1418-10-710-064', '1418-11-311-001', '1418-11-311-003', '1418-11-311-015', '1418-11-311-006', '1418-11-311-007', '1418-11-311-008', '1418-11-311-009', '1418-11-311-010', '1418-11-311-014', '1418-11-412-029', '1418-11-410-003', '1418-10-810-007', '1418-10-810-018', '1418-11-410-001', '1418-10-810-020', '1418-10-810-012', '1418-10-810-017', '1418-10-810-013', '1418-10-810-016', '1418-11-410-007', '1418-10-810-015', '1418-10-810-014', '1418-11-410-006', '1418-10-810-021', '1418-10-810-022', '1418-11-410-005', '1418-10-810-023', '1418-15-510-013', '1418-15-510-012', '1418-15-510-011', '1418-11-410-004', '1418-15-510-010', '1418-15-510-007', '1418-15-510-008', '1418-15-510-009', '1418-15-510-006', '1418-15-510-004', '1418-10-710-055', '1418-10-710-062', '1418-11-411-002', '1418-15-501-001', '1418-10-611-001', '1418-10-710-065', '1418-10-710-001', '1418-10-710-078', '1418-10-710-075', '1418-10-710-076', '1418-10-710-077', '1418-10-710-074', '1418-10-710-073', '1418-10-710-072', '1418-10-710-002', '1418-10-710-003', '1418-11-110-010', '1418-11-110-011', '1418-11-110-009', '1418-10-511-008', '1418-11-311-013', '1418-11-412-030', '1418-15-601-002', '1418-15-701-006', '1418-15-701-007', '1418-22-501-003', '1418-22-502-001', '1418-15-601-003', '1418-10-511-021', '1418-11-110-015', '1418-11-411-001', '1418-11-311-011', '1418-11-311-012', '1418-11-312-001', '1318-22-002-014', '1318-22-002-067', '1318-22-002-066', '1318-22-002-065', '1318-22-002-069', '1318-22-002-064', '1318-22-002-036', '1318-22-002-063', '1318-22-002-037', '1318-22-002-079', '1318-22-002-060', '1318-22-002-018', '1318-22-002-028', '1318-22-002-044', '1318-22-002-030', '1318-22-002-084', '1318-22-002-083', '1318-22-002-049', '1318-22-002-040', '1318-22-002-050', '1318-22-002-068', '1318-22-002-087', '1318-22-002-039', '1318-22-002-080', '1318-22-002-088', '1318-22-002-089', '1318-22-002-090', '1318-22-002-076', '1318-22-002-059', '1318-22-002-058', '1318-22-002-057', '1318-22-002-101', '1318-22-002-100', '1318-22-002-013', '1318-22-002-035', '1318-22-002-007', '1318-22-002-071', '1318-22-002-077', '1318-22-002-038', '1318-22-311-000', '1318-22-311-022', '1318-22-312-020', '1418-03-801-001', '1318-22-310-008', '1318-22-310-009', '1318-22-310-010', '1318-24-302-003', '1418-34-401-003', '1318-15-410-021', '1318-23-814-009', '1318-23-814-008', '1319-19-411-030', '1318-22-310-012', '1318-22-310-013', '1318-22-002-114', '015-370-001', '014-291-011', '014-283-008', '014-283-007', '014-261-001', '014-281-004', '014-231-009', '014-284-011', '014-284-012', '014-286-006', '014-262-008', '014-021-013', '015-351-001', '014-292-011', '015-031-018', '015-031-009', '015-031-010', '014-292-012', '015-032-024', '014-234-009', '015-381-001', '014-234-006', '015-033-017', '015-331-010', '015-033-016', '015-033-014', '014-233-010', '014-291-010', '014-261-002', '015-033-015', '014-233-013', '014-232-001', '014-232-003', '015-034-019', '015-034-020', '015-381-002', '015-034-021', '015-035-001', '015-035-002', '015-034-022', '015-034-017', '014-262-009', '014-284-010', '014-284-013', '015-351-002', '015-351-036', '015-381-003', '014-283-006', '015-351-035', '014-292-010', '015-351-034', '014-281-002', '015-351-033', '015-032-025', '015-331-002', '015-351-032', '015-351-031', '015-351-003', '015-381-004', '015-351-030', '014-286-005', '015-331-003', '014-292-013', '014-021-012', '015-351-004', '015-381-005', '014-291-009', '014-261-003', '015-381-006', '015-032-023', '014-231-010', '014-234-010', '015-033-025', '015-031-019', '015-033-026', '014-234-011', '015-351-005', '015-034-002', '015-034-015', '014-292-009', '014-233-011', '015-381-007', '014-233-012', '014-284-014', '015-351-029', '014-232-002', '014-262-010', '015-331-004', '015-351-006', '015-351-026', '015-322-001', '015-381-008', '015-340-015', '014-292-014', '015-035-003', '015-351-028', '014-283-010', '014-283-005', '015-351-007', '015-381-009', '014-281-006', '015-033-024', '014-286-004', '015-034-003', '015-031-012', '015-033-027', '015-351-025', '015-034-014', '015-351-027', '015-351-008', '015-331-011', '014-291-008', '014-261-004', '015-381-010', '014-292-008', '015-031-007', '015-351-024', '015-032-003', '015-032-022', '015-381-011', '014-234-015', '015-033-009', '014-231-007', '015-033-023', '014-234-002', '015-034-004', '014-284-015', '014-284-008', '015-351-023', '015-034-013', '014-233-015', '014-292-015', '015-340-001', '014-233-014', '015-351-011', '015-351-012', '014-232-008', '015-381-012', '015-351-009', '015-322-002', '015-031-017', '014-262-011', '015-031-006', '015-351-022', '014-283-011', '014-291-007', '014-261-005', '014-283-004', '014-262-001', '015-032-004', '015-331-005', '015-381-013', '014-286-003', '015-032-014', '015-033-021', '015-033-011', '015-034-005', '014-281-007', '014-292-007', '015-034-012', '015-340-010', '015-035-004', '014-232-010', '015-381-014', '015-351-021', '015-322-003', '015-381-015', '015-323-001', '015-031-016', '014-292-016', '015-031-005', '015-381-016', '015-351-013', '014-284-016', '014-284-007', '015-351-020', '015-032-005', '015-033-019', '014-234-014', '015-033-008', '015-340-016', '014-234-003', '015-340-019', '014-231-008', '015-034-006', '015-381-017', '014-233-008', '014-262-012', '015-034-026', '014-233-009', '014-291-006', '014-261-006', '014-262-002', '015-381-018', '015-351-019', '015-032-013', '014-232-007', '015-351-018', '015-381-019', '015-351-015', '014-292-006', '015-031-003', '015-322-004', '014-283-012', '014-283-003', '015-031-013', '014-286-002', '015-032-019', '015-033-020', '015-351-017', '015-033-007', '014-292-017', '015-034-007', '014-281-003', '015-034-025', '015-381-020', '014-284-006', '014-262-013', '014-284-017', '015-031-020', '014-291-005', '014-261-007', '015-381-021', '014-262-003', '015-032-018', '015-032-020', '014-234-013', '014-234-012', '015-033-006', '015-331-019', '014-233-005', '014-292-005', '015-034-024', '014-231-005', '014-233-004', '015-323-034', '015-034-010', '014-232-009', '014-292-018', '015-031-015', '014-283-002', '014-286-001', '015-322-006', '015-031-021', '015-032-008', '015-032-021', '015-323-002', '014-262-014', '014-281-001', '015-034-023', '014-262-004', '014-291-004', '014-261-008', '015-034-009', '015-331-025', '015-340-017', '014-284-005', '014-284-018', '015-322-026', '015-331-020', '015-340-018', '015-061-014', '014-283-014', '015-061-016', '014-262-015', '014-238-004', '015-062-024', '014-238-012', '015-331-017', '015-062-010', '014-262-005', '014-237-004', '014-283-001', '014-291-012', '014-261-009', '014-237-001', '015-063-001', '014-284-004', '015-063-019', '015-323-004', '014-236-006', '015-064-001', '014-236-008', '015-323-033', '015-064-026', '015-065-001', '014-292-020', '014-284-019', '015-061-018', '015-061-017', '015-062-025', '014-285-005', '015-062-009', '014-283-015', '015-331-028', '015-063-002', '015-063-020', '015-323-005', '015-323-032', '014-262-016', '015-064-002', '014-262-006', '014-284-003', '014-292-002', '014-238-006', '014-238-011', '015-061-019', '015-061-015', '014-237-007', '014-292-021', '015-062-011', '014-237-005', '014-282-001', '015-062-008', '014-236-005', '014-236-007', '015-063-010', '015-323-006', '015-064-003', '015-064-021', '014-284-020', '014-284-001', '015-061-010', '014-285-004', '014-262-007', '015-061-008', '015-370-003', '014-292-001', '015-062-012', '014-291-001', '015-340-009', '015-063-014', '015-340-005', '015-322-011', '014-282-002', '014-292-022', '015-323-007', '015-324-023', '015-064-004', '015-370-004', '015-064-011', '014-238-015', '014-284-002', '014-237-008', '015-331-012', '015-061-027', '014-237-003', '015-370-005', '015-062-021', '014-282-003', '014-236-011', '015-062-020', '014-235-005', '014-236-010', '015-370-006', '015-063-015', '014-273-001', '014-273-005', '015-323-008', '014-285-003', '015-064-014', '015-323-029', '014-271-001', '014-284-021', '015-064-016', '015-370-007', '014-272-007', '014-272-001', '015-324-003', '015-340-004', '015-062-022', '015-331-015', '015-062-023', '015-063-016', '015-370-008', '015-063-008', '015-323-009', '014-282-004', '015-064-013', '015-323-028', '015-064-020', '014-238-013', '014-282-008', '014-273-006', '014-273-002', '015-065-003', '014-237-011', '014-271-002', '014-285-010', '014-237-013', '014-285-011', '015-331-014', '014-272-006', '015-061-005', '014-272-002', '015-062-017', '014-236-012', '015-062-006', '014-292-024', '014-236-003', '014-235-008', '015-063-017', '015-322-013', '015-323-010', '015-064-006', '015-323-027', '015-324-005', '014-282-009', '015-064-023', '015-324-022', '015-340-006', '015-340-007', '015-061-024', '015-061-026', '015-062-016', '014-273-007', '014-273-003', '015-062-013', '015-325-001', '014-271-003', '015-322-014', '015-063-006', '014-282-010', '014-272-005', '014-272-004', '015-064-007', '014-272-003', '015-323-035', '014-292-025', '014-238-014', '015-064-025', '015-324-021', '014-237-010', '014-237-012', '014-282-011', '015-061-021', '015-061-025', '014-236-004', '014-282-005', '014-282-012', '015-062-014', '014-235-006', '014-282-013', '015-322-015', '014-282-007', '015-063-005', '014-282-014', '014-273-004', '015-064-008', '014-273-008', '014-271-004', '015-324-007', '015-064-024', '015-324-020', '015-325-003', '015-061-022', '015-061-023', '014-292-026', '015-062-018', '015-322-016', '015-063-004', '015-323-013', '015-323-024', '015-391-001', '015-064-009', '015-324-008', '015-325-002', '015-324-019', '014-271-005', '015-322-017', '015-323-014', '015-323-023', '014-271-006', '015-324-009', '015-324-018', '014-244-010', '015-101-001', '015-101-006', '014-021-011', '014-243-010', '015-102-028', '014-274-001', '014-241-006', '015-326-004', '015-326-005', '015-102-023', '015-103-029', '014-271-007', '015-103-030', '014-271-008', '014-271-009', '015-103-010', '014-242-006', '015-104-011', '015-104-018', '015-326-001', '015-323-022', '015-326-002', '015-326-003', '015-391-027', '015-324-010', '015-324-017', '015-101-002', '015-101-021', '015-102-027', '015-102-024', '015-391-026', '015-103-011', '015-104-017', '015-104-010', '015-323-016', '015-323-021', '014-274-002', '015-324-011', '015-324-016', '015-101-020', '014-244-007', '015-102-025', '014-244-011', '015-391-025', '014-243-012', '015-103-021', '015-104-002', '015-323-017', '014-271-010', '015-391-024', '015-323-020', '015-324-012', '015-326-006', '015-324-024', '014-242-008', '015-101-004', '015-391-023', '015-101-019', '015-102-031', '014-274-003', '015-391-022', '015-103-022', '015-104-003', '015-102-026', '015-323-018', '015-323-019', '015-324-013', '015-326-007', '015-101-007', '015-101-008', '015-103-018', '015-103-008', '015-370-013', '014-244-005', '015-104-004', '014-244-006', '014-274-004', '014-274-013', '015-101-009', '014-241-007', '015-101-018', '015-102-013', '014-243-007', '015-103-017', '015-103-013', '015-112-011', '015-104-016', '015-104-008', '015-391-021', '015-113-001', '014-274-014', '015-114-006', '015-101-023', '015-101-017', '015-102-012', '015-102-022', '015-391-020', '015-370-019', '014-274-005', '015-103-016', '015-103-007', '015-112-010', '015-104-015', '015-104-007', '015-391-019', '015-101-022', '015-114-002', '015-102-011', '015-370-024', '015-102-006', '015-391-018', '015-103-006', '014-244-012', '015-112-009', '015-112-004', '015-104-014', '015-104-020', '015-113-003', '014-274-006', '015-391-017', '015-114-007', '015-101-012', '015-101-015', '015-102-010', '015-102-007', '015-391-016', '015-103-005', '015-112-008', '015-104-013', '015-113-004', '015-101-013', '015-370-021', '014-332-011', '015-101-014', '015-102-009', '014-274-007', '015-102-008', '015-103-012', '015-391-015', '015-103-004', '015-112-007', '015-104-012', '015-114-005', '014-301-009', '015-370-025', '015-391-014', '014-274-008', '015-151-019', '014-332-010', '015-151-008', '015-391-013', '015-152-007', '015-152-006', '014-301-008', '015-153-011', '015-153-010', '015-154-016', '015-154-011', '015-391-012', '015-161-005', '015-162-017', '015-163-001', '015-151-020', '015-370-020', '015-151-007', '015-152-008', '015-152-005', '015-391-011', '014-274-009', '015-153-012', '015-154-017', '015-153-009', '014-301-007', '015-154-010', '015-161-002', '015-162-016', '015-391-010', '015-163-002', '015-370-015', '014-332-009', '015-151-002', '015-152-001', '014-302-001', '015-152-017', '015-153-008', '015-154-018', '014-301-006', '015-161-003', '015-154-009', '015-391-009', '015-162-020', '015-163-003', '015-151-022', '015-152-009', '015-391-008', '014-301-005', '015-152-018', '015-153-007', '015-153-014', '015-154-002', '015-154-008', '014-332-008', '015-162-019', '015-391-007', '015-163-004', '015-162-021', '015-151-017', '015-391-006', '015-370-016', '014-301-004', '015-152-010', '014-331-003', '015-152-003', '014-248-016', '015-153-001', '015-154-019', '015-154-024', '015-161-019', '015-162-015', '015-162-006', '014-301-003', '015-163-005', '015-391-005', '015-151-018', '015-151-009', '015-391-003', '015-391-004', '015-152-011', '015-152-019', '015-153-015', '014-301-002', '015-154-004', '015-391-002', '015-161-018', '015-370-010', '015-162-005', '015-163-006', '014-301-001', '015-151-004', '014-248-018', '014-332-007', '015-151-010', '015-370-017', '015-152-012', '014-331-004', '015-152-022', '015-153-016', '015-153-005', '014-303-013', '015-154-022', '015-161-017', '015-154-012', '015-162-013', '015-163-007', '015-151-016', '015-151-011', '015-152-013', '015-153-017', '015-152-020', '015-154-023', '015-154-013', '015-161-016', '015-162-012', '015-163-010', '015-151-015', '015-151-021', '014-332-006', '015-152-014', '015-370-018', '014-331-005', '015-153-018', '014-302-004', '015-152-021', '014-303-015', '015-164-001', '015-164-008', '015-164-009', '015-164-010', '015-154-021', '015-164-012', '015-164-011', '015-164-013', '015-164-014', '014-303-011', '015-164-005', '015-161-012', '015-162-011', '015-163-009', '015-151-014', '015-152-015', '015-152-016', '015-153-002', '015-153-003', '015-154-015', '015-161-014', '015-161-013', '015-162-010', '014-304-001', '014-303-010', '014-332-005', '014-331-006', '015-201-024', '015-202-005', '015-202-004', '015-203-001', '014-302-006', '015-203-012', '015-211-001', '015-204-019', '015-211-011', '015-212-001', '015-212-011', '015-213-021', '015-213-004', '015-215-001', '015-202-006', '015-202-003', '014-303-017', '014-303-009', '015-203-016', '014-332-004', '015-203-029', '014-331-007', '015-211-012', '015-212-015', '015-212-010', '015-213-020', '015-213-005', '015-214-022', '015-215-002', '015-201-023', '015-201-007', '015-202-007', '015-202-002', '015-203-017', '015-204-003', '015-211-019', '015-211-013', '015-212-014', '015-212-019', '015-213-002', '015-213-006', '015-201-022', '015-215-003', '015-201-005', '014-332-003', '014-304-003', '015-202-001', '014-331-008', '014-303-008', '015-211-003', '015-204-022', '015-211-014', '015-212-013', '015-213-007', '015-201-009', '015-215-004', '014-302-009', '015-203-034', '015-201-010', '015-202-008', '015-204-024', '015-211-004', '015-204-023', '015-211-015', '015-212-012', '014-302-011', '015-213-008', '015-214-019', '014-332-002', '015-212-020', '015-201-003', '015-215-005', '015-201-011', '015-202-009', '014-331-009', '015-203-007', '014-303-007', '015-204-025', '015-204-016', '015-211-005', '015-211-010', '014-304-004', '015-212-003', '015-213-009', '015-214-017', '015-214-009', '015-201-004', '015-215-009', '015-201-012', '015-203-006', '015-204-021', '015-204-010', '015-211-016', '015-212-004', '014-302-014', '014-302-015', '015-212-017', '015-213-017', '015-213-010', '015-214-018', '014-332-001', '015-214-014', '015-201-018', '015-201-013', '015-215-010', '014-331-010', '015-203-018', '015-204-015', '015-211-006', '015-204-009', '015-211-017', '014-303-006', '015-212-005', '015-212-024', '015-213-016', '015-213-011', '015-214-004', '015-214-008', '015-201-017', '015-215-012', '015-202-012', '015-202-015', '015-203-019', '015-204-014', '015-211-018', '015-212-022', '015-212-025', '015-213-015', '015-213-023', '015-213-022', '014-342-006', '015-214-023', '015-201-016', '015-214-007', '014-302-013', '015-201-015', '015-215-011', '015-202-013', '014-341-001', '015-203-020', '015-204-007', '015-204-008', '015-211-009', '014-302-016', '015-212-023', '015-212-007', '015-213-014', '015-214-024', '014-303-005', '015-214-006', '015-215-008', '015-251-001', '014-342-005', '015-251-023', '015-252-017', '015-253-001', '014-341-002', '015-253-011', '015-254-019', '015-254-021', '015-261-033', '015-261-025', '015-262-001', '015-262-016', '015-263-016', '015-263-017', '014-302-017', '015-264-001', '015-264-015', '015-251-024', '015-265-013', '015-252-019', '015-252-016', '014-302-018', '015-253-002', '015-254-020', '015-254-022', '015-261-024', '015-262-022', '014-303-004', '015-263-015', '015-263-018', '014-342-004', '015-251-013', '015-264-014', '014-303-003', '015-252-020', '014-341-003', '015-252-015', '015-253-024', '015-253-021', '014-302-019', '014-304-007', '015-254-017', '015-254-015', '015-261-021', '015-261-012', '015-262-023', '015-262-015', '015-263-014', '015-263-019', '015-251-017', '014-323-001', '015-265-003', '015-252-014', '015-253-023', '015-253-022', '015-254-016', '015-254-011', '015-261-003', '015-261-018', '015-262-003', '015-262-014', '014-342-003', '015-263-013', '015-263-020', '015-251-004', '014-341-004', '015-265-004', '014-303-001', '015-252-013', '015-253-026', '015-253-027', '015-254-023', '015-254-024', '015-262-020', '015-262-024', '015-263-012', '015-263-001', '014-304-008', '015-251-016', '015-265-010', '015-265-011', '015-252-012', '014-322-011', '015-253-017', '015-253-018', '015-254-025', '014-342-002', '015-262-019', '014-341-005', '015-263-011', '015-263-002', '015-251-006', '015-264-002', '014-322-012', '015-264-009', '015-252-011', '015-265-012', '015-253-015', '015-253-016', '015-254-026', '015-262-017', '014-322-010', '015-263-010', '014-323-003', '015-263-003', '015-251-007', '015-264-003', '014-321-001', '015-264-018', '015-252-005', '015-252-010', '015-253-007', '015-253-012', '014-322-013', '015-254-007', '014-342-001', '015-262-005', '015-262-010', '015-263-009', '015-263-004', '015-251-025', '015-251-021', '015-251-022', '015-264-019', '015-252-006', '015-265-006', '015-252-009', '015-253-008', '015-253-025', '014-323-004', '014-321-002', '015-262-006', '015-262-009', '015-263-008', '015-263-005', '015-251-026', '015-264-004', '014-322-014', '015-264-007', '015-252-007', '014-322-009', '015-265-007', '015-252-008', '015-253-009', '015-262-007', '015-262-008', '015-263-007', '015-263-006', '015-264-005', '015-264-006', '015-265-008', '015-301-001', '015-301-014', '014-343-006', '015-302-030', '014-322-008', '015-302-018', '015-303-027', '015-303-028', '015-304-001', '015-304-025', '015-311-031', '015-311-017', '014-321-004', '014-322-015', '015-312-001', '015-312-032', '014-351-001', '015-313-033', '015-313-023', '015-314-001', '015-314-019', '015-315-013', '015-302-017', '015-315-014', '014-324-001', '015-303-017', '015-304-002', '015-304-026', '015-311-016', '015-312-002', '015-312-033', '014-321-005', '015-313-034', '015-314-002', '015-302-023', '015-314-018', '014-322-007', '015-302-016', '014-343-005', '015-303-018', '015-303-016', '015-311-025', '015-312-003', '015-312-021', '015-314-003', '014-321-006', '015-314-017', '015-302-015', '015-315-002', '015-303-019', '015-303-015', '014-351-002', '015-311-026', '015-311-029', '015-312-023', '015-312-027', '015-313-030', '015-301-024', '015-314-004', '015-314-016', '015-315-003', '015-302-026', '015-303-020', '014-322-017', '015-303-014', '015-304-024', '015-304-020', '015-311-003', '015-311-030', '014-322-006', '015-312-034', '015-312-028', '015-313-003', '015-313-015', '015-314-005', '015-301-009', '015-314-028', '015-302-004', '015-302-027', '015-315-004', '015-303-026', '015-304-023', '015-304-021', '014-321-007', '015-311-004', '015-311-014', '014-351-003', '015-312-025', '015-313-004', '015-313-016', '015-301-029', '015-301-008', '014-343-003', '015-314-006', '015-302-013', '014-322-018', '015-315-005', '015-312-036', '015-303-025', '015-304-022', '015-304-017', '015-311-005', '015-311-013', '014-322-005', '015-312-026', '015-313-005', '015-313-017', '015-301-021', '015-314-007', '015-302-034', '015-315-006', '015-302-024', '014-321-008', '015-303-029', '015-304-006', '015-304-016', '015-311-019', '014-351-004', '015-311-022', '015-312-006', '015-312-015', '014-343-002', '015-313-018', '015-314-022', '015-314-013', '015-302-033', '015-315-007', '015-302-025', '015-303-030', '015-303-024', '014-322-004', '015-304-007', '015-304-015', '015-311-006', '014-322-019', '014-321-009', '015-311-024', '015-312-007', '015-312-014', '015-313-006', '015-301-025', '015-313-019', '015-314-023', '015-302-039', '015-314-021', '015-315-008', '014-351-005', '015-302-021', '015-303-006', '015-303-012', '015-304-008', '015-304-014', '015-311-020', '015-311-023', '014-324-005', '015-312-008', '015-312-013', '014-322-003', '014-343-001', '015-313-021', '015-301-006', '015-302-008', '015-314-020', '015-315-009', '015-303-021', '015-303-011', '014-321-010', '015-304-013', '015-311-007', '015-311-011', '014-322-020', '015-312-009', '015-312-012', '015-313-031', '015-301-020', '014-351-006', '015-301-017', '015-314-025', '015-315-012', '014-324-006', '014-322-002', '015-303-022', '015-303-010', '015-304-030', '015-304-012', '015-311-008', '015-311-010', '015-312-029', '015-312-011', '015-313-010', '015-301-018', '015-302-029', '014-321-011', '015-314-026', '015-303-008', '015-303-009', '014-324-007', '015-304-029', '015-304-011', '015-311-009', '015-312-030', '015-313-014', '014-322-001', '015-313-009', '015-314-027', '014-352-005', '014-351-008', '014-352-004', '014-351-009', '014-352-003', '014-352-002', '014-352-001', '016-031-027', '016-300-038', '016-531-012', '016-531-013', '016-531-014', '016-531-007', '016-531-018', '016-531-017', '016-531-004', '016-531-002', '016-531-016', '016-531-015', '016-532-028', '016-532-027', '016-532-026', '016-532-007', '016-532-003', '016-532-004', '016-532-005', '016-300-007', '016-532-021', '016-532-009', '016-534-001', '016-532-014', '016-583-030', '016-300-027', '016-532-016', '016-300-053', '016-581-001', '016-583-002', '016-532-017', '016-583-004', '016-583-005', '016-300-009', '016-583-022', '016-535-010', '016-533-001', '016-300-026', '016-583-003', '016-535-006', '016-300-023', '016-535-005', '016-300-010', '016-535-003', '016-583-008', '016-583-009', '016-300-045', '016-300-046', '016-582-010', '016-583-010', '016-535-001', '016-582-009', '016-583-011', '016-582-007', '016-582-006', '016-583-012', '016-582-005', '016-582-003', '016-590-010', '016-590-008', '016-583-014', '016-300-037', '016-582-001', '016-583-015', '016-590-009', '016-583-016', '016-590-005', '016-583-017', '016-583-018', '016-583-021', '016-590-002', '016-583-019', '016-300-003', '016-261-002', '016-261-001', '016-261-011', '016-261-006', '016-261-012', '016-261-013', '016-051-003', '016-051-058', '016-051-033', '016-051-035', '016-051-034', '016-410-001', '016-051-049', '016-051-050', '016-051-051', '016-052-003', '016-051-052', '016-333-007', '016-052-004', '016-051-009', '016-333-008', '016-052-005', '016-051-010', '016-051-037', '016-052-006', '016-051-026', '016-051-012', '016-332-010', '016-052-007', '016-051-013', '016-053-008', '016-332-011', '016-544-012', '016-333-006', '016-051-014', '016-544-011', '016-051-057', '016-053-003', '016-543-002', '016-544-010', '016-051-056', '016-332-039', '016-053-004', '016-541-001', '016-323-007', '016-051-055', '016-053-005', '016-544-008', '016-053-006', '016-410-009', '016-323-008', '016-061-001', '016-053-007', '016-544-007', '016-332-026', '016-061-006', '016-323-002', '016-062-001', '016-063-001', '016-062-002', '016-063-016', '016-332-030', '016-410-003', '016-544-005', '016-062-008', '016-542-011', '016-332-036', '016-062-004', '016-321-001', '016-544-017', '016-323-005', '016-542-009', '016-323-006', '016-063-014', '016-551-004', '016-321-015', '016-063-015', '016-324-004', '016-324-006', '016-063-010', '016-324-005', '016-324-001', '016-063-005', '016-410-005', '016-332-035', '016-553-010', '016-063-012', '016-063-011', '016-322-012', '016-322-017', '016-322-011', '016-063-008', '016-553-011', '016-321-006', '016-552-009', '016-502-001', '016-552-008', '016-554-011', '016-554-001', '016-081-020', '016-554-010', '016-552-014', '016-502-002', '016-552-013', '016-554-002', '016-502-003', '016-081-030', '016-501-009', '016-081-002', '016-081-003', '016-554-006', '016-081-004', '016-501-010', '016-561-005', '016-081-039', '016-561-004', '016-081-006', '016-081-007', '016-563-007', '016-081-008', '016-081-009', '016-561-008', '016-081-010', '016-501-013', '016-081-036', '016-561-009', '016-563-010', '016-081-012', '016-563-003', '016-563-009', '016-512-002', '016-081-037', '016-513-014', '016-081-041', '016-512-004', '016-081-040', '016-513-012', '016-512-005', '016-081-021', '016-081-029', '016-511-001', '016-081-028', '016-512-006', '016-081-018', '016-081-038', '016-513-006', '016-091-025', '016-091-045', '016-511-002', '016-091-035', '016-091-055', '016-521-007', '016-091-056', '016-511-004', '016-091-040', '016-513-001', '016-521-005', '016-522-001', '016-091-041', '016-522-002', '016-091-049', '016-091-004', '016-091-048', '016-091-005', '016-091-016', '016-091-026', '016-091-046', '016-091-057', '016-522-006', '016-091-018', '016-091-058', '016-091-019', '016-522-005', '016-522-008', '016-091-047', '016-091-008', '016-091-027', '016-091-028', '016-091-021', '016-524-004', '016-091-059', '016-091-022', '016-522-013', '016-091-030', '016-091-023', '016-091-033', '016-524-003', '016-101-077', '016-101-078', '016-101-004', '016-101-065', '016-101-049', '016-524-007', '016-101-093', '016-522-017', '016-101-066', '016-101-050', '016-101-067', '016-101-006', '016-101-073', '016-101-068', '016-101-072', '016-101-069', '016-101-083', '016-101-070', '016-101-084', '016-101-080', '016-101-085', '016-101-081', '016-101-086', '016-101-061', '016-493-009', '016-101-063', '016-101-002', '016-101-010', '016-101-092', '016-101-057', '016-101-091', '016-493-004', '016-101-055', '016-101-090', '016-101-059', '016-101-088', '016-493-010', '016-101-031', '016-493-002', '016-101-003', '016-101-013', '016-101-032', '016-101-033', '016-101-051', '016-161-039', '016-161-035', '016-161-023', '016-143-010', '016-142-009', '016-161-017', '016-143-009', '016-142-008', '016-161-038', '016-161-022', '016-143-017', '016-143-012', '016-142-007', '016-161-012', '016-143-018', '016-143-011', '016-142-006', '016-151-026', '016-161-016', '016-143-016', '016-161-015', '016-143-014', '016-142-005', '016-161-026', '016-151-023', '016-161-031', '016-151-025', '016-143-007', '016-161-028', '016-142-026', '016-151-022', '016-161-030', '016-161-027', '016-142-025', '016-142-023', '016-151-039', '016-151-021', '016-142-016', '016-161-021', '016-161-008', '016-151-019', '016-142-024', '016-171-009', '016-142-022', '016-142-029', '016-151-040', '010-180-004', '016-161-020', '016-142-021', '016-142-013', '016-161-019', '016-151-017', '016-142-020', '016-161-018', '016-151-007', '016-142-019', '016-151-035', '016-171-005', '016-142-027', '016-151-030', '016-141-007', '016-171-007', '016-142-028', '016-171-006', '016-151-029', '016-151-031', '016-141-005', '016-142-001', '016-151-028', '016-151-033', '016-141-006', '016-142-010', '016-151-027', '016-142-011', '016-151-032', '016-390-007', '016-390-005', '016-390-002', '016-181-013', '016-282-001', '016-282-002', '016-390-008', '016-181-019', '016-282-004', '016-282-005', '016-361-012', '016-181-016', '016-401-014', '016-181-020', '016-362-002', '016-401-004', '016-362-009', '016-361-013', '016-181-021', '016-362-003', '016-361-014', '016-401-012', '016-181-015', '016-361-011', '016-362-008', '016-361-015', '016-281-010', '016-401-006', '016-362-005', '016-281-002', '016-361-024', '016-362-006', '016-401-007', '016-362-007', '016-181-006', '016-281-003', '016-361-023', '016-361-030', '016-361-022', '016-401-021', '016-361-010', '016-361-021', '016-361-020', '016-361-002', '016-401-020', '016-361-029', '016-361-003', '016-211-009', '016-284-004', '016-284-003', '016-211-015', '016-363-002', '016-203-003', '016-281-007', '016-203-002', '016-361-005', '016-363-001', '016-281-008', '016-284-002', '016-281-009', '016-283-002', '016-191-030', '016-191-020', '016-211-013', '016-191-021', '016-202-018', '016-283-007', '016-363-004', '016-283-001', '016-202-017', '016-284-001', '016-211-012', '016-371-001', '016-191-031', '016-373-008', '016-191-022', '016-292-001', '016-211-011', '016-202-013', '016-294-002', '016-291-001', '016-191-032', '016-191-023', '016-371-002', '016-211-008', '016-373-002', '016-202-021', '016-202-022', '016-191-033', '016-191-024', '016-461-004', '016-211-006', '016-461-003', '016-461-001', '016-373-003', '016-373-001', '016-373-004', '016-191-034', '016-191-018', '016-202-024', '016-202-023', '016-221-002', '016-373-005', '016-191-009', '016-191-010', '016-461-013', '016-221-003', '016-462-008', '016-371-004', '016-372-006', '016-293-005', '016-202-006', '016-202-016', '016-191-011', '016-191-012', '016-372-005', '016-221-004', '016-462-007', '016-293-002', '016-202-005', '016-202-019', '016-461-007', '016-372-004', '016-221-006', '016-191-013', '016-202-020', '016-191-014', '016-221-005', '016-462-006', '016-371-006', '016-372-003', '016-202-004', '016-372-001', '016-372-002', '016-293-001', '016-461-008', '016-191-025', '016-191-026', '016-462-003', '016-202-003', '016-202-008', '016-221-015', '016-461-009', '016-221-007', '016-191-028', '016-371-009', '016-292-032', '016-292-033', '016-191-027', '016-461-010', '016-371-010', '016-202-007', '016-371-011', '016-292-034', '016-221-014', '016-202-002', '016-292-031', '016-461-011', '016-462-009', '016-251-007', '016-202-001', '016-421-004', '016-201-002', '016-232-007', '016-461-012', '016-232-006', '016-251-006', '016-232-002', '016-201-003', '016-421-003', '016-421-002', '016-232-005', '016-251-005', '016-421-001', '016-232-001', '016-231-014', '016-381-016', '016-381-015', '016-232-004', '016-381-018', '016-251-004', '016-231-013', '016-232-008', '016-231-012', '016-232-003', '016-471-004', '016-232-009', '016-231-009', '016-382-001', '016-381-012', '016-421-006', '016-242-006', '016-471-003', '016-232-010', '016-382-009', '016-382-008', '016-231-010', '016-382-007', '016-382-010', '016-382-006', '016-381-011', '016-382-004', '016-242-005', '016-242-007', '016-231-007', '016-381-010', '016-382-011', '016-251-002', '016-471-001', '016-422-007', '016-231-011', '016-242-013', '016-242-002', '016-381-009', '016-422-005', '016-382-012', '016-422-009', '016-442-002', '016-423-003', '016-381-008', '016-442-001', '016-422-010', '016-382-013', '016-242-008', '016-242-015', '016-472-009', '016-241-007', '016-381-007', '016-422-011', '016-382-014', '016-241-005', '016-442-011', '016-242-012', '016-381-006', '016-242-014', '016-382-015', '016-422-012', '016-381-005', '016-241-004', '016-382-016', '016-425-002', '016-443-003', '016-422-013', '016-242-009', '016-443-002', '016-424-002', '016-381-004', '016-382-017', '016-422-014', '016-443-006', '016-241-003', '016-242-010', '016-381-003', '016-443-007', '016-382-018', '016-433-001', '016-241-002', '016-251-001', '016-431-001', '016-433-002', '016-241-001', '016-381-002', '016-443-008', '016-432-009', '016-382-020', '016-251-008', '016-432-007', '016-311-010', '016-451-010', '016-483-002', '016-435-001', '016-434-010', '016-313-011', '016-131-001', '016-432-006', '016-432-010', '016-483-004', '016-481-015', '016-451-009', '016-435-002', '016-313-010', '016-311-008', '016-483-005', '016-481-008', '016-434-009', '016-451-008', '016-435-003', '016-482-010', '016-313-012', '016-432-012', '016-311-007', '016-432-003', '016-432-013', '016-483-006', '016-481-017', '016-451-007', '016-435-004', '016-313-013', '016-311-006', '016-434-008', '016-452-001', '016-432-014', '016-483-007', '016-432-002', '016-451-006', '016-313-008', '016-131-007', '016-435-005', '016-311-005', '016-432-015', '016-482-015', '016-313-016', '016-131-006', '016-435-006', '016-311-004', '016-434-006', '016-432-016', '016-131-004', '016-435-007', '016-481-003', '016-313-003', '016-312-001', '016-432-017', '016-435-008', '016-481-002', '016-313-014', '016-435-010', '016-312-003', '016-483-011', '016-435-009', '016-451-002', '016-434-002', '016-481-001', '016-131-005', '016-434-001', '016-483-012', '016-452-006', '016-313-015', '016-312-004', '016-451-001', '016-313-001', '016-600-008', '016-600-010', '016-600-013', '017-021-009', '017-021-001', '016-600-020', '016-600-019', '017-021-017', '017-021-016', '016-600-007', '016-600-021', '017-021-021', '017-021-006', '017-011-001', '017-021-010', '017-021-014', '017-021-008', '017-021-011', '017-041-021', '017-041-029', '017-041-031', '017-041-010', '017-041-009', '017-041-027', '017-031-001', '017-041-025', '017-041-017', '017-041-018', '017-041-023', '017-061-006', '017-061-007', '017-061-005', '017-061-003', '029-010-016', '029-010-017', '029-010-013', '029-010-014', '029-010-012', '029-010-011', '029-010-009', '029-010-015', '029-601-001', '029-041-041', '029-051-001', '029-610-003', '029-610-002', '029-610-001', '029-041-042', '029-610-004', '029-041-002', '029-610-021', '029-041-003', '029-602-003', '029-602-004', '029-610-005', '029-041-056', '029-602-010', '029-041-055', '029-051-020', '029-602-009', '029-602-012', '029-602-008', '029-602-007', '029-602-006', '029-602-005', '029-061-001', '029-610-006', '029-041-007', '029-610-007', '029-603-001', '029-041-009', '029-610-008', '029-604-010', '029-610-009', '029-061-002', '029-603-002', '029-610-010', '029-604-011', '029-605-001', '029-603-003', '029-610-025', '029-061-011', '029-610-011', '029-605-002', '029-604-012', '029-051-019', '029-041-059', '029-604-017', '029-041-047', '029-603-004', '029-605-009', '029-604-023', '029-610-012', '029-604-001', '029-051-008', '029-061-015', '029-603-011', '029-061-012', '029-604-019', '029-604-024', '029-041-060', '029-603-012', '029-605-013', '029-610-020', '029-604-025', '029-041-057', '029-603-005', '029-605-012', '029-604-002', '029-051-010', '029-061-010', '029-061-014', '029-605-003', '029-604-013', '029-603-006', '029-610-019', '029-604-003', '029-051-015', '029-051-014', '029-610-026', '029-061-008', '029-190-032', '029-605-004', '029-061-013', '029-604-014', '029-610-027', '029-604-004', '029-603-007', '029-041-052', '029-061-009', '029-610-023', '029-604-005', '029-051-016', '029-604-015', '029-603-009', '029-610-018', '029-610-022', '029-604-006', '029-603-010', '029-041-017', '029-052-001', '029-605-010', '029-604-020', '029-610-013', '029-604-021', '029-041-018', '029-603-008', '029-052-002', '029-605-011', '029-610-014', '029-052-003', '029-604-022', '029-604-018', '029-610-015', '029-071-001', '029-041-020', '029-062-002', '029-604-007', '029-604-016', '029-610-016', '029-062-003', '029-041-021', '029-071-002', '029-052-019', '029-605-005', '029-604-008', '029-610-024', '029-610-017', '029-072-001', '029-041-038', '029-071-003', '029-605-006', '029-604-009', '029-605-007', '029-072-002', '029-072-005', '029-605-008', '029-052-005', '029-090-001', '029-072-003', '029-052-017', '029-190-033', '029-072-004', '029-052-013', '029-093-001', '029-073-008', '029-190-027', '029-062-020', '029-052-008', '029-072-006', '029-073-005', '029-073-007', '029-041-054', '029-052-009', '029-052-018', '029-073-006', '029-092-001', '029-073-002', '029-093-002', '029-074-018', '029-073-003', '029-074-017', '029-041-030', '029-074-007', '029-062-018', '029-073-004', '029-093-003', '029-074-006', '029-062-012', '029-074-002', '029-091-021', '029-074-003', '029-041-031', '029-053-013', '029-074-010', '029-093-012', '029-074-004', '029-074-011', '029-063-004', '029-091-027', '029-063-002', '029-093-011', '029-091-026', '029-053-008', '029-063-005', '029-053-005', '029-053-006', '029-093-010', '029-094-005', '029-091-002', '029-091-025', '029-064-006', '029-093-005', '029-081-017', '029-064-007', '029-091-016', '029-065-001', '029-064-004', '029-081-018', '029-093-006', '029-065-009', '029-081-008', '029-091-018', '029-065-003', '029-081-004', '029-082-022', '029-091-017', '029-081-015', '029-065-004', '029-093-007', '029-081-010', '029-065-008', '029-091-023', '029-081-003', '029-082-017', '029-065-006', '029-081-002', '029-082-014', '029-091-019', '029-081-009', '029-065-007', '029-093-008', '029-082-013', '029-091-010', '029-083-020', '029-082-009', '029-081-006', '029-091-022', '029-082-011', '029-082-020', '029-081-005', '029-101-011', '029-091-008', '029-082-012', '029-082-023', '029-091-011', '029-082-002', '029-101-010', '029-091-009', '029-094-001', '029-102-001', '029-101-009', '029-082-019', '029-095-012', '029-082-018', '029-083-003', '029-094-002', '029-083-019', '029-441-003', '029-103-014', '029-095-011', '029-095-007', '029-094-003', '029-095-013', '029-083-016', '029-091-014', '029-101-013', '029-095-008', '029-095-010', '029-103-019', '029-095-017', '029-094-004', '029-095-015', '029-095-014', '029-101-014', '029-103-011', '029-095-018', '029-103-010', '029-103-009', '029-096-008', '029-104-011', '029-095-009', '029-095-016', '029-103-008', '029-331-010', '029-103-007', '029-331-007', '029-170-005', '029-104-002', '029-331-008', '029-332-002', '029-101-006', '029-331-004', '029-103-006', '029-332-003', '029-104-003', '029-170-004', '029-332-004', '029-332-021', '029-341-001', '029-332-008', '029-332-020', '029-103-023', '029-101-005', '029-105-001', '029-332-012', '029-341-002', '029-104-004', '029-332-018', '029-333-001', '029-351-001', '029-332-023', '029-341-003', '029-332-024', '029-170-003', '029-105-002', '029-101-004', '029-104-005', '029-240-005', '029-333-004', '029-351-020', '029-343-017', '029-332-010', '029-333-003', '029-103-004', '029-351-019', '029-341-004', '029-333-002', '029-170-002', '029-343-016', '029-332-011', '029-343-002', '029-351-004', '029-104-006', '029-101-003', '029-103-021', '029-351-005', '029-342-001', '029-343-006', '029-343-003', '029-351-006', '029-101-002', '029-342-004', '029-343-007', '029-105-005', '029-103-024', '029-351-015', '029-342-003', '029-162-002', '029-343-008', '029-343-011', '029-351-016', '029-351-007', '029-103-022', '029-101-001', '029-343-005', '029-104-007', '029-343-019', '029-162-007', '029-352-010', '029-351-012', '029-351-008', '029-105-007', '029-343-018', '029-344-003', '029-361-001', '029-351-013', '029-101-026', '029-104-008', '029-343-012', '029-103-001', '029-361-002', '029-351-021', '029-343-010', '029-352-015', '029-103-017', '029-240-008', '029-361-003', '029-371-016', '029-351-022', '029-352-014', '029-101-025', '029-361-025', '029-161-005', '029-352-013', '029-164-002', '029-104-009', '029-371-017', '029-108-001', '029-361-026', '029-161-024', '029-352-006', '029-103-018', '029-371-019', '029-164-001', '029-361-022', '029-361-007', '029-103-002', '029-101-024', '029-371-020', '029-106-011', '029-162-003', '029-352-008', '029-105-010', '029-361-023', '029-161-025', '029-361-016', '029-352-009', '029-104-010', '029-361-011', '029-361-012', '029-101-023', '029-162-004', '029-361-013', '029-371-008', '029-106-010', '029-363-007', '029-371-012', '029-361-020', '029-371-009', '029-363-008', '029-162-005', '029-361-021', '029-163-013', '029-372-001', '029-106-002', '029-105-011', '029-363-002', '029-371-013', '029-372-002', '029-101-022', '029-106-009', '029-372-023', '027-083-008', '029-363-003', '029-372-022', '029-371-003', '029-106-001', '029-363-004', '029-374-001', '018-291-014', '029-372-024', '029-372-004', '029-101-021', '029-363-005', '029-106-003', '029-106-008', '029-163-001', '029-372-025', '029-372-028', '029-372-005', '029-371-010', '029-363-006', '029-107-001', '029-163-008', '029-372-013', '029-372-006', '029-374-006', '029-372-015', '029-150-007', '018-291-018', '029-161-017', '029-372-026', '029-374-009', '029-101-020', '029-371-011', '029-364-003', '029-106-004', '018-291-007', '029-374-008', '029-372-027', '018-291-008', '029-374-007', '018-291-010', '029-371-001', '029-161-016', '029-163-006', '029-161-006', '029-107-002', '029-373-001', '018-291-009', '029-163-007', '029-106-005', '029-101-019', '029-150-010', '029-372-010', '029-372-017', '029-372-009', '029-373-002', '029-372-008', '029-374-002', '029-375-004', '029-161-015', '029-374-010', '029-161-007', '029-374-011', '018-291-016', '029-374-004', '029-373-003', '029-106-007', '018-281-006', '029-373-004', '029-375-001', '018-281-005', '029-372-007', '029-161-008', '029-161-014', '018-281-004', '029-161-009', '029-101-018', '029-375-006', '029-373-005', '029-373-022', '029-161-010', '029-375-008', '029-161-011', '029-373-029', '029-373-028', '029-161-023', '029-375-009', '029-373-030', '029-373-027', '018-292-004', '029-161-026', '029-373-007', '029-373-008', '018-291-005', '029-161-027', '029-373-009', '029-373-010', '029-373-020', '029-373-011', '018-281-011', '029-391-021', '029-160-003', '029-373-019', '029-381-049', '018-292-003', '029-373-018', '029-373-012', '018-281-012', '029-161-022', '029-373-017', '018-292-005', '029-373-024', '029-391-020', '018-291-004', '018-292-002', '029-373-026', '029-373-025', '018-292-001', '029-381-046', '029-373-031', '027-082-013', '029-373-032', '029-373-014', '027-083-006', '018-282-008', '029-391-007', '029-381-032', '027-082-012', '027-085-005', '029-381-033', '029-391-012', '018-281-010', '029-373-013', '018-291-003', '029-391-009', '029-381-035', '027-085-004', '027-441-001', '029-391-008', '027-082-014', '029-381-005', '029-181-034', '018-281-009', '018-090-055', '029-373-023', '018-292-006', '029-381-006', '027-085-003', '027-082-015', '018-282-010', '029-181-023', '029-381-044', '027-082-016', '027-085-001', '018-291-002', '027-085-008', '027-082-007', '029-381-043', '029-181-022', '027-082-017', '029-181-033', '027-082-006', '029-381-042', '029-181-024', '029-381-012', '029-381-028', '027-082-030', '027-085-002', '029-381-009', '027-082-005', '029-181-010', '029-422-015', '027-085-007', '027-082-029', '029-240-012', '018-280-001', '027-072-033', '029-422-003', '029-401-008', '027-082-004', '029-181-025', '029-381-013', '027-085-009', '029-381-010', '027-082-028', '018-282-005', '018-291-001', '029-181-021', '018-292-007', '027-082-003', '029-381-038', '027-084-029', '018-282-002', '029-401-009', '029-401-010', '029-381-039', '029-181-012', '027-085-010', '027-082-002', '029-381-011', '029-381-029', '029-381-017', '029-381-030', '029-401-012', '029-181-020', '027-084-028', '029-391-025', '027-072-032', '029-381-040', '027-481-001', '027-084-018', '027-082-023', '029-401-011', '027-084-007', '027-451-006', '029-381-041', '018-282-001', '027-141-010', '027-451-007', '027-074-012', '027-084-017', '018-191-015', '029-391-023', '029-181-019', '027-082-024', '027-084-006', '029-381-018', '018-282-004', '027-084-016', '027-074-011', '029-181-037', '029-403-029', '027-082-025', '029-381-019', '027-141-009', '027-084-005', '027-071-029', '029-381-020', '029-401-006', '027-491-001', '027-084-015', '027-074-010', '029-381-021', '027-084-004', '029-403-030', '029-401-007', '027-491-002', '029-381-026', '029-181-036', '027-084-021', '029-381-022', '018-191-023', '018-090-056', '027-074-022', '029-411-001', '027-084-003', '027-491-003', '027-451-003', '027-072-023', '029-381-023', '018-282-003', '027-084-020', '029-181-031', '027-451-002', '027-451-004', '029-381-036', '029-411-002', '027-451-001', '027-491-004', '027-074-021', '027-084-002', '029-403-031', '029-403-026', '027-451-005', '027-141-004', '027-072-010', '029-181-014', '027-084-014', '027-491-006', '027-491-005', '029-402-009', '027-074-020', '027-084-027', '029-181-015', '027-084-013', '029-403-032', '029-411-004', '027-142-029', '029-421-001', '027-074-019', '029-403-025', '029-402-029', '029-411-005', '029-421-002', '029-402-010', '027-491-007', '027-084-012', '027-541-001', '027-141-007', '027-142-018', '029-402-028', '029-411-007', '027-074-018', '029-181-029', '029-411-008', '029-403-007', '027-084-011', '027-074-026', '027-491-008', '029-181-030', '027-142-019', '029-411-009', '027-491-013', '029-421-003', '029-402-001', '027-074-017', '029-411-015', '029-402-007', '029-422-011', '027-076-020', '029-412-012', '027-491-009', '029-411-016', '027-491-014', '027-142-020', '027-142-033', '027-074-016', '027-076-015', '027-491-015', '027-491-010', '027-491-016', '027-491-017', '027-142-021', '029-402-011', '029-181-027', '029-402-032', '029-412-013', '029-421-004', '027-142-032', '027-074-015', '027-461-007', '027-076-029', '027-491-011', '029-412-011', '027-142-011', '027-461-008', '029-181-028', '027-142-003', '027-461-009', '029-403-009', '027-076-028', '029-402-033', '029-422-013', '027-491-012', '027-461-010', '027-511-001', '027-141-008', '027-461-011', '029-402-012', '027-142-027', '029-181-017', '029-422-017', '027-076-021', '029-403-022', '027-142-004', '029-412-014', '027-076-005', '029-413-004', '027-551-008', '018-191-009', '029-320-001', '027-142-025', '027-501-018', '027-551-009', '027-076-027', '027-142-005', '027-461-006', '027-501-015', '027-074-023', '027-143-001', '027-551-010', '029-403-010', '027-501-017', '027-076-004', '029-320-003', '029-402-013', '027-551-011', '027-551-001', '027-142-023', '029-412-009', '027-501-019', '027-461-005', '027-076-026', '029-181-005', '027-142-006', '029-403-021', '029-413-005', '029-412-015', '027-431-031', '027-501-016', '027-143-002', '027-074-013', '027-501-020', '027-551-002', '018-191-010', '027-076-003', '018-090-050', '027-142-012', '027-501-021', '027-551-003', '029-422-020', '027-461-004', '027-076-012', '027-501-022', '027-142-007', '029-413-002', '029-422-012', '027-143-007', '027-551-004', '029-421-006', '029-320-002', '029-403-008', '027-521-002', '027-461-003', '027-076-002', '029-402-014', '027-521-001', '027-551-005', '027-142-015', '027-501-024', '029-412-016', '029-413-006', '029-181-004', '027-076-019', '027-461-002', '027-551-006', '027-521-003', '029-403-002', '018-191-016', '027-501-026', '027-461-001', '027-501-014', '027-076-014', '027-551-007', '027-521-005', '029-320-005', '027-521-004', '027-142-016', '027-143-006', '027-501-023', '027-076-025', '027-501-025', '027-501-012', '027-143-003', '029-402-017', '029-422-014', '029-413-012', '029-422-023', '027-521-006', '027-132-009', '027-501-013', '018-191-014', '027-431-029', '027-501-011', '029-403-011', '029-402-015', '029-320-004', '027-076-032', '029-412-002', '027-142-013', '029-413-007', '027-521-007', '029-412-025', '027-076-024', '027-501-010', '029-403-003', '027-431-027', '027-431-025', '027-132-008', '027-501-008', '027-020-017', '029-421-007', '027-501-009', '027-076-030', '029-422-021', '027-561-007', '018-191-017', '027-143-024', '027-431-023', '029-413-013', '027-431-021', '027-561-004', '027-076-022', '029-402-018', '027-143-020', '027-561-005', '027-501-007', '027-132-021', '029-402-016', '027-561-001', '029-412-007', '027-076-031', '027-561-003', '027-561-006', '029-422-026', '027-501-006', '027-143-021', '027-561-008', '029-403-033', '018-191-020', '027-132-020', '027-561-009', '027-561-002', '029-413-014', '027-501-005', '029-412-026', '027-143-025', '027-132-018', '018-191-021', '029-422-022', '029-421-008', '029-402-019', '027-143-004', '027-561-010', '027-431-019', '027-132-006', '029-402-025', '029-415-024', '029-403-013', '027-471-015', '027-075-010', '027-501-004', '027-132-017', '027-431-017', '027-561-011', '027-076-009', '027-143-010', '027-132-005', '027-501-001', '027-471-016', '027-471-014', '027-075-009', '027-501-003', '027-561-012', '029-413-015', '029-181-035', '027-143-017', '018-090-073', '029-403-035', '027-531-001', '027-132-016', '027-431-015', '029-402-020', '027-143-011', '027-076-008', '029-412-005', '029-421-009', '027-531-003', '029-402-026', '027-501-002', '029-403-014', '027-431-013', '027-132-004', '027-471-013', '029-422-024', '027-144-017', '027-075-008', '029-412-020', '027-531-002', '027-132-019', '027-531-004', '018-090-048', '027-471-017', '027-143-005', '027-571-017', '027-471-012', '027-132-003', '027-471-018', '027-531-029', '027-531-005', '029-422-028', '027-075-007', '029-413-016', '027-531-028', '029-403-036', '029-300-008', '027-471-019', '027-132-015', '029-402-021', '027-531-027', '027-431-011', '027-571-018', '027-531-006', '027-075-018', '027-144-016', '027-571-019', '027-471-011', '029-415-006', '029-403-015', '027-471-020', '027-571-020', '029-402-027', '027-132-026', '027-531-026', '027-531-023', '027-531-019', '027-471-024', '029-412-021', '027-431-009', '027-075-006', '027-471-021', '027-531-020', '027-531-024', '027-132-028', '027-571-004', '027-471-023', '027-531-025', '029-421-010', '029-422-025', '027-531-021', '027-075-017', '027-144-002', '027-371-015', '027-471-022', '027-132-025', '027-531-022', '027-571-003', '029-422-030', '029-413-017', '029-403-020', '027-075-005', '029-181-001', '018-090-063', '027-571-005', '029-402-022', '027-132-027', '027-371-014', '029-415-005', '029-403-016', '029-300-014', '027-471-010', '027-144-013', '029-413-023', '029-402-005', '027-132-001', '027-571-002', '029-412-022', '027-075-004', '027-371-013', '027-471-009', '027-132-013', '029-412-003', '027-431-005', '027-431-007', '027-531-007', '027-144-021', '027-471-005', '027-571-001', '029-300-010', '027-144-004', '029-403-017', '027-431-001', '029-422-029', '027-471-006', '027-371-012', '027-531-008', '027-431-003', '027-531-013', '027-075-003', '027-531-009', '027-471-008', '029-402-023', '027-053-012', '029-415-004', '029-402-024', '027-531-010', '027-144-022', '027-075-016', '027-531-014', '027-471-007', '027-531-011', '027-371-011', '027-531-016', '027-571-008', '027-531-018', '029-421-011', '027-075-002', '027-531-012', '027-571-009', '027-471-003', '027-053-013', '027-134-018', '027-571-010', '027-132-011', '027-571-007', '027-371-010', '027-144-020', '027-531-015', '029-405-006', '027-075-015', '027-571-006', '027-531-017', '029-422-031', '027-471-002', '027-131-009', '027-571-015', '027-571-016', '027-144-014', '027-371-009', '027-134-017', '029-415-003', '027-132-022', '027-471-004', '027-054-004', '027-075-020', '027-571-011', '027-571-012', '027-134-021', '027-471-001', '027-571-014', '027-571-013', '027-075-019', '029-405-002', '027-144-023', '027-134-016', '027-132-024', '027-144-018', '027-075-014', '027-054-005', '027-134-020', '029-415-014', '029-415-015', '029-415-013', '029-415-012', '018-090-027', '027-131-023', '029-415-011', '029-415-010', '029-404-005', '029-415-008', '029-404-004', '029-415-009', '027-621-002', '027-144-024', '029-404-006', '029-404-003', '029-404-002', '027-134-025', '027-075-013', '027-054-015', '029-405-009', '027-131-022', '027-020-010', '027-144-008', '027-132-023', '027-370-004', '027-134-015', '027-020-015', '027-131-019', '027-075-012', '027-134-004', '027-131-007', '027-121-003', '027-144-009', '027-090-017', '027-134-027', '027-621-025', '027-131-018', '027-621-024', '027-136-008', '027-075-011', '027-134-003', '027-621-023', '027-054-013', '027-131-006', '027-145-001', '027-121-008', '027-621-012', '027-131-017', '027-621-011', '018-090-030', '027-134-026', '027-136-007', '027-621-010', '027-134-028', '027-621-009', '027-131-005', '027-621-008', '027-621-007', '027-621-006', '027-621-005', '027-111-011', '027-621-004', '027-136-020', '027-621-003', '028-291-001', '027-054-010', '027-621-001', '027-621-037', '027-131-004', '027-621-013', '027-621-026', '027-621-014', '027-621-015', '027-621-016', '027-134-013', '027-621-017', '027-131-015', '027-621-036', '027-621-018', '027-145-003', '027-145-002', '027-621-019', '028-311-033', '028-041-003', '028-041-002', '028-041-001', '027-621-020', '027-131-003', '027-621-021', '027-621-027', '027-621-022', '027-136-019', '027-621-035', '027-136-017', '027-121-018', '027-122-003', '027-131-014', '027-621-028', '027-122-020', '027-133-012', '027-020-009', '028-042-018', '028-011-064', '027-621-034', '027-131-002', '028-041-004', '028-311-018', '027-136-016', '027-121-017', '028-311-017', '027-101-017', '027-134-011', '028-311-031', '028-311-034', '027-131-021', '027-621-033', '018-090-026', '027-145-007', '027-136-004', '027-621-029', '027-133-011', '028-042-017', '027-090-016', '028-311-019', '027-131-024', '027-621-032', '027-136-015', '027-121-016', '027-134-010', '027-131-020', '027-122-019', '027-145-004', '027-621-030', '027-136-003', '027-101-027', '027-621-031', '028-311-030', '027-133-010', '028-311-035', '028-311-016', '027-101-002', '028-042-005', '027-136-014', '028-311-015', '027-131-012', '027-122-018', '027-121-015', '028-311-013', '027-145-008', '028-311-020', '027-101-024', '027-136-002', '028-311-014', '027-133-009', '028-041-005', '027-131-025', '027-122-009', '028-042-024', '027-136-013', '028-311-029', '028-041-006', '027-133-020', '027-145-005', '027-350-001', '027-131-027', '027-112-025', '027-122-017', '028-311-012', '027-136-023', '027-133-008', '028-311-021', '028-311-036', '027-122-010', '027-136-012', '027-133-019', '028-301-060', '028-042-004', '028-301-059', '027-131-026', '027-122-016', '018-090-023', '027-136-024', '027-152-009', '027-133-007', '027-122-011', '028-311-011', '028-311-028', '027-123-001', '027-145-006', '027-136-027', '028-042-023', '027-133-018', '028-311-022', '027-122-015', '027-152-008', '027-136-030', '027-133-006', '027-350-020', '027-101-028', '027-136-028', '028-311-037', '028-311-051', '027-131-011', '027-145-014', '027-122-014', '028-311-050', '027-136-029', '027-152-019', '028-042-021', '027-133-005', '027-123-023', '018-090-013', '028-311-052', '028-311-010', '028-311-049', '027-122-012', '027-136-025', '027-123-003', '027-133-017', '028-311-023', '027-010-013', '028-311-027', '028-311-048', '028-042-022', '027-112-012', '027-122-006', '027-112-009', '027-112-005', '027-112-026', '027-112-027', '027-152-020', '027-145-012', '028-042-020', '027-371-003', '027-133-004', '028-301-061', '028-043-006', '027-112-008', '027-136-026', '027-152-018', '027-123-004', '027-112-010', '027-350-006', '028-311-053', '028-311-038', '028-311-047', '027-010-002', '027-152-006', '027-135-016', '028-042-019', '027-133-003', '027-101-014', '027-111-008', '028-042-002', '028-311-009', '028-311-024', '028-311-026', '027-136-032', '027-152-017', '027-123-005', '027-133-026', '027-145-015', '027-152-005', '027-135-015', '027-350-016', '027-133-027', '027-371-002', '028-042-010', '028-311-054', '028-043-005', '027-152-016', '027-123-006', '027-112-011', '028-311-039', '028-301-062', '027-101-003', '027-136-034', '028-311-025', '027-350-015', '027-111-007', '028-042-001', '027-152-015', '028-311-046', '027-123-007', '027-133-023', '027-152-004', '027-135-031', '028-311-008', '028-311-055', '027-152-014', '027-135-017', '028-042-011', '027-350-024', '028-154-003', '027-133-024', '027-123-008', '028-043-004', '027-135-030', '027-152-003', '028-311-040', '027-123-015', '027-101-004', '027-101-013', '027-135-018', '027-152-013', '028-311-045', '027-133-025', '027-123-009', '027-135-023', '028-311-056', '028-311-007', '027-154-033', '028-152-046', '027-113-016', '027-113-017', '027-113-038', '027-113-014', '027-113-015', '027-113-006', '027-113-035', '027-123-016', '027-113-037', '027-124-001', '027-113-036', '028-301-035', '027-152-023', '027-010-012', '027-090-020', '027-113-029', '028-042-012', '027-123-010', '027-152-001', '027-113-001', '027-135-022', '027-154-035', '028-043-003', '027-123-017', '027-101-005', '027-135-020', '027-152-024', '028-301-036', '027-123-011', '028-311-057', '028-311-044', '027-135-003', '027-154-007', '027-124-002', '028-311-006', '027-010-014', '028-153-024', '027-123-018', '027-124-003', '027-135-021', '027-152-022', '028-301-037', '027-090-022', '027-123-012', '027-113-002', '027-154-006', '027-135-029', '028-301-034', '027-350-023', '027-123-019', '027-101-022', '028-043-002', '027-152-021', '027-135-025', '027-154-012', '027-151-009', '028-241-001', '028-301-038', '027-124-008', '028-311-043', '027-135-028', '028-311-005', '027-113-019', '027-113-023', '027-113-024', '027-113-025', '027-113-020', '027-113-034', '027-113-033', '027-113-032', '027-123-020', '027-152-010', '027-154-034', '027-113-027', '027-113-009', '027-151-008', '027-154-019', '027-135-026', '027-090-011', '027-361-004', '027-113-007', '027-135-012', '027-090-006', '028-301-033', '028-153-005', '027-123-021', '027-135-027', '027-101-029', '027-111-003', '027-124-004', '027-154-015', '028-043-001', '028-301-039', '027-151-015', '027-124-018', '028-152-049', '028-154-002', '028-311-042', '027-170-015', '027-123-022', '028-311-004', '027-135-010', '027-154-016', '027-353-001', '027-151-007', '027-135-002', '028-251-001', '027-154-031', '027-124-019', '027-113-008', '027-151-025', '027-090-023', '027-101-030', '027-101-032', '027-154-017', '028-251-002', '027-111-004', '027-151-006', '027-353-003', '018-090-057', '027-154-030', '028-311-041', '028-301-040', '028-251-003', '027-124-010', '028-311-003', '027-090-009', '027-154-018', '028-301-032', '028-251-004', '027-353-004', '027-124-005', '028-301-041', '027-154-013', '027-124-011', '027-125-026', '027-151-012', '028-251-005', '027-101-009', '027-353-005', '027-101-011', '027-111-005', '027-180-022', '027-124-021', '028-301-013', '027-154-028', '028-152-051', '028-301-042', '028-311-002', '028-251-006', '027-180-019', '028-252-001', '027-124-012', '028-301-031', '027-151-013', '018-110-011', '027-135-006', '027-180-023', '027-114-007', '027-114-008', '027-114-021', '027-114-022', '027-114-009', '027-114-015', '027-114-023', '027-114-006', '028-292-002', '027-156-006', '027-154-029', '027-124-020', '027-353-002', '028-252-002', '027-151-026', '027-114-012', '027-114-002', '027-125-025', '027-114-020', '027-124-013', '027-151-014', '027-125-024', '028-252-003', '028-301-014', '027-101-010', '027-180-013', '027-114-019', '028-301-043', '027-357-001', '027-124-007', '028-311-001', '028-252-004', '028-301-030', '027-124-014', '027-125-023', '027-154-022', '028-252-005', '028-153-010', '027-153-007', '028-262-001', '027-125-016', '027-151-018', '028-301-015', '028-252-006', '027-154-021', '027-124-015', '028-262-002', '027-125-028', '027-352-001', '027-111-010', '028-301-012', '028-252-007', '028-301-044', '028-301-029', '027-156-005', '028-262-003', '027-151-023', '028-252-008', '027-154-009', '027-124-016', '027-153-030', '027-357-002', '028-301-016', '027-125-017', '028-262-004', '027-354-019', '027-361-005', '027-354-006', '027-354-017', '027-203-010', '027-203-011', '027-354-018', '027-354-020', '027-203-016', '027-203-017', '027-354-001', '027-151-024', '028-301-011', '028-262-005', '027-124-017', '027-125-004', '027-203-009', '028-163-004', '028-301-045', '027-153-029', '027-203-008', '027-156-016', '027-125-018', '018-090-031', '028-262-006', '027-364-025', '028-301-028', '027-352-002', '028-153-014', '027-153-018', '028-253-001', '026-043-015', '027-125-005', '028-152-043', '027-153-022', '027-156-003', '027-201-010', '027-125-032', '026-043-009', '028-301-010', '026-043-017', '028-253-002', '027-153-017', '028-151-001', '026-043-016', '027-357-003', '027-125-006', '027-153-021', '027-156-002', '028-253-003', '028-153-012', '027-125-033', '028-301-018', '028-163-003', '027-311-009', '026-042-006', '027-153-016', '026-280-007', '028-152-009', '028-301-027', '028-301-046', '026-042-008', '028-253-004', '027-352-003', '027-201-011', '026-042-009', '027-125-007', '026-042-010', '028-153-013', '026-042-003', '028-301-009', '027-125-011', '026-042-002', '027-170-009', '027-153-027', '027-362-007', '028-253-005', '027-362-009', '027-651-001', '027-156-010', '027-362-003', '028-271-001', '027-156-013', '027-153-035', '027-125-021', '028-301-019', '027-354-009', '028-253-006', '028-090-047', '028-271-002', '027-354-011', '027-125-012', '027-354-012', '027-311-023', '028-301-026', '027-354-013', '027-354-014', '027-354-016', '027-354-015', '028-272-001', '027-156-009', '027-357-004', '027-201-005', '027-651-002', '028-301-047', '027-153-005', '028-301-008', '028-253-007', '027-125-022', '028-271-003', '028-272-002', '027-352-004', '026-280-003', '027-203-002', '027-125-030', '027-153-028', '026-041-015', '027-203-013', '027-311-022', '027-156-018', '027-205-008', '027-202-005', '027-205-003', '028-180-034', '028-253-008', '028-301-020', '028-271-004', '026-041-020', '027-202-001', '027-311-024', '027-155-023', '026-041-016', '026-041-019', '028-272-003', '027-180-017', '027-153-004', '028-261-001', '026-041-018', '026-280-006', '028-162-033', '028-151-002', '026-041-014', '028-301-007', '026-280-002', '026-041-013', '027-153-012', '028-301-048', '027-125-029', '028-272-004', '028-271-005', '026-033-010', '027-354-010', '028-261-002', '027-155-019', '027-153-001', '026-033-015', '026-280-005', '028-152-010', '028-152-025', '028-301-021', '028-272-005', '028-271-006', '026-033-011', '026-280-001', '027-153-023', '027-156-007', '028-301-024', '027-357-013', '028-261-003', '027-311-025', '027-125-014', '027-362-010', '027-364-022', '026-033-012', '028-301-049', '027-352-005', '027-155-021', '028-272-006', '027-311-005', '026-033-013', '026-280-004', '027-153-032', '028-261-004', '027-201-002', '028-272-007', '027-125-015', '027-311-017', '027-153-003', '028-162-034', '027-155-018', '028-163-009', '027-205-001', '027-205-002', '027-203-003', '028-261-005', '028-301-022', '028-152-013', '028-272-008', '028-301-050', '027-155-022', '027-203-006', '027-153-031', '027-202-004', '027-170-012', '027-311-004', '028-152-047', '027-311-027', '028-261-006', '027-352-006', '028-152-024', '026-041-008', '027-155-014', '028-301-023', '028-121-021', '027-205-004', '027-155-026', '026-032-017', '027-162-011', '027-311-028', '027-364-023', '027-355-007', '026-032-016', '028-301-051', '027-355-006', '027-355-005', '027-355-004', '027-355-001', '027-355-003', '027-355-002', '028-273-005', '027-311-003', '028-152-011', '028-273-006', '028-162-014', '027-312-030', '026-032-015', '026-046-018', '026-033-009', '028-273-003', '028-273-004', '027-155-013', '028-301-005', '026-046-014', '026-032-014', '028-273-001', '028-273-002', '026-046-011', '026-032-018', '026-032-013', '026-046-016', '026-022-014', '027-311-015', '026-032-012', '027-201-006', '026-046-015', '027-311-026', '027-155-012', '026-046-001', '028-152-048', '027-205-013', '027-205-010', '028-301-052', '026-032-011', '027-352-007', '027-155-025', '026-022-008', '026-032-004', '026-045-006', '027-311-029', '026-022-010', '026-045-005', '027-357-007', '026-031-008', '027-155-011', '026-022-007', '026-045-004', '027-355-008', '027-202-012', '028-152-023', '028-301-004', '027-162-006', '026-045-011', '026-022-009', '026-045-003', '027-155-002', '027-311-006', '026-045-010', '026-031-007', '027-364-017', '027-204-035', '027-204-007', '027-204-036', '027-205-014', '027-204-034', '028-121-019', '027-204-014', '026-031-009', '027-312-016', '026-031-010', '028-121-031', '027-312-029', '027-161-011', '028-301-003', '026-044-009', '027-352-008', '026-046-008', '026-044-018', '027-381-001', '027-363-001', '027-363-012', '027-363-011', '026-044-017', '027-363-008', '027-363-009', '027-363-010', '026-046-007', '027-363-007', '027-363-006', '027-363-005', '027-660-005', '027-155-027', '027-363-004', '027-363-002', '027-162-012', '027-363-003', '028-301-053', '026-031-005', '028-121-018', '026-044-019', '026-046-009', '026-021-005', '027-161-018', '026-044-015', '027-201-003', '028-152-022', '026-046-010', '027-660-004', '027-660-003', '027-357-008', '026-046-005', '026-021-004', '027-205-005', '027-660-002', '026-044-013', '027-205-011', '028-301-002', '027-355-011', '027-355-015', '027-355-016', '027-355-012', '027-355-013', '027-355-014', '026-271-028', '027-155-029', '026-044-011', '027-660-001', '026-046-004', '028-161-001', '027-312-014', '026-036-004', '026-021-003', '027-161-021', '028-121-017', '028-301-054', '026-271-015', '026-036-009', '026-021-009', '026-045-007', '026-271-017', '026-271-016', '026-271-018', '027-364-016', '026-045-013', '026-271-019', '026-036-010', '026-271-020', '027-202-009', '027-202-010', '027-202-008', '026-271-021', '026-036-002', '027-352-009', '026-045-022', '028-121-030', '027-312-013', '028-301-001', '027-161-017', '027-381-002', '026-271-022', '026-271-023', '027-204-006', '026-045-021', '027-204-027', '028-152-021', '026-271-024', '026-036-012', '028-121-016', '026-271-025', '026-045-020', '026-036-011', '027-161-024', '028-301-055', '027-155-030', '026-271-027', '026-271-026', '026-045-008', '026-035-017', '027-161-019', '027-312-012', '026-271-014', '027-205-006', '027-357-009', '026-044-010', '026-035-015', '026-035-004', '027-161-025', '026-044-007', '027-155-004', '028-301-056', '027-201-008', '027-161-020', '026-271-013', '026-044-008', '026-035-003', '027-204-028', '028-121-014', '027-312-011', '026-044-005', '027-352-010', '026-035-002', '027-312-021', '027-313-009', '026-044-004', '026-035-001', '028-152-020', '027-161-027', '026-035-018', '027-364-015', '026-271-012', '026-034-015', '028-121-032', '028-121-029', '027-161-022', '028-301-057', '026-040-005', '026-034-014', '027-312-010', '026-034-004', '028-161-011', '027-381-004', '027-312-022', '028-122-003', '026-036-008', '026-034-003', '027-161-026', '027-155-028', '026-271-009', '026-271-011', '026-036-007', '028-161-010', '026-271-006', '026-034-002', '026-271-002', '027-204-018', '026-271-007', '026-271-008', '026-034-011', '026-271-001', '027-357-010', '027-364-014', '026-271-004', '027-364-009', '027-364-026', '026-034-013', '026-271-005', '027-364-012', '027-364-007', '027-364-008', '027-364-013', '027-241-016', '027-241-015', '027-241-017', '027-364-003', '027-364-004', '027-364-002', '027-364-005', '027-364-006', '027-170-011', '027-364-001', '027-312-009', '028-301-058', '027-381-006', '026-036-005', '028-180-037', '026-034-012', '026-271-003', '026-036-006', '026-271-010', '027-312-023', '027-161-031', '027-381-005', '028-122-020', '028-122-004', '028-152-040', '027-356-001', '026-035-020', '027-356-002', '027-356-003', '027-356-007', '027-356-005', '027-356-004', '027-356-006', '026-025-006', '026-048-003', '027-204-001', '027-204-032', '027-204-033', '027-312-008', '027-204-031', '026-025-005', '027-204-025', '027-204-026', '026-035-019', '027-204-016', '027-204-011', '027-204-020', '027-204-021', '027-204-022', '027-201-007', '027-313-008', '026-048-010', '027-312-024', '026-025-004', '026-035-009', '027-161-032', '028-121-024', '026-048-018', '028-111-009', '027-321-015', '026-025-003', '026-035-008', '026-048-019', '028-180-040', '026-035-011', '026-025-020', '028-121-012', '026-048-013', '027-161-030', '027-163-004', '026-035-010', '026-025-015', '026-048-012', '026-035-006', '026-025-018', '026-023-021', '027-161-028', '028-171-009', '027-312-025', '027-313-007', '027-357-011', '026-025-019', '028-180-033', '026-034-016', '026-035-021', '026-047-005', '026-023-022', '028-171-005', '028-122-006', '026-047-025', '026-023-023', '026-023-008', '028-180-020', '027-356-008', '026-047-024', '027-312-005', '026-034-018', '026-047-003', '027-163-022', '026-034-017', '026-047-002', '026-034-009', '026-023-007', '027-241-007', '026-047-012', '027-241-018', '026-023-011', '026-034-006', '026-047-011', '026-024-014', '026-039-013', '026-023-010', '028-122-007', '028-111-006', '026-039-014', '026-040-006', '027-221-009', '028-180-008', '027-221-045', '027-221-046', '027-221-037', '027-221-039', '027-221-043', '026-025-009', '027-221-040', '027-221-011', '027-312-004', '028-121-011', '026-039-008', '027-221-018', '027-221-017', '027-221-012', '027-221-016', '027-221-015', '027-221-013', '027-221-014', '027-163-021', '026-039-015', '026-025-008', '027-357-012', '028-121-025', '028-171-010', '028-111-005', '026-039-009', '026-048-014', '027-321-013', '026-025-012', '028-111-008', '027-290-004', '026-023-005', '026-048-015', '026-039-007', '027-290-007', '028-162-037', '026-025-013', '027-290-009', '026-048-017', '026-039-001', '027-290-008', '026-025-021', '026-048-016', '026-048-021', '028-171-011', '026-048-020', '026-025-016', '026-038-023', '028-180-045', '027-163-034', '026-025-017', '026-047-008', '027-344-001', '025-701-001', '026-024-015', '026-038-004', '026-023-017', '025-702-001', '027-163-012', '027-341-001', '030-401-015', '026-038-003', '025-250-006', '027-341-002', '027-321-014', '027-313-014', '027-341-003', '027-341-004', '027-221-038', '027-341-005', '027-341-006', '026-038-018', '027-341-007', '027-341-008', '025-701-005', '026-047-019', '026-038-019', '028-161-014', '026-047-022', '025-702-002', '028-121-010', '026-047-023', '027-163-033', '026-023-002', '027-241-008', '026-047-026', '027-344-045', '027-241-003', '028-111-004', '026-024-016', '026-023-016', '026-037-004', '026-038-024', '027-322-019', '026-039-012', '025-701-007', '027-290-010', '028-161-017', '025-702-004', '026-037-008', '027-163-013', '026-083-007', '028-111-007', '026-039-016', '027-344-002', '026-037-009', '026-083-003', '026-039-017', '028-161-018', '026-037-014', '028-081-005', '026-039-018', '027-163-038', '027-321-017', '027-224-011', '026-024-013', '026-037-013', '027-224-010', '028-123-018', '026-083-009', '026-039-021', '028-122-009', '026-037-001', '027-341-009', '027-222-002', '027-163-003', '026-023-015', '026-039-020', '027-224-001', '027-222-001', '027-222-018', '027-223-003', '028-121-009', '026-038-021', '025-702-006', '026-039-005', '027-223-001', '026-026-019', '027-223-004', '025-701-009', '027-221-008', '027-222-017', '028-162-032', '026-026-018', '028-123-017', '028-320-003', '026-038-013', '027-344-052', '026-082-004', '027-223-005', '027-221-023', '027-344-003', '026-038-012', '026-082-011', '026-023-014', '026-038-011', '027-163-035', '027-223-018', '027-223-006', '027-222-016', '026-082-012', '028-171-012', '027-321-016', '025-702-008', '027-223-021', '026-026-022', '026-038-008', '026-082-002', '027-163-025', '025-701-011', '026-038-006', '027-341-018', '026-026-021', '026-039-019', '027-341-017', '028-070-013', '027-341-016', '028-081-013', '028-123-016', '027-341-015', '026-026-020', '027-313-010', '027-341-014', '027-341-013', '027-341-012', '028-122-010', '028-121-008', '027-341-010', '026-026-001', '027-241-024', '026-037-007', '027-241-028', '026-038-022', '027-241-031', '028-171-017', '026-024-009', '026-037-011', '026-081-011', '027-163-016', '030-401-014', '026-083-005', '026-037-012', '026-081-004', '028-180-038', '027-322-020', '026-024-004', '025-702-010', '027-344-004', '025-701-002', '027-321-001', '026-081-003', '027-321-004', '027-221-030', '026-081-002', '028-320-002', '026-083-010', '027-223-020', '027-221-007', '026-038-007', '026-081-019', '027-163-026', '025-701-013', '026-023-020', '027-223-002', '026-039-022', '026-081-018', '028-090-097', '028-070-018', '026-073-008', '028-121-007', '025-702-012', '027-224-002', '027-224-018', '026-073-021', '028-081-004', '026-038-010', '028-081-014', '026-023-019', '027-222-003', '027-341-011', '027-222-015', '026-024-011', '026-082-010', '026-073-022', '025-701-015', '027-344-059', '028-171-029', '026-082-009', '026-073-014', '026-026-014', '028-070-026', '028-123-003', '028-320-012', '027-321-003', '026-082-008', '026-073-025', '028-123-033', '027-223-015', '027-241-029', '026-026-015', '026-082-007', '027-223-014', '026-073-024', '027-221-024', '026-026-013', '027-221-006', '026-082-006', '026-073-016', '027-223-013', '027-411-001', '026-026-012', '026-023-018', '027-344-005', '027-223-008', '027-223-011', '027-223-010', '027-223-017', '027-223-012', '028-123-004', '027-223-016', '025-702-014', '026-026-016', '026-072-024', '028-121-006', '027-224-003', '027-224-017', '026-072-003', '028-171-028', '027-241-030', '026-081-017', '028-123-032', '027-222-004', '027-222-014', '026-072-006', '026-024-012', '028-123-005', '027-321-002', '028-320-004', '026-081-010', '026-072-027', '027-411-008', '025-701-017', '028-070-021', '025-702-016', '026-081-013', '026-072-026', '026-081-012', '026-072-001', '027-221-025', '028-081-009', '026-081-008', '027-221-005', '027-241-023', '026-081-014', '027-163-027', '026-081-015', '028-070-024', '027-411-009', '028-081-002', '026-071-004', '027-223-009', '026-072-025', '027-344-006', '025-701-019', '026-026-017', '026-073-027', '027-342-001', '026-071-006', '028-320-001', '027-224-013', '026-073-026', '026-071-009', '027-342-002', '028-171-013', '027-342-003', '026-073-019', '027-342-004', '026-071-008', '025-251-008', '027-342-020', '028-123-013', '027-222-005', '027-222-013', '028-081-007', '025-702-018', '026-024-010', '026-073-018', '027-342-007', '027-411-010', '026-071-007', '027-342-008', '027-323-011', '026-073-020', '026-061-008', '026-071-002', '028-121-028', '026-073-023', '026-071-001', '027-221-032', '027-322-003', '027-344-041', '027-163-030', '028-070-008', '027-221-034', '028-123-002', '026-086-005', '025-701-003', '027-411-007', '028-070-007', '026-061-007', '025-702-020', '027-225-001', '030-401-040', '026-061-006', '026-072-021', '026-085-022', '028-121-034', '026-063-002', '025-701-021', '028-171-014', '027-224-004', '027-224-012', '026-085-021', '030-401-012', '028-070-020', '026-072-017', '026-085-008', '027-411-006', '026-063-008', '027-222-019', '027-241-025', '028-070-023', '026-072-016', '026-085-007', '028-090-093', '027-411-011', '026-072-015', '027-225-003', '026-085-003', '028-171-026', '027-322-002', '027-331-011', '028-123-007', '026-061-003', '026-063-018', '026-072-014', '026-085-002', '026-061-009', '025-701-023', '027-411-005', '026-063-017', '026-072-013', '028-123-001', '026-085-019', '027-225-004', '025-702-022', '027-342-009', '026-072-012', '027-344-007', '026-085-020', '026-063-006', '027-411-012', '025-251-007', '027-221-035', '027-225-009', '027-225-008', '027-225-005', '027-225-010', '026-063-005', '027-225-007', '026-071-016', '028-090-011', '026-084-004', '026-072-022', '027-225-023', '028-171-015', '027-225-025', '027-225-024', '027-344-040', '028-081-006', '026-071-017', '026-084-006', '026-062-018', '027-224-005', '027-221-033', '027-224-008', '026-071-010', '025-702-024', '027-411-013', '028-121-033', '030-403-011', '026-071-021', '028-070-014', '027-222-020', '027-222-012', '026-062-017', '026-084-014', '026-062-002', '026-071-020', '026-084-015', '028-051-029', '026-062-016', '025-252-010', '028-280-013', '026-061-010', '026-084-001', '025-701-025', '027-411-004', '026-062-015', '027-342-017', '027-342-018', '026-086-010', '027-342-016', '026-062-014', '028-063-007', '028-070-019', '026-076-011', '027-342-015', '026-086-002', '027-342-014', '027-323-013', '027-342-013', '026-076-010', '027-342-019', '027-344-056', '028-171-016', '027-342-012', '027-342-010', '026-072-004', '028-123-008', '026-085-018', '027-225-026', '026-076-009', '026-063-009', '025-251-006', '025-702-026', '027-225-011', '027-323-005', '026-085-015', '027-224-014', '028-070-010', '026-076-017', '027-224-015', '027-224-016', '027-224-007', '025-701-027', '026-063-010', '028-051-030', '028-082-009', '026-085-016', '026-076-016', '027-411-003', '027-241-013', '027-241-021', '027-241-014', '027-241-022', '026-063-011', '025-252-012', '027-241-026', '027-222-011', '027-222-007', '026-085-017', '026-076-015', '026-063-012', '026-085-010', '026-076-008', '027-344-039', '026-063-013', '026-085-012', '028-280-011', '028-063-006', '026-061-011', '026-076-007', '025-702-028', '026-063-014', '026-085-013', '027-411-002', '028-082-008', '018-180-004', '027-221-028', '026-085-014', '026-063-015', '026-075-013', '026-075-012', '026-063-016', '027-344-058', '025-701-004', '026-084-010', '027-331-010', '028-123-010', '026-075-011', '028-123-009', '027-323-004', '026-084-012', '026-062-010', '026-075-010', '030-401-037', '027-225-018', '027-225-017', '025-252-013', '027-225-027', '027-225-013', '026-084-013', '027-225-021', '027-225-019', '027-225-016', '027-225-015', '027-225-028', '027-225-022', '027-222-008', '027-222-010', '025-701-029', '026-062-012', '027-342-011', '026-075-009', '026-084-011', '025-251-005', '028-063-005', '026-062-013', '026-075-005', '026-084-009', '025-702-030', '026-075-008', '026-062-006', '026-084-005', '028-051-004', '026-075-007', '026-062-009', '026-061-012', '025-701-031', '027-322-018', '026-076-012', '027-344-038', '027-221-001', '028-082-006', '026-076-005', '025-702-032', '027-221-029', '026-074-018', '028-320-010', '027-421-023', '026-076-003', '026-074-020', '026-089-007', '028-081-008', '027-344-010', '027-225-012', '027-331-020', '027-225-020', '026-074-019', '026-089-010', '028-063-004', '027-222-009', '027-251-002', '027-251-005', '028-051-005', '026-074-012', '025-252-003', '026-074-011', '026-076-013', '025-251-004', '028-082-005', '026-089-009', '026-074-010', '026-062-008', '026-061-013', '026-076-018', '026-076-014', '025-702-034', '025-701-033', '026-075-021', '026-065-011', '026-088-012', '027-421-001', '026-088-013', '028-171-025', '030-402-001', '027-226-001', '027-343-001', '028-082-004', '026-075-019', '026-088-009', '027-343-004', '027-343-003', '028-052-016', '026-076-019', '028-063-003', '027-343-002', '026-088-010', '027-343-005', '026-065-008', '028-320-009', '026-088-007', '027-343-006', '027-344-037', '025-702-036', '027-344-011', '025-701-035', '027-343-007', '026-088-017', '026-075-016', '027-343-008', '027-343-009', '027-343-010', '030-401-020', '027-421-002', '026-075-024', '026-088-018', '026-061-014', '026-088-019', '028-082-003', '025-252-004', '026-065-004', '025-251-003', '028-052-015', '026-087-016', '026-075-022', '028-051-007', '026-064-008', '027-233-025', '027-233-026', '027-233-004', '027-233-005', '027-233-001', '027-233-006', '026-087-017', '026-074-009', '027-233-007', '026-064-007', '027-232-016', '027-233-008', '027-233-009', '027-232-015', '027-232-001', '027-421-003', '026-087-015', '026-074-008', '027-421-006', '026-064-004', '028-061-027', '025-702-038', '028-082-002', '026-087-004', '026-074-007', '027-231-001', '026-064-006', '025-253-012', '026-087-003', '026-074-015', '027-251-006', '026-064-019', '028-320-008', '026-087-002', '026-074-016', '028-052-003', '026-075-025', '026-064-018', '027-344-012', '028-063-002', '026-087-001', '026-074-003', '026-061-015', '026-064-002', '027-421-007', '026-079-020', '023-741-027', '031-077-012', '023-652-001', '023-645-023', '031-112-015', '031-063-010', '031-115-019', '031-203-007', '023-653-004', '031-153-005', '031-156-008', '031-151-010', '031-192-003', '023-731-007', '023-904-009', '023-111-033', '031-156-011', '023-111-014', '031-205-002', '031-203-014', '023-741-005', '031-153-016', '031-077-001', '023-645-011', '023-732-031', '023-644-017', '031-192-015', '031-063-006', '023-732-016', '023-646-004', '031-113-016', '031-115-005', '031-232-004', '023-733-029', '023-665-012', '031-077-013', '023-733-012', '023-661-037', '023-261-005', '031-203-008', '031-115-008', '031-153-006', '023-645-022', '031-142-001', '031-112-016', '031-232-010', '023-904-011', '031-063-011', '031-205-003', '023-653-005', '023-741-028', '023-911-005', '031-192-004', '031-151-011', '031-205-017', '031-153-015', '031-203-020', '023-652-002', '031-063-005', '023-646-005', '031-077-003', '023-645-012', '023-111-032', '023-653-025', '023-904-008', '031-115-004', '023-731-008', '031-192-016', '031-155-001', '031-232-005', '023-911-009', '031-077-014', '031-063-002', '023-741-004', '031-112-012', '031-115-009', '031-153-007', '023-732-032', '023-665-013', '023-732-015', '031-205-004', '031-142-002', '031-203-009', '023-645-021', '031-112-018', '031-192-005', '031-063-012', '023-903-011', '031-153-014', '023-733-011', '023-646-006', '031-156-010', '023-911-003', '023-131-008', '023-653-006', '031-063-001', '023-911-008', '023-653-024', '023-661-038', '023-652-003', '031-192-014', '031-205-018', '023-561-021', '023-741-029', '023-904-012', '031-115-001', '023-645-013', '023-261-008', '031-077-015', '031-155-002', '023-911-006', '031-232-006', '031-112-011', '031-115-010', '023-131-009', '031-153-008', '031-063-003', '031-232-009', '031-142-003', '023-911-007', '031-205-023', '023-904-007', '031-192-006', '023-561-020', '031-203-010', '031-153-013', '031-142-017', '023-646-007', '023-731-009', '031-112-017', '023-645-020', '031-205-016', '023-131-010', '031-063-014', '023-741-003', '031-236-008', '031-192-013', '023-732-033', '031-155-003', '023-732-014', '023-653-007', '031-194-013', '023-903-012', '023-652-004', '023-911-004', '031-115-003', '023-131-011', '023-645-014', '023-903-010', '031-112-010', '031-232-007', '031-115-011', '031-155-017', '023-741-030', '031-142-004', '023-561-019', '031-153-009', '023-653-023', '023-661-039', '031-112-014', '023-904-013', '023-111-039', '031-192-007', '031-153-012', '023-646-008', '031-205-015', '031-231-019', '031-142-018', '031-192-012', '031-203-011', '031-063-013', '023-645-019', '031-155-004', '023-911-002', '023-821-027', '023-904-006', '023-561-018', '023-131-016', '031-194-014', '031-112-009', '023-271-016', '031-115-012', '023-821-028', '023-731-010', '031-142-005', '023-741-002', '031-236-009', '023-653-008', '023-821-026', '031-232-008', '023-645-015', '023-652-008', '031-061-014', '023-111-036', '023-902-013', '031-155-018', '031-205-007', '031-153-010', '023-732-013', '031-112-013', '023-821-025', '031-061-002', '023-646-009', '031-142-016', '023-903-013', '031-192-008', '023-903-009', '031-114-007', '031-205-014', '023-653-022', '031-192-011', '031-061-003', '031-231-002', '031-155-005', '023-561-017', '031-194-015', '023-821-029', '023-904-014', '023-645-018', '023-261-009', '023-661-040', '031-061-004', '023-271-012', '023-131-014', '023-911-001', '031-112-001', '023-821-024', '031-155-016', '031-194-011', '031-142-006', '023-763-007', '023-731-011', '031-061-015', '031-115-013', '031-112-006', '031-205-008', '031-142-015', '023-904-005', '023-652-009', '031-061-005', '031-153-011', '023-646-010', '031-231-003', '023-653-009', '031-290-039', '023-741-001', '031-205-013', '031-192-009', '031-236-010', '031-114-008', '023-654-001', '023-821-030', '031-155-006', '023-131-015', '023-561-016', '023-821-051', '031-061-006', '023-902-014', '031-234-003', '031-194-016', '023-261-010', '031-231-017', '023-271-023', '023-821-050', '023-653-021', '023-902-012', '023-903-014', '031-061-007', '031-155-015', '031-142-007', '023-903-008', '031-191-001', '023-645-017', '023-131-012', '031-194-012', '023-763-006', '031-061-008', '023-904-015', '031-112-005', '031-142-014', '023-111-030', '023-562-001', '031-205-009', '031-114-006', '023-131-022', '031-231-004', '031-205-012', '031-061-013', '023-821-031', '023-763-008', '023-131-013', '031-155-019', '023-561-015', '023-271-024', '031-192-010', '023-111-037', '023-821-049', '023-821-023', '023-562-002', '031-114-009', '023-653-010', '023-904-004', '031-061-009', '023-654-002', '031-231-018', '031-155-014', '031-142-019', '031-194-010', '023-271-013', '023-671-021', '023-131-020', '031-236-011', '023-821-032', '031-061-018', '023-653-020', '023-902-015', '031-142-013', '023-821-052', '023-902-011', '023-645-016', '031-231-005', '023-903-015', '023-271-017', '031-114-005', '023-821-022', '031-205-010', '023-904-016', '023-903-007', '023-131-019', '031-144-003', '023-763-005', '023-646-012', '023-562-003', '023-821-048', '031-234-002', '023-561-014', '031-231-016', '031-155-020', '023-821-033', '023-763-009', '023-653-011', '031-114-010', '031-155-013', '031-194-009', '023-131-017', '031-191-003', '023-661-043', '023-901-001', '031-196-017', '031-142-012', '023-671-020', '023-654-003', '031-112-020', '023-821-053', '023-562-004', '031-114-004', '023-904-003', '031-231-006', '023-653-019', '031-144-004', '031-236-012', '023-821-034', '023-821-021', '031-205-011', '023-561-013', '031-194-002', '031-114-011', '023-654-024', '031-231-015', '023-902-016', '023-902-010', '031-155-012', '023-904-017', '023-763-004', '031-233-017', '031-194-008', '031-144-021', '023-903-016', '031-191-004', '023-653-012', '023-903-006', '023-141-025', '023-821-035', '031-196-002', '023-562-005', '023-671-019', '031-231-007', '031-114-003', '031-191-018', '031-111-011', '031-144-005', '023-763-003', '023-531-001', '023-901-002', '023-654-004', '031-114-012', '031-231-014', '023-271-020', '023-904-002', '031-141-004', '023-653-018', '031-194-004', '023-561-012', '031-155-021', '023-821-054', '023-646-014', '031-194-007', '023-821-020', '023-141-036', '031-191-005', '031-236-013', '031-196-003', '023-654-023', '023-562-006', '023-821-055', '023-902-017', '031-191-016', '023-821-036', '023-671-018', '031-233-002', '031-231-008', '031-114-001', '023-902-009', '023-821-046', '031-196-015', '023-821-047', '031-144-006', '023-903-017', '023-903-005', '023-821-056', '031-114-013', '031-231-013', '023-763-002', '031-144-015', '031-141-014', '031-111-010', '031-191-006', '023-531-003', '023-654-005', '031-194-005', '031-146-001', '023-653-013', '023-646-015', '023-653-017', '023-904-001', '023-561-011', '031-196-004', '023-821-019', '023-562-007', '031-191-015', '023-654-022', '023-821-037', '023-671-017', '031-144-007', '023-531-004', '031-114-002', '031-193-001', '023-762-003', '031-231-009', '031-196-016', '031-236-014', '031-231-012', '031-114-014', '031-233-003', '031-141-015', '031-144-014', '023-902-018', '023-531-005', '031-233-005', '023-821-018', '023-763-001', '031-111-003', '023-902-008', '023-271-025', '031-191-007', '023-821-045', '023-903-018', '031-141-012', '031-194-006', '023-903-004', '031-146-002', '031-196-005', '023-531-006', '023-141-020', '023-141-026', '031-191-014', '023-655-030', '023-561-010', '023-672-002', '023-654-006', '023-653-016', '023-562-008', '031-144-008', '023-821-038', '023-646-016', '031-196-014', '031-193-015', '023-762-002', '023-531-007', '023-654-021', '031-114-015', '031-231-010', '031-144-013', '023-141-014', '031-141-019', '023-271-018', '031-233-006', '023-531-014', '031-146-003', '023-821-058', '023-131-005', '023-821-017', '023-141-022', '031-141-013', '023-762-004', '031-196-006', '031-111-004', '031-222-017', '031-191-013', '023-672-003', '023-821-057', '023-902-019', '031-196-013', '023-562-009', '023-902-007', '023-671-015', '023-903-019', '031-144-009', '031-193-016', '023-141-035', '023-531-013', '023-561-009', '023-271-019', '023-653-014', '023-655-001', '023-903-003', '023-654-007', '031-144-012', '031-141-018', '031-193-014', '031-231-011', '031-114-016', '031-233-007', '023-830-051', '023-821-039', '023-271-026', '031-233-015', '023-821-016', '031-235-002', '031-146-004', '023-821-044', '023-654-020', '031-141-011', '023-762-001', '023-531-012', '023-141-011', '031-196-007', '023-821-059', '031-196-012', '031-222-002', '023-672-004', '023-141-015', '031-146-018', '023-671-014', '031-193-017', '023-562-010', '031-144-010', '031-141-020', '023-686-001', '023-762-005', '023-141-006', '031-233-008', '023-903-020', '023-655-002', '023-902-020', '031-193-013', '031-233-014', '031-235-011', '023-821-015', '031-146-005', '023-531-010', '023-561-008', '023-902-006', '023-654-008', '023-821-043', '031-141-010', '023-151-022', '023-903-002', '031-222-003', '031-196-008', '031-143-001', '031-146-016', '023-141-027', '023-654-019', '031-196-011', '023-653-015', '023-672-005', '031-193-019', '031-222-015', '023-671-013', '023-821-060', '031-141-021', '023-901-006', '031-193-012', '023-141-028', '023-531-008', '023-271-015', '031-111-009', '031-144-011', '031-233-009', '023-562-011', '031-233-013', '023-771-010', '031-146-006', '023-821-040', '023-151-034', '031-141-009', '023-821-014', '023-132-023', '023-141-021', '031-222-004', '031-146-015', '023-654-009', '031-196-009', '031-143-002', '023-821-042', '023-655-003', '023-821-061', '023-771-008', '023-672-006', '031-193-020', '023-902-021', '023-902-005', '023-686-002', '031-182-001', '023-671-012', '023-561-007', '023-141-003', '031-222-016', '031-141-016', '023-654-018', '023-903-001', '031-193-011', '031-233-012', '023-821-013', '031-146-007', '031-141-008', '031-195-019', '023-271-022', '023-655-027', '031-233-010', '023-562-012', '031-222-005', '031-143-003', '031-146-014', '023-151-007', '023-132-021', '023-821-012', '025-342-001', '031-111-008', '031-193-018', '031-222-014', '023-672-007', '023-654-010', '023-671-011', '031-196-010', '031-143-015', '023-821-041', '023-561-006', '023-686-003', '031-182-002', '031-193-010', '023-655-004', '031-146-008', '031-235-005', '023-771-011', '023-761-010', '031-141-006', '031-195-002', '023-654-017', '031-020-027', '023-902-022', '023-902-004', '023-151-023', '031-222-006', '031-146-013', '025-331-014', '031-143-004', '023-681-014', '023-271-021', '023-151-016', '031-222-013', '031-182-003', '031-233-011', '023-821-011', '023-671-010', '031-261-022', '023-672-008', '023-562-020', '023-141-019', '031-143-016', '031-193-009', '031-224-015', '023-132-031', '031-182-017', '023-655-026', '031-020-017', '023-654-011', '023-561-005', '031-235-006', '031-195-003', '023-771-012', '031-146-009', '031-141-007', '023-771-007', '023-686-004', '023-151-017', '023-655-005', '031-195-017', '031-146-012', '031-222-007', '031-143-005', '023-132-032', '023-654-016', '023-902-023', '031-222-012', '031-182-004', '023-761-009', '031-143-014', '023-141-030', '031-261-021', '023-821-010', '023-902-003', '031-193-008', '031-235-007', '031-182-018', '023-151-014', '031-195-004', '031-224-002', '023-141-029', '023-141-031', '023-771-013', '031-146-010', '031-143-006', '023-681-013', '023-561-004', '031-222-008', '031-195-018', '031-222-011', '023-151-011', '025-342-021', '023-901-009', '031-182-005', '023-655-006', '031-261-020', '023-141-008', '031-143-013', '023-654-012', '023-686-005', '023-281-001', '023-654-015', '023-821-009', '023-672-010', '023-671-008', '031-020-041', '031-224-004', '031-193-006', '023-771-006', '031-145-001', '023-821-003', '023-562-021', '031-182-016', '023-902-024', '031-235-008', '023-761-008', '031-195-005', '023-772-034', '023-821-008', '031-143-007', '031-195-016', '023-902-002', '031-146-011', '023-771-014', '023-532-003', '023-221-028', '031-222-020', '031-224-003', '023-151-006', '031-182-006', '023-561-003', '023-681-012', '023-671-007', '031-224-005', '023-672-011', '023-132-033', '031-262-014', '031-182-015', '023-141-018', '031-193-007', '031-145-002', '031-235-012', '025-342-005', '023-655-007', '023-654-013', '031-184-019', '031-195-006', '023-821-002', '031-221-019', '023-901-010', '023-811-019', '023-686-006', '023-821-007', '023-132-004', '031-195-015', '023-761-007', '031-143-008', '023-772-033', '023-771-005', '023-151-003', '031-182-007', '031-143-017', '023-771-015', '023-221-031', '031-224-006', '023-671-006', '031-145-003', '031-222-019', '023-672-012', '023-562-016', '031-224-014', '031-182-014', '023-902-001', '025-331-013', '031-261-018', '031-262-012', '023-821-001', '023-151-035', '023-681-011', '023-821-006', '031-195-007', '031-184-002', '023-772-001', '031-145-015', '023-561-002', '023-221-044', '031-221-002', '023-654-014', '031-195-014', '023-655-008', '025-341-001', '023-532-008', '023-901-011', '031-182-008', '023-686-007', '023-772-032', '023-682-001', '031-224-007', '023-761-006', '031-145-004', '023-821-005', '023-671-005', '031-226-010', '031-224-013', '023-672-013', '023-771-016', '023-151-032', '031-182-013', '031-195-008', '023-532-007', '023-655-023', '031-221-003', '031-184-003', '023-142-001', '031-262-013', '023-771-004', '023-562-017', '031-145-016', '031-184-017', '031-195-013', '023-682-002', '031-221-017', '023-681-010', '023-811-018', '023-532-006', '023-561-001', '023-821-004', '023-655-009', '023-772-002', '025-341-002', '031-145-005', '031-224-008', '031-182-009', '023-151-020', '023-772-031', '023-132-034', '031-226-002', '031-224-012', '023-221-058', '023-672-014', '031-182-012', '023-151-033', '023-682-027', '023-771-017', '031-221-004', '031-184-004', '031-145-014', '023-686-008', '023-761-005', '031-195-012', '023-811-017', '031-262-015', '031-221-018', '031-184-018', '023-281-002', '023-562-018', '025-341-003', '023-811-009', '023-682-003', '031-145-006', '031-226-003', '031-224-011', '023-772-003', '023-671-003', '023-681-009', '023-771-003', '023-811-016', '023-655-010', '023-672-015', '031-182-010', '025-331-012', '023-571-020', '031-221-005', '031-184-005', '031-145-013', '023-151-030', '031-224-009', '023-771-018', '031-195-010', '031-221-016', '031-184-022', '023-132-037', '031-181-001', '031-262-016', '023-151-031', '023-811-008', '023-686-009', '023-682-026', '025-341-004', '023-811-007', '023-761-004', '023-655-037', '023-901-013', '031-145-007', '031-226-004', '023-773-033', '023-151-021', '023-672-016', '023-811-015', '023-671-002', '023-562-019', '023-772-004', '031-145-012', '031-221-006', '023-682-004', '031-182-011', '023-132-017', '031-184-006', '023-655-011', '023-772-029', '023-681-008', '031-221-015', '025-341-016', '023-771-019', '031-195-011', '031-184-021', '031-223-017', '023-811-014', '031-186-019', '031-181-002', '031-224-010', '023-132-019', '031-262-017', '023-781-008', '023-571-019', '031-226-005', '031-145-008', '023-761-003', '023-682-025', '023-671-001', '023-672-017', '023-773-032', '023-142-003', '023-686-010', '031-221-007', '023-132-014', '023-811-006', '031-145-011', '031-184-007', '023-132-038', '023-772-005', '031-221-014', '023-784-001', '023-772-028', '031-261-064', '031-184-014', '031-181-003', '023-655-012', '023-811-013', '031-262-009', '023-682-005', '023-221-053', '023-681-007', '023-771-020', '025-342-009', '031-186-002', '031-181-015', '031-226-006', '031-262-018', '031-223-002', '023-686-018', '025-331-018', '023-672-018', '025-331-017', '023-811-012', '031-221-008', '023-811-005', '031-145-009', '031-184-008', '023-571-018', '023-685-005', '023-773-031', '031-221-013', '023-682-024', '023-761-002', '031-184-013', '023-772-006', '023-132-036', '031-181-004', '023-686-011', '023-781-007', '031-020-036', '023-772-027', '031-223-004', '031-186-003', '025-341-005', '031-262-008', '023-655-013', '023-161-029', '023-771-021', '023-132-026', '031-186-017', '023-221-052', '031-226-007', '023-152-006', '023-682-006', '031-181-016', '031-262-019', '023-686-017', '023-151-009', '023-672-019', '023-811-004', '023-681-006', '023-572-002', '025-342-010', '023-811-011', '031-221-009', '023-784-002', '031-184-009', '031-145-010', '031-261-060', '031-221-012', '031-223-003', '023-132-018', '025-341-015', '031-184-012', '023-773-030', '031-181-005', '023-281-016', '023-571-017', '031-223-005', '023-772-007', '031-186-004', '031-020-030', '023-132-028', '023-682-023', '023-772-026', '031-181-014', '023-685-006', '023-761-001', '023-132-020', '023-811-010', '031-262-007', '031-226-008', '031-186-018', '023-221-035', '023-221-020', '023-771-022', '023-655-014', '031-262-020', '023-672-020', '023-811-003', '023-221-050', '025-341-006', '023-682-029', '031-221-010', '023-221-011', '031-184-010', '031-181-006', '023-784-003', '031-223-006', '023-773-029', '031-186-005', '023-142-022', '023-572-003', '031-223-016', '023-781-006', '023-772-008', '031-181-013', '031-226-009', '023-161-031', '031-186-016', '031-183-017', '025-342-011', '023-772-025', '023-571-016', '025-331-010', '031-262-006', '025-341-014', '023-221-046', '023-672-021', '023-771-023', '023-682-022', '023-761-013', '031-262-021', '031-261-063', '023-811-002', '023-655-015', '023-686-013', '023-281-017', '031-221-011', '031-223-007', '031-181-007', '031-184-011', '031-186-006', '023-221-023', '031-225-011', '031-223-015', '023-152-012', '023-784-004', '031-212-019', '023-773-028', '031-181-012', '023-681-016', '031-186-015', '023-685-003', '023-772-009', '023-321-020', '023-572-004', '023-132-027', '025-341-007', '023-772-037', '023-811-001', '031-262-005', '023-761-015', '023-687-001', '031-183-002', '023-655-018', '023-771-024', '023-571-015', '031-262-022', '023-142-015', '023-781-005', '023-682-021', '025-342-012', '031-223-008', '023-684-005', '023-221-045', '031-186-007', '025-341-013']

* Error updating 1319-30-615-002: Client tried to access password-protected page without proper authorization.
Updating Attributes Finished: 2023-05-13 02:52:16
There were issues with the following APNs
* ['1318-23-602-003', '1318-23-610-027', '1318-23-611-013', '1318-23-610-029', '1318-23-610-030', '1318-23-610-031', '1318-23-710-018', '1318-23-710-052', '1318-23-710-033', '1318-23-710-034', '1318-23-710-057', '1318-23-610-013', '1318-23-610-017', '1318-23-610-015', '1318-23-510-016', '1318-23-510-023', '1318-23-610-012', '1318-23-610-018', '1318-24-301-009', '1318-24-302-001', '1318-24-302-002', '1318-24-311-005', '1318-24-311-015', '1318-24-601-003', '1318-24-601-002', '1318-24-601-001', '1318-24-601-007', '1318-24-311-008', '1318-24-710-018', '1318-24-311-010', '1318-24-710-011', '1318-24-710-015', '1318-24-710-008', '1318-24-710-003', '1318-24-710-002', '1418-10-710-049', '1418-10-710-071', '1418-10-710-004', '1418-11-310-001', '1319-30-710-000', '1319-30-615-000', '1319-30-615-006', '1319-30-615-003', '1319-30-612-000', '1319-30-622-010', '1319-30-622-007', '1319-30-622-008', '1319-30-622-004', '1319-30-622-005', '1319-30-622-000', '1319-30-622-009', '1319-30-710-013', '1319-30-710-014', '1319-30-710-007', '1319-30-710-008', '1319-30-612-010', '1319-30-612-009', '1319-30-626-000', '1319-30-625-000', '1319-30-623-000', '1319-30-622-006', '1319-30-614-005', '1319-30-621-000', '1319-30-614-000', '1319-30-612-008', '1319-30-612-007', '1319-30-612-006', '1319-30-612-003', '1319-30-612-012', '1319-30-612-011', '1319-30-614-004', '1319-30-614-001', '1319-30-614-007', '1319-30-614-006', '1319-30-614-008', '1319-30-612-005', '1319-30-612-002', '1319-30-617-000', '1319-30-515-000', '1319-30-514-020', '1319-30-514-009', '1319-30-514-018', '1319-30-514-019', '1319-30-514-021', '1319-30-514-022', '1319-30-514-023', '1319-30-514-024', '1319-30-514-016', '1319-30-514-014', '1319-30-514-013', '1319-30-514-012', '1319-30-514-011', '1319-30-514-010', '1319-30-514-017', '1319-30-514-015', '1319-30-514-000', '1319-30-611-000', '1319-30-622-001', '1319-30-614-002', '1319-30-624-000', '1319-30-615-005', '1319-30-615-004', '1319-30-615-001', '1319-30-615-002']

In [22]:
#Update Geometry Parcels_Master
#Need to define field list for parcel master
fields = list(set(dfparcelMaster.columns) & set(dfparcelNew.columns))
parcel_master_old_apn = old_new_parcels_list(featureLayer, parcelNew, 'No', 
                                             prefix_remove,'Old APN')
print(parcel_master_old_apn)


Function make_old_new_dataframe took 68.85989952087402 seconds to execute.
Function old_new_parcels_list took 68.8608946800232 seconds to execute.
['048-140-01', '048-140-02', '1219-00-001-001', '1319-06-001-006', '1319-06-001-005', '1319-06-001-004', '1319-19-721-001', '1319-19-721-002', '1319-19-721-003', '1319-19-720-040', '1319-30-712-001', '1319-30-516-049', '036-170-021', '036-170-022', '036-160-007', '036-160-005', '036-160-006', '036-170-006', '036-160-001', '036-170-007', '036-170-008', '036-160-002', '036-160-003', '036-160-004', '036-170-005', '036-170-009', '036-170-004', '036-170-015', '036-170-010', '036-170-011', '036-170-012', '036-170-003', '036-170-013', '036-170-002', '036-170-014', '036-170-001', '110-070-008', '110-070-009']


C:\Users\amcclary\AppData\Local\Temp\25\ipykernel_17332\914419495.py:284: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_merge['DiscoveryDate'] = date
C:\Users\amcclary\AppData\Local\Temp\25\ipykernel_17332\914419495.py:286: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_merge['TRPA_Boundary'] = df_merge[TRPA_BNDY_Fields].sum(axis=1)


In [52]:


delete_old_parcels(featureLayer, parcel_master_old_apn)

6 rows deleted from parcelMasterVersion.
Function delete_old_parcels took 109.7309582233429 seconds to execute.


In [59]:
fields = arcpy.ListFields(featureLayer)
field_names_master = [field.name for field in fields ]
fields_new = arcpy.ListFields(parcelNew)
field_names_new = [field.name for field in fields_new ]

common_fields = list(set(field_names_master) & set(field_names_new))
insert_new_parcels(featureLayer, parcel_master_new_apn, parcelNew, common_fields)

1 rows inserted into parcelMasterVersion.
2 rows inserted into parcelMasterVersion.
3 rows inserted into parcelMasterVersion.
4 rows inserted into parcelMasterVersion.
5 rows inserted into parcelMasterVersion.
6 rows inserted into parcelMasterVersion.
7 rows inserted into parcelMasterVersion.
8 rows inserted into parcelMasterVersion.
9 rows inserted into parcelMasterVersion.
10 rows inserted into parcelMasterVersion.
11 rows inserted into parcelMasterVersion.
12 rows inserted into parcelMasterVersion.
13 rows inserted into parcelMasterVersion.
14 rows inserted into parcelMasterVersion.
15 rows inserted into parcelMasterVersion.
16 rows inserted into parcelMasterVersion.
17 rows inserted into parcelMasterVersion.
18 rows inserted into parcelMasterVersion.
19 rows inserted into parcelMasterVersion.
20 rows inserted into parcelMasterVersion.
21 rows inserted into parcelMasterVersion.
22 rows inserted into parcelMasterVersion.
23 rows inserted into parcelMasterVersion.
24 rows inserted int

In [83]:

update_parcel_geometry(featureLayer, parcelNew)

Started data transfer: 2023-05-19 13:05:55
Finished data transfer: 2023-05-19 13:06:07
Function fieldJoinCalc_multikey took 11.853193759918213 seconds to execute.
3512 shapes shifted.
Function update_parcel_geometry took 146.6663978099823 seconds to execute.


### Update Parcel Base

In [18]:
# set up mapping for conversion from feature class to data frame
data_type_mapping = {
    "String": str,
    "Integer": int,
    "SmallInteger": int,
    "Single": float,
    "Double": float,
    "Date": pd.to_datetime
}

fields_to_exclude = ['SHAPE']
dfparcelNew = generate_spatial_dataframe(parcelNew, data_type_mapping, fields_to_exclude)
dfparcelBase = generate_spatial_dataframe(featureLayer_Base, data_type_mapping, fields_to_exclude)

# filter to new parcels within TRPA boundary
dfparcelNew=dfparcelNew.loc[dfparcelNew['WITHIN_TRPA_BNDY']==1]
df_special_parcels = pd.read_excel("//Trpa-fs01/GIS/PARCELUPDATE/Workspace/special_parcels.xlsx")
matching_apns_parcel_base = return_matching_apns(featureLayer_Base, parcelNew, df_special_parcels)

# filter parcel master to matching APNs
dfparcelBase = dfparcelBase[dfparcelBase['APN'].isin(matching_apns_parcel_base)]
# filter staging data to matching APNs
dfparcelNew    = dfparcelNew[dfparcelNew['APN'].isin(matching_apns_parcel_base)]


fields_to_ignore = []

# get the differences as dictionaries
differences_base = differenceDictionary(dfparcelBase, dfparcelNew, 'APN', fields_to_ignore)

Function generate_spatial_dataframe took 4.94793701171875 seconds to execute.
Function generate_spatial_dataframe took 10.60365891456604 seconds to execute.
Function return_matching_apns took 22.762329578399658 seconds to execute.
Function differenceDictionary took 1.142991304397583 seconds to execute.


In [21]:
print(len(differences_base))
df_dif=pd.DataFrame(differences_base)
df_dif.to_csv('base_differences.csv')

49115


In [ ]:
dfparcelBase = dfparcelBase[dfparcelBase['APN'].isin(matching_apns_parcel_base)]
#This needs to be redefined
dfparcelNew = dfparcelNew[dfparcelNew['APN'].isin(matching_apns_parcel_base)]
fields_to_ignore = []
differences_base = differenceDictionary(dfparcelBase, dfparcelNew, 'APN', fields_to_ignore)

update_fc_from_dict(differences_base, 'APN', featureLayer_Base)

In [11]:
#Geometry Updates
fields = ['APN','PPNO','PARCEL_ACRES','PARCEL_SQFT','JURISDICTION','SHAPE@']
delete_old_parcels(featureLayer_Base, parcel_base_old_apn)


41 rows deleted from parcelBaseVersion.
Function delete_old_parcels took 115.85176134109497 seconds to execute.


In [12]:
insert_new_parcels(featureLayer_Base, parcel_base_new_apn, parcelNew, fields)


"APN" IN ('090-282-018', '097-140-043', '116-080-003', '097-130-030', '092-100-030', '090-153-011', '112-010-012', '112-010-011', '117-160-010', '117-190-060', '094-440-010', '085-343-022', '112-190-048', '094-350-021', '094-470-026', '094-233-004', '097-091-009', '117-200-054', '117-180-008', '117-180-055', '090-074-021', '094-070-002', '094-300-011', '098-180-016', '097-200-014', '097-092-011', '098-180-015', '117-230-004', '097-193-004', '097-060-040', '084-010-007', '097-140-046', '097-192-010', '023-960-008', '023-960-002', '023-960-003', '023-960-004', '023-960-005', '023-960-006', '023-960-007', '090-282-019', '1418-34-401-030', '1418-34-111-040', '1418-34-111-041', '1418-34-111-042', '1318-09-810-118', '1318-09-810-119', '1318-15-110-014', '110-070-021', '1319-00-001-001', '1419-00-001-002', '1319-30-516-050')
1 rows inserted into parcelBaseVersion.
2 rows inserted into parcelBaseVersion.
3 rows inserted into parcelBaseVersion.
4 rows inserted into parcelBaseVersion.
5 rows ins

In [14]:
update_parcel_geometry(featureLayer_Base, parcelNew)

Started data transfer: 2023-05-25 07:01:21
Updating row 1000
Updating row 2000
Finished data transfer: 2023-05-25 07:05:01
Function fieldJoinCalc_multikey took 219.7470772266388 seconds to execute.
5684 shapes shifted.
Function update_parcel_geometry took 290.80825543403625 seconds to execute.


In [23]:
old_feature_class = 'ParcelMaster_Backup'

New_Old_Parcels = make_old_new_dataframe(old_feature_class, featureLayer, 'Yes', prefix_remove)

New_Old_Parcels['OBJECTID'] = New_Old_Parcels.reset_index().index
New_Old_Parcels = New_Old_Parcels.iloc[:,[4,0,1,2,3]]
#Need to fix this so it keys off objectID or just use insert cursor. That's probably best practice
New_Old_Parcels['OBJECTID'] = New_Old_Parcels['OBJECTID']+181


print(New_Old_Parcels)

Function make_old_new_dataframe took 56.45237588882446 seconds to execute.
       OBJECTID              APN   Status DiscoveryDate  TRPA_Boundary
499         181       122-181-64  Old APN    2023-05-25            1.0
500         182       122-181-65  Old APN    2023-05-25            1.0
1031        183       123-071-38  Old APN    2023-05-25            1.0
7385        184       130-331-05  Old APN    2023-05-25            1.0
9676        185  1418-27-401-001  Old APN    2023-05-25            1.0
...         ...              ...      ...           ...            ...
61303       322  1319-00-001-001  New APN    2023-05-25            1.0
61304       323  1419-00-001-002  New APN    2023-05-25            1.0
61305       324  1319-30-516-050  New APN    2023-05-25            1.0
61306       325      110-070-018  New APN    2023-05-25            1.0
61307       326      031-213-016  New APN    2023-05-25            1.0

[146 rows x 5 columns]


In [24]:
from sqlalchemy import create_engine
import urllib
quoted = urllib.parse.quote_plus("DRIVER={SQL Server};SERVER=sql12;DATABASE=sde_tabular;UID=sde;PWD=staff")
engine = create_engine('mssql+pyodbc:///?odbc_connect={}'.format(quoted))

# dfNew.set_index('APN')
New_Old_Parcels.to_sql('Parcel_APN_NewOld', con = engine, 
             chunksize=200, method='multi',index=False, if_exists='append')

C:\Program Files\ArcGIS\Pro\bin\Python\envs\arcgispro-py3\lib\site-packages\pandas\io\sql.py:1666: UserWarning: The provided table name 'Parcel_APN_NewOld' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  warnings.warn(msg, UserWarning)


146